# 01 — VocaLip Thai: Supervised Word-Level Baseline

**Experiment 0 — Baseline for the VocaLip Thai research pipeline**

Research direction:

`video → visual preprocessing → representation learning → temporal modeling/alignment → Thai VSR → LLM-assisted decoding`

This notebook intentionally starts with a **simple supervised closed-vocabulary baseline** before adding the proposed advanced components.

## Current task

- 44 Thai words
- Mouth-only baseline
- 16 frames per word clip
- 112×112 RGB mouth ROI
- CNN frame encoder
- BiGRU temporal encoder
- Temporal attention
- 44-class classifier
- CrossEntropyLoss
- Accuracy + Macro F1 + Top-k + Confusion Matrix

## Not included yet

- Self-supervised pretraining
- Explicit temporal alignment loss
- Face + Mouth dual-stream fusion
- LLM-assisted decoding
- Continuous phrase/sentence recognition

These will be developed in later notebooks so each contribution can be measured separately.


## VocaLip Thai Experimental Roadmap

This folder is organized as a progressive research pipeline:

1. **01 — Supervised Word-Level Baseline** ← current notebook
2. **02 — Self-Supervised Visual Pretraining**
3. **03 — SSL Fine-Tuning for 44 Thai Words**
4. **04 — Temporal Modeling / Alignment Ablation**
5. **05 — Face + Mouth Dual-Stream Ablation**
6. **06 — LLM-Assisted Decoding / Rescoring**
7. **07 — Real-Time Inference + Prototype Evaluation**

The baseline is kept intentionally simple so later improvements can be evaluated with a clear ablation study.


In [ ]:
# CELL 1 : ENVIRONMENT / IMPORTS

import os
import sys
import json
import math
import random
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", DEVICE)

In [ ]:
# CELL 2 : CONFIG

ROOT = Path(r"D:\Face_detection\dataset\word_level_v5\video001")

MANIFEST_PATH = ROOT / "word_train_manifest_v5_training_ready.csv"
LABEL_MAP_PATH = ROOT / "word_label_map_v5.json"

CACHE_ROOT = ROOT / "mouth_cache_v3"

NUM_CLASSES = 44
TARGET_FRAMES = 16
ROI_SIZE = 112

BATCH_SIZE = 16
NUM_WORKERS = 0          # Windows/Jupyter safe baseline
LR = 3e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 40
PATIENCE = 8

SEED = 42

# Imbalance strategy:
# Start with weighted CrossEntropyLoss.
# Do NOT use WeightedRandomSampler at the same time unless testing intentionally.
USE_CLASS_WEIGHTED_LOSS = True
USE_WEIGHTED_SAMPLER = False

MODEL_DIR = ROOT / "models_01_baseline"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "01_word_level_baseline_best.pt"

print("Manifest :", MANIFEST_PATH)
print("Label map:", LABEL_MAP_PATH)
print("Cache    :", CACHE_ROOT)
print("Model dir:", MODEL_DIR)

In [ ]:
# CELL 3 : REPRODUCIBILITY

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Reproducibility first. Can be relaxed later for speed.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print("Seed:", SEED)

In [ ]:
# CELL 4 : LOAD FINAL WORD-LEVEL DATASET

assert MANIFEST_PATH.exists(), f"Manifest not found: {MANIFEST_PATH}"
assert LABEL_MAP_PATH.exists(), f"Label map not found: {LABEL_MAP_PATH}"

df = pd.read_csv(MANIFEST_PATH, encoding="utf-8-sig")

with open(LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    label_map = json.load(f)

required_cols = {
    "split",
    "word",
    "model_class_id",
    "video_clip_path"
}

missing = required_cols - set(df.columns)
assert not missing, f"Missing columns: {missing}"

df["model_class_id"] = df["model_class_id"].astype(int)

print("Samples :", len(df))
print("Classes :", df["model_class_id"].nunique())
print("Words   :", df["word"].nunique())
print()
print(df["split"].value_counts())

assert df["model_class_id"].nunique() == NUM_CLASSES
assert df["model_class_id"].min() == 0
assert df["model_class_id"].max() == NUM_CLASSES - 1

display(df.head())

In [ ]:
# CELL 5 : DATASET AUDIT — DO NOT RESPLIT

split_summary = (
    df.groupby("split")
      .agg(
          samples=("model_class_id", "size"),
          classes=("model_class_id", "nunique"),
          words=("word", "nunique"),
      )
)

print(split_summary)

for split in ["train", "val", "test"]:
    sub = df[df["split"] == split]
    counts = sub["model_class_id"].value_counts()

    print(
        f"{split:5s} | samples={len(sub):4d} | "
        f"classes={sub['model_class_id'].nunique():2d} | "
        f"min/class={counts.min():2d} | "
        f"max/class={counts.max():3d} | "
        f"mean/class={counts.mean():.2f}"
    )

# Existing split is preserved by design.
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

assert train_df["model_class_id"].nunique() == NUM_CLASSES
assert val_df["model_class_id"].nunique() == NUM_CLASSES
assert test_df["model_class_id"].nunique() == NUM_CLASSES

print("\n✅ Existing train/val/test split preserved")

## Mouth cache policy

Version 3 should **reuse cache if it already exists**.

Expected cache file per sample:

`CACHE_ROOT / split / <word_clip_id or clip stem>.npz`

Each `.npz` contains:

- `frames`: shape `(16, 112, 112, 3)`, uint8

The training loop never runs MediaPipe.  
Mouth extraction is a **one-time preprocessing stage** and can be replaced by any better cropper later without rewriting the model/training code.

In [ ]:
# ============================================================
# CELL 6 : CACHE AUDIT — NON-BLOCKING
# ตรวจสอบ mouth cache ปัจจุบัน
# ถ้ายังไม่ครบ ให้แจ้งเตือน แต่ไม่หยุด Notebook
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# CHECK CACHE EXISTENCE
# ------------------------------------------------------------

df["mouth_cache_exists"] = df[
    "mouth_cache_path"
].map(
    lambda p: Path(str(p)).exists()
)

total_samples = len(df)

existing_cache = int(
    df["mouth_cache_exists"].sum()
)

missing_count = (
    total_samples - existing_cache
)

cache_pct = (
    existing_cache / total_samples * 100
    if total_samples > 0
    else 0.0
)

print(
    f"Existing cache: "
    f"{existing_cache:,}/{total_samples:,} "
    f"({cache_pct:.2f}%)"
)

print()
print("By split:")

# ------------------------------------------------------------
# SPLIT SUMMARY
# ------------------------------------------------------------

for split_name in [
    "train",
    "val",
    "test"
]:
    split_df = df[
        df["split"] == split_name
    ]

    split_total = len(split_df)

    split_existing = int(
        split_df[
            "mouth_cache_exists"
        ].sum()
    )

    print(
        f"{split_name:<5}: "
        f"{split_existing:4d}/"
        f"{split_total:4d}"
    )


# ------------------------------------------------------------
# MISSING CACHE TABLE
# ------------------------------------------------------------

missing_cache = df.loc[
    ~df["mouth_cache_exists"]
].copy()

print()

if len(missing_cache) > 0:

    print(
        f"⚠️ Cache incomplete: "
        f"{len(missing_cache):,} files missing."
    )

    print(
        "This is expected before preprocessing."
    )

    print(
        "Continue to Cell 7 to load the mouth extractor."
    )

    print(
        "Then run Cell 7.1 to test one real clip."
    )

    print(
        "Do NOT start training yet."
    )

    print()
    print(
        "Example missing cache rows:"
    )

    display(
        missing_cache[
            [
                "split",
                "word",
                "video_clip_path",
                "mouth_cache_path"
            ]
        ].head(20)
    )

else:

    print(
        "✅ Cache complete."
    )

    print(
        "All samples have mouth-cache files."
    )


# ------------------------------------------------------------
# OPTIONAL SAMPLE SHAPE CHECK
# ทำเฉพาะเมื่อมี cache อย่างน้อย 1 ไฟล์
# ------------------------------------------------------------

existing_rows = df.loc[
    df["mouth_cache_exists"]
]

if len(existing_rows) > 0:

    sample_cache = Path(
        str(
            existing_rows.iloc[0][
                "mouth_cache_path"
            ]
        )
    )

    try:

        sample_data = np.load(
            sample_cache
        )

        if "frames" in sample_data.files:
            sample_seq = sample_data[
                "frames"
            ]

        else:
            sample_seq = sample_data[
                sample_data.files[0]
            ]

        print()
        print(
            "Sample cache:"
        )

        print(
            sample_cache
        )

        print(
            "Sample shape:",
            sample_seq.shape
        )

        print(
            "Sample dtype:",
            sample_seq.dtype
        )

    except Exception as e:

        print()
        print(
            "⚠️ Could not inspect sample cache:"
        )

        print(
            repr(e)
        )


print()
print("=" * 60)
print("CELL 6 COMPLETE")
print("=" * 60)

if len(missing_cache) > 0:

    print(
        "Next step: run Cell 7 → Cell 7.1"
    )

else:

    print(
        "Next step: proceed to cache validation / training gate"
    )

In [ ]:
# ============================================================
# CELL 7 : ROBUST MOUTH ROI EXTRACTOR + CACHE BUILDER
# VocaLip Thai — Word-Level Baseline
#
# Pipeline
# ------------------------------------------------------------
# Full-frame video
#   -> MediaPipe VIDEO mode
#   -> if face not found:
#        multi-region crop search using IMAGE mode
#   -> lip landmarks
#   -> map coordinates back to original frame
#   -> interpolate missing frames
#   -> EMA stabilization
#   -> mouth ROI
#   -> resize 112x112
#   -> temporal resample to 16 frames
#   -> save .npz
#
# IMPORTANT:
# - VIDEO detector is used only once per frame
# - IMAGE detector handles fallback crops
# - Therefore there is NO duplicate timestamp problem
# ============================================================

import cv2
import numpy as np
import pandas as pd
import mediapipe as mp

from pathlib import Path
from tqdm.auto import tqdm

from mediapipe.tasks import python
from mediapipe.tasks.python import vision


# ============================================================
# CONFIG
# ============================================================

FACE_LANDMARKER_MODEL = Path(
    r"D:\Face_detection\models\face_landmarker.task"
)

assert FACE_LANDMARKER_MODEL.exists(), (
    f"FaceLandmarker model not found:\n"
    f"{FACE_LANDMARKER_MODEL}"
)


# Use existing notebook config if already defined
TARGET_FRAMES = globals().get(
    "TARGET_FRAMES",
    16
)

ROI_SIZE = globals().get(
    "ROI_SIZE",
    112
)


# FaceLandmarker thresholds
MIN_FACE_DETECTION_CONF = 0.25
MIN_FACE_PRESENCE_CONF = 0.25
MIN_TRACKING_CONF = 0.25


# Mouth crop
MOUTH_SCALE = 2.4

# Temporal stabilization
SMOOTH_ALPHA = 0.30

# Fallback crop upscale
FALLBACK_UPSCALE = 2.5


# MediaPipe lip landmark IDs
LIP_INDICES = [
    61, 146, 91, 181, 84, 17,
    314, 405, 321, 375, 291,
    308, 324, 318, 402, 317,
    14, 87, 178, 88, 95,
    185, 40, 39, 37, 0,
    267, 269, 270, 409, 415,
    310, 311, 312, 13, 82,
    81, 80, 191, 78
]


print("TARGET_FRAMES :", TARGET_FRAMES)
print("ROI_SIZE      :", ROI_SIZE)
print("Face model    :", FACE_LANDMARKER_MODEL)


# ============================================================
# CREATE MEDIAPIPE LANDMARKERS
# ============================================================

def create_video_landmarker():
    """
    Primary detector.

    Used on the ORIGINAL full video frame.
    VIDEO mode gives temporal tracking.
    """

    base_options = python.BaseOptions(
        model_asset_path=str(
            FACE_LANDMARKER_MODEL
        )
    )

    options = vision.FaceLandmarkerOptions(
        base_options=base_options,

        running_mode=vision.RunningMode.VIDEO,

        num_faces=1,

        min_face_detection_confidence=(
            MIN_FACE_DETECTION_CONF
        ),

        min_face_presence_confidence=(
            MIN_FACE_PRESENCE_CONF
        ),

        min_tracking_confidence=(
            MIN_TRACKING_CONF
        ),

        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False
    )

    return vision.FaceLandmarker.create_from_options(
        options
    )


def create_image_landmarker():
    """
    Fallback detector.

    IMAGE mode is intentionally used here because
    multiple crops can be tested from the same video
    frame without timestamp restrictions.
    """

    base_options = python.BaseOptions(
        model_asset_path=str(
            FACE_LANDMARKER_MODEL
        )
    )

    options = vision.FaceLandmarkerOptions(
        base_options=base_options,

        running_mode=vision.RunningMode.IMAGE,

        num_faces=1,

        min_face_detection_confidence=(
            MIN_FACE_DETECTION_CONF
        ),

        min_face_presence_confidence=(
            MIN_FACE_PRESENCE_CONF
        ),

        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False
    )

    return vision.FaceLandmarker.create_from_options(
        options
    )


# ============================================================
# DETECTION HELPERS
# ============================================================

def detect_video_frame(
    landmarker,
    rgb,
    timestamp_ms
):
    """
    Detect face landmarks using VIDEO mode.
    """

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=np.ascontiguousarray(rgb)
    )

    result = landmarker.detect_for_video(
        mp_image,
        timestamp_ms
    )

    if not result.face_landmarks:
        return None

    return result.face_landmarks[0]


def detect_image(
    landmarker,
    rgb
):
    """
    Detect face landmarks using IMAGE mode.
    No timestamp is required.
    """

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=np.ascontiguousarray(rgb)
    )

    result = landmarker.detect(
        mp_image
    )

    if not result.face_landmarks:
        return None

    return result.face_landmarks[0]


# ============================================================
# FALLBACK SEARCH REGIONS
# ============================================================

def get_face_search_regions(
    frame_width,
    frame_height
):
    """
    Returns several broad crop regions.

    This is more robust than assuming the speaker
    is always exactly in the center.

    Coordinates:
        x1, y1, x2, y2
    """

    w = frame_width
    h = frame_height

    regions = [

        # Upper center
        (
            int(w * 0.15),
            int(h * 0.00),
            int(w * 0.85),
            int(h * 0.75)
        ),

        # Upper left
        (
            int(w * 0.00),
            int(h * 0.00),
            int(w * 0.65),
            int(h * 0.75)
        ),

        # Upper right
        (
            int(w * 0.35),
            int(h * 0.00),
            int(w * 1.00),
            int(h * 0.75)
        ),

        # Center area
        (
            int(w * 0.15),
            int(h * 0.10),
            int(w * 0.85),
            int(h * 0.90)
        ),
    ]

    return regions


# ============================================================
# LANDMARK -> MOUTH BOX
# ============================================================

def mouth_box_from_landmarks(
    landmarks,
    image_width,
    image_height
):
    """
    Convert normalized MediaPipe lip landmarks
    to a square mouth ROI.

    Returns:
        cx
        cy
        side
    """

    xs = []
    ys = []

    for landmark_id in LIP_INDICES:

        lm = landmarks[
            landmark_id
        ]

        xs.append(
            lm.x * image_width
        )

        ys.append(
            lm.y * image_height
        )

    xs = np.asarray(
        xs,
        dtype=np.float32
    )

    ys = np.asarray(
        ys,
        dtype=np.float32
    )


    x_min = float(
        np.min(xs)
    )

    x_max = float(
        np.max(xs)
    )

    y_min = float(
        np.min(ys)
    )

    y_max = float(
        np.max(ys)
    )


    cx = (
        x_min + x_max
    ) / 2.0

    cy = (
        y_min + y_max
    ) / 2.0


    mouth_width = (
        x_max - x_min
    )

    mouth_height = (
        y_max - y_min
    )


    side = max(
        mouth_width,
        mouth_height
    ) * MOUTH_SCALE


    return (
        float(cx),
        float(cy),
        float(side)
    )


# ============================================================
# ROBUST FACE / MOUTH DETECTION
# ============================================================

def detect_mouth_box_robust(
    video_landmarker,
    image_landmarker,
    rgb,
    timestamp_ms
):
    """
    Detection strategy:

    1. Try original full frame with VIDEO mode.

    2. If failed:
       search multiple crop regions with IMAGE mode.

    Returns coordinates in ORIGINAL FRAME space.
    """

    frame_h, frame_w = (
        rgb.shape[:2]
    )


    # ========================================================
    # TRY 1 : ORIGINAL FULL FRAME
    # ========================================================

    landmarks = detect_video_frame(
        video_landmarker,
        rgb,
        timestamp_ms
    )


    if landmarks is not None:

        cx, cy, side = (
            mouth_box_from_landmarks(
                landmarks,
                frame_w,
                frame_h
            )
        )

        return {
            "cx": cx,
            "cy": cy,
            "side": side,
            "method": "full_video"
        }


    # ========================================================
    # TRY 2 : MULTI-REGION IMAGE-MODE SEARCH
    # ========================================================

    search_regions = (
        get_face_search_regions(
            frame_w,
            frame_h
        )
    )


    for region_index, region in enumerate(
        search_regions
    ):

        x1, y1, x2, y2 = region


        crop = rgb[
            y1:y2,
            x1:x2
        ]


        if crop.size == 0:
            continue


        crop_h, crop_w = (
            crop.shape[:2]
        )


        # ------------------------------------------
        # Upscale the face-search crop
        # ------------------------------------------

        upscale_w = int(
            round(
                crop_w
                * FALLBACK_UPSCALE
            )
        )

        upscale_h = int(
            round(
                crop_h
                * FALLBACK_UPSCALE
            )
        )


        crop_up = cv2.resize(
            crop,
            (
                upscale_w,
                upscale_h
            ),
            interpolation=cv2.INTER_CUBIC
        )


        landmarks = detect_image(
            image_landmarker,
            crop_up
        )


        if landmarks is None:
            continue


        # ------------------------------------------
        # Mouth coordinates inside UPSCALED crop
        # ------------------------------------------

        cx_up, cy_up, side_up = (
            mouth_box_from_landmarks(
                landmarks,
                upscale_w,
                upscale_h
            )
        )


        # ------------------------------------------
        # Map:
        # upscale crop
        # -> original crop
        # ------------------------------------------

        cx_crop = (
            cx_up
            / FALLBACK_UPSCALE
        )

        cy_crop = (
            cy_up
            / FALLBACK_UPSCALE
        )

        side_crop = (
            side_up
            / FALLBACK_UPSCALE
        )


        # ------------------------------------------
        # Map:
        # crop coordinates
        # -> ORIGINAL frame coordinates
        # ------------------------------------------

        cx_full = (
            x1 + cx_crop
        )

        cy_full = (
            y1 + cy_crop
        )


        return {
            "cx": float(
                cx_full
            ),

            "cy": float(
                cy_full
            ),

            "side": float(
                side_crop
            ),

            "method": (
                f"fallback_region_"
                f"{region_index}"
            )
        }


    # Nothing found
    return None


# ============================================================
# INTERPOLATION
# ============================================================

def interpolate_missing(
    values
):
    """
    Linear interpolation for missing values.
    """

    values = np.asarray(
        values,
        dtype=np.float32
    )


    valid = np.isfinite(
        values
    )


    if valid.sum() == 0:

        raise RuntimeError(
            "NO_VALID_VALUES_FOR_INTERPOLATION"
        )


    if valid.sum() == 1:

        values[:] = (
            values[
                valid
            ][0]
        )

        return values


    indices = np.arange(
        len(values)
    )


    values[
        ~valid
    ] = np.interp(

        indices[
            ~valid
        ],

        indices[
            valid
        ],

        values[
            valid
        ]
    )


    return values


# ============================================================
# EMA STABILIZATION
# ============================================================

def ema_smooth(
    values,
    alpha=0.30
):
    """
    Exponential moving average.
    """

    values = np.asarray(
        values,
        dtype=np.float32
    )


    output = (
        values.copy()
    )


    for i in range(
        1,
        len(output)
    ):

        output[i] = (
            alpha
            * values[i]
            +
            (
                1.0
                - alpha
            )
            * output[i - 1]
        )


    return output


# ============================================================
# SAFE MOUTH CROP
# ============================================================

def crop_mouth_roi(
    frame_bgr,
    cx,
    cy,
    side
):
    """
    Crop square mouth ROI from ORIGINAL frame.

    Output:
        RGB uint8
        [ROI_SIZE, ROI_SIZE, 3]
    """

    frame_h, frame_w = (
        frame_bgr.shape[:2]
    )


    side = max(
        float(side),
        8.0
    )


    half = (
        side / 2.0
    )


    x1 = int(
        round(
            cx - half
        )
    )

    x2 = int(
        round(
            cx + half
        )
    )

    y1 = int(
        round(
            cy - half
        )
    )

    y2 = int(
        round(
            cy + half
        )
    )


    # ========================================================
    # BORDER PADDING
    # ========================================================

    pad_left = max(
        0,
        -x1
    )

    pad_top = max(
        0,
        -y1
    )

    pad_right = max(
        0,
        x2 - frame_w
    )

    pad_bottom = max(
        0,
        y2 - frame_h
    )


    if (
        pad_left
        or pad_top
        or pad_right
        or pad_bottom
    ):

        frame_bgr = cv2.copyMakeBorder(
            frame_bgr,

            pad_top,
            pad_bottom,
            pad_left,
            pad_right,

            borderType=(
                cv2.BORDER_REPLICATE
            )
        )


        x1 += pad_left
        x2 += pad_left

        y1 += pad_top
        y2 += pad_top


    roi = frame_bgr[
        y1:y2,
        x1:x2
    ]


    if roi.size == 0:

        raise RuntimeError(
            "EMPTY_MOUTH_ROI"
        )


    roi = cv2.resize(
        roi,

        (
            ROI_SIZE,
            ROI_SIZE
        ),

        interpolation=(
            cv2.INTER_AREA
        )
    )


    roi = cv2.cvtColor(
        roi,
        cv2.COLOR_BGR2RGB
    )


    return roi


# ============================================================
# TEMPORAL RESAMPLE
# ============================================================

def temporal_resample(
    frames,
    target_frames=16
):
    """
    Uniform temporal sampling.

    Input:
        T frames

    Output:
        target_frames
    """

    if len(frames) == 0:

        raise RuntimeError(
            "EMPTY_SEQUENCE"
        )


    indices = np.linspace(
        0,
        len(frames) - 1,
        target_frames
    )


    indices = np.round(
        indices
    ).astype(
        np.int32
    )


    sampled = [
        frames[i]
        for i in indices
    ]


    return np.stack(
        sampled,
        axis=0
    )


# ============================================================
# MAIN VIDEO -> MOUTH SEQUENCE
# ============================================================

def extract_mouth_sequence(
    video_path,
    return_debug=False
):
    """
    Extract stabilized mouth ROI sequence.

    Returns:
        seq:
            [TARGET_FRAMES, ROI_SIZE, ROI_SIZE, 3]

        optionally:
            debug dict
    """

    video_path = Path(
        str(video_path)
    )


    if not video_path.exists():

        raise FileNotFoundError(
            f"VIDEO_NOT_FOUND: "
            f"{video_path}"
        )


    # ========================================================
    # LOAD VIDEO
    # ========================================================

    cap = cv2.VideoCapture(
        str(video_path)
    )


    if not cap.isOpened():

        raise RuntimeError(
            f"CANNOT_OPEN_VIDEO: "
            f"{video_path}"
        )


    fps = cap.get(
        cv2.CAP_PROP_FPS
    )


    if (
        fps is None
        or fps <= 0
        or not np.isfinite(fps)
    ):

        fps = 30.0


    frames_bgr = []


    while True:

        ok, frame = cap.read()


        if not ok:
            break


        frames_bgr.append(
            frame
        )


    cap.release()


    if len(frames_bgr) == 0:

        raise RuntimeError(
            "VIDEO_HAS_NO_FRAMES"
        )


    # ========================================================
    # CREATE DETECTORS FOR THIS VIDEO
    # ========================================================

    video_landmarker = (
        create_video_landmarker()
    )

    image_landmarker = (
        create_image_landmarker()
    )


    boxes = []
    methods = []


    try:

        for frame_idx, frame_bgr in enumerate(
            frames_bgr
        ):

            rgb = cv2.cvtColor(
                frame_bgr,
                cv2.COLOR_BGR2RGB
            )


            # Strictly increasing timestamp
            timestamp_ms = int(
                round(
                    frame_idx
                    * 1000.0
                    / fps
                )
            )


            # Safety against duplicate integer timestamps
            timestamp_ms = max(
                timestamp_ms,
                frame_idx
            )


            detection = (
                detect_mouth_box_robust(
                    video_landmarker,
                    image_landmarker,
                    rgb,
                    timestamp_ms
                )
            )


            if detection is None:

                boxes.append(
                    (
                        np.nan,
                        np.nan,
                        np.nan
                    )
                )

                methods.append(
                    "missing"
                )

                continue


            boxes.append(
                (
                    detection["cx"],
                    detection["cy"],
                    detection["side"]
                )
            )


            methods.append(
                detection["method"]
            )


    finally:

        video_landmarker.close()
        image_landmarker.close()


    # ========================================================
    # DETECTION SUMMARY
    # ========================================================

    boxes = np.asarray(
        boxes,
        dtype=np.float32
    )


    detected_mask = np.isfinite(
        boxes[:, 0]
    )


    detected_count = int(
        detected_mask.sum()
    )


    total_frames = len(
        frames_bgr
    )


    if detected_count == 0:

        raise RuntimeError(
            "ZERO_FACE_DETECTION_AFTER_ALL_FALLBACKS"
        )


    # ========================================================
    # INTERPOLATE MISSING DETECTIONS
    # ========================================================

    cx = interpolate_missing(
        boxes[:, 0]
    )

    cy = interpolate_missing(
        boxes[:, 1]
    )

    side = interpolate_missing(
        boxes[:, 2]
    )


    # ========================================================
    # STABILIZE ROI
    # ========================================================

    cx = ema_smooth(
        cx,
        SMOOTH_ALPHA
    )

    cy = ema_smooth(
        cy,
        SMOOTH_ALPHA
    )

    side = ema_smooth(
        side,
        SMOOTH_ALPHA
    )


    # ========================================================
    # CROP MOUTH FROM ORIGINAL VIDEO
    # ========================================================

    mouth_frames = []


    for i, frame_bgr in enumerate(
        frames_bgr
    ):

        roi = crop_mouth_roi(

            frame_bgr,

            cx[i],
            cy[i],
            side[i]
        )


        mouth_frames.append(
            roi
        )


    # ========================================================
    # TEMPORAL RESAMPLING
    # ========================================================

    sequence = temporal_resample(
        mouth_frames,
        TARGET_FRAMES
    )


    sequence = sequence.astype(
        np.uint8
    )


    expected_shape = (
        TARGET_FRAMES,
        ROI_SIZE,
        ROI_SIZE,
        3
    )


    if sequence.shape != expected_shape:

        raise RuntimeError(
            f"BAD_OUTPUT_SHAPE: "
            f"{sequence.shape} "
            f"expected={expected_shape}"
        )


    # ========================================================
    # DEBUG
    # ========================================================

    if return_debug:

        debug = {

            "video_path":
                str(video_path),

            "frames":
                total_frames,

            "detected_frames":
                detected_count,

            "detection_rate":
                detected_count
                / total_frames,

            "full_video_count":
                sum(
                    x == "full_video"
                    for x in methods
                ),

            "fallback_count":
                sum(
                    str(x).startswith(
                        "fallback_region_"
                    )
                    for x in methods
                ),

            "missing_count":
                methods.count(
                    "missing"
                ),

            "methods":
                methods
        }


        return (
            sequence,
            debug
        )


    return sequence


# ============================================================
# CACHE OUTPUT PATH
# ============================================================

def get_cache_output_path(
    row
):
    """
    Prefer mouth_cache_path already generated
    by earlier notebook cells.
    """

    if (
        "mouth_cache_path"
        in row.index
    ):

        return Path(
            str(
                row[
                    "mouth_cache_path"
                ]
            )
        )


    # Fallback if dataframe has no mouth_cache_path
    split = str(
        row["split"]
    )

    class_id = int(
        row["model_class_id"]
    )

    word = str(
        row["word"]
    )


    clip_id = str(

        row.get(
            "word_clip_id",

            row.get(
                "clip_id",
                row.name
            )
        )
    )


    return (
        Path(CACHE_ROOT)
        / split
        / f"{class_id:03d}_{word}"
        / f"{clip_id}.npz"
    )


# ============================================================
# BUILD CACHE
# ============================================================

def build_missing_cache(
    dataframe,
    verbose_fail=True
):
    """
    Build ONLY cache files that do not already exist.
    """

    successes = 0
    skipped = 0

    failures = []


    for row_index, row in tqdm(

        dataframe.iterrows(),

        total=len(dataframe),

        desc="Building mouth cache"
    ):

        cache_path = (
            get_cache_output_path(
                row
            )
        )


        # ----------------------------------------------------
        # SKIP EXISTING CACHE
        # ----------------------------------------------------

        if cache_path.exists():

            skipped += 1

            continue


        cache_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )


        try:

            seq, debug = (
                extract_mouth_sequence(

                    row[
                        "video_clip_path"
                    ],

                    return_debug=True
                )
            )


            np.savez_compressed(

                cache_path,

                frames=seq,

                word=str(
                    row["word"]
                ),

                model_class_id=(
                    np.int16(
                        row[
                            "model_class_id"
                        ]
                    )
                ),

                detection_rate=(
                    np.float32(
                        debug[
                            "detection_rate"
                        ]
                    )
                ),

                full_video_count=(
                    np.int16(
                        debug[
                            "full_video_count"
                        ]
                    )
                ),

                fallback_count=(
                    np.int16(
                        debug[
                            "fallback_count"
                        ]
                    )
                ),

                missing_count=(
                    np.int16(
                        debug[
                            "missing_count"
                        ]
                    )
                )
            )


            successes += 1


        except Exception as e:

            failures.append({

                "index":
                    row_index,

                "split":
                    row["split"],

                "word":
                    row["word"],

                "video_clip_path":
                    row[
                        "video_clip_path"
                    ],

                "error":
                    repr(e)
            })


            if verbose_fail:

                print()
                print(
                    "FAILED"
                )

                print(
                    "Word :",
                    row["word"]
                )

                print(
                    "Video:",
                    row[
                        "video_clip_path"
                    ]
                )

                print(
                    "Error:",
                    repr(e)
                )


    # ========================================================
    # SUMMARY
    # ========================================================

    failure_df = pd.DataFrame(
        failures
    )


    print()
    print("=" * 65)
    print("CACHE BUILD SUMMARY")
    print("=" * 65)

    print(
        f"New cache : "
        f"{successes:,}"
    )

    print(
        f"Skipped   : "
        f"{skipped:,}"
    )

    print(
        f"Failed    : "
        f"{len(failures):,}"
    )


    if len(failure_df) > 0:

        if "ROOT" in globals():

            failure_path = (
                Path(ROOT)
                / "mouth_cache_failures.csv"
            )


            failure_df.to_csv(
                failure_path,
                index=False,
                encoding="utf-8-sig"
            )


            print()
            print(
                "Failure report:"
            )

            print(
                failure_path
            )


    return failure_df


# ============================================================
# READY
# ============================================================

print()
print("=" * 65)
print("CELL 7 READY")
print("=" * 65)

print(
    "Detection strategy:"
)

print(
    "1. Full frame / VIDEO mode"
)

print(
    "2. Multi-region crop / IMAGE mode"
)

print(
    "3. Crop upscale x",
    FALLBACK_UPSCALE
)

print(
    "4. Missing-frame interpolation"
)

print(
    "5. EMA mouth stabilization"
)

print(
    f"6. Output = "
    f"{TARGET_FRAMES} x "
    f"{ROI_SIZE} x "
    f"{ROI_SIZE} x 3"
)

print()
print(
    "Next: run Cell 7.1 single-clip test."
)

In [ ]:
# ============================================================
# CELL 7.1 : SINGLE-CLIP MOUTH ROI TEST — FIXED
# Compatible with current Cell 7
# ============================================================

import matplotlib.pyplot as plt


# ------------------------------------------------------------
# SELECT SAME SAMPLE EVERY RUN
# ------------------------------------------------------------

row = (
    df.sample(
        1,
        random_state=42
    )
    .iloc[0]
)


print("=" * 65)
print("SINGLE CLIP TEST")
print("=" * 65)

print(
    "Word :",
    row["word"]
)

print(
    "Class:",
    row["model_class_id"]
)

print(
    "Video:"
)

print(
    row["video_clip_path"]
)


# ------------------------------------------------------------
# EXTRACT
# ------------------------------------------------------------

seq, debug = extract_mouth_sequence(
    row["video_clip_path"],
    return_debug=True
)


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print()
print("=" * 65)
print("DETECTION RESULT")
print("=" * 65)

print(
    "Output shape    :",
    seq.shape
)

print(
    "dtype           :",
    seq.dtype
)

print(
    "Video frames    :",
    debug["frames"]
)

print(
    "Detected frames :",
    debug["detected_frames"]
)

print(
    "Detection rate  :",
    f'{debug["detection_rate"]:.2%}'
)

print(
    "Full-frame      :",
    debug["full_video_count"]
)

print(
    "Fallback crop   :",
    debug["fallback_count"]
)

print(
    "Missing         :",
    debug["missing_count"]
)


# ------------------------------------------------------------
# DETECTION METHOD DETAIL
# ------------------------------------------------------------

print()
print("Detection methods:")

method_counts = {}

for method in debug["methods"]:
    method_counts[method] = (
        method_counts.get(method, 0) + 1
    )

for method, count in method_counts.items():

    print(
        f"  {method:<22}: {count}"
    )


# ------------------------------------------------------------
# DISPLAY 5 ROI FRAMES
# ------------------------------------------------------------

indices = [
    0,
    4,
    8,
    12,
    15
]


fig, axes = plt.subplots(
    1,
    len(indices),
    figsize=(15, 3.5)
)


for ax, frame_idx in zip(
    axes,
    indices
):

    ax.imshow(
        seq[frame_idx]
    )

    ax.set_title(
        f"Frame {frame_idx}"
    )

    ax.axis(
        "off"
    )


plt.suptitle(
    f'Mouth ROI — {row["word"]}',
    fontsize=14
)

plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# BASIC AUTOMATIC CHECK
# ------------------------------------------------------------

print()
print("=" * 65)
print("BASIC QUALITY CHECK")
print("=" * 65)


if seq.shape == (
    TARGET_FRAMES,
    ROI_SIZE,
    ROI_SIZE,
    3
):

    print(
        "✅ Output shape correct"
    )

else:

    print(
        "❌ Output shape incorrect"
    )


if debug["detected_frames"] > 0:

    print(
        "✅ Face / mouth detected"
    )

else:

    print(
        "❌ No detection"
    )


if debug["detection_rate"] >= 0.80:

    print(
        "✅ Detection rate is strong"
    )

elif debug["detection_rate"] >= 0.50:

    print(
        "⚠️ Detection rate is moderate"
    )

else:

    print(
        "⚠️ Detection rate is low"
    )


if debug["missing_count"] == 0:

    print(
        "✅ No missing landmark frames"
    )

else:

    print(
        f'ℹ️ Missing {debug["missing_count"]} frame(s); '
        'temporal interpolation was used'
    )


# ------------------------------------------------------------
# FINAL VISUAL INSTRUCTION
# ------------------------------------------------------------

print()
print("=" * 65)
print("VISUAL CHECK")
print("=" * 65)

print(
    "ตรวจภาพ ROI ด้านบนว่า:"
)

print(
    "1. ปากอยู่บริเวณกลางภาพ"
)

print(
    "2. ริมฝีปากบนและล่างไม่ถูกตัด"
)

print(
    "3. ROI ไม่กระโดดไปตา/จมูก/คาง"
)

print(
    "4. ตำแหน่งปากค่อนข้างนิ่งตลอด sequence"
)

print(
    "5. ยังเห็นการเปิด-ปิดและรูปทรงริมฝีปากชัดเจน"
)

print()
print(
    "ถ้าภาพถูกต้อง ส่ง output + ภาพ ROI มาให้ตรวจ "
    "ก่อนสร้าง cache ทั้ง 2,253 คลิป"
)

In [ ]:
# ============================================================
# CELL 7.2 : MULTI-CLIP MOUTH ROI QUALITY AUDIT
# VocaLip Thai — Word-Level Baseline
#
# Purpose:
# - Test multiple clips before building all 2,253 caches
# - Cover train / val / test
# - Inspect detection rate
# - Inspect fallback usage
# - Visualize mouth ROI montage
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm


# ============================================================
# CONFIG
# ============================================================

N_PER_SPLIT = 4

RANDOM_STATE = 42

DISPLAY_FRAME_INDEX = 8

MIN_ACCEPTABLE_DETECTION_RATE = 0.80


# ============================================================
# SELECT TEST CLIPS
# ============================================================

audit_parts = []

for split_name in ["train", "val", "test"]:

    temp = df[
        df["split"] == split_name
    ].copy()

    n_take = min(
        N_PER_SPLIT,
        len(temp)
    )

    if n_take == 0:
        continue

    sampled = temp.sample(
        n=n_take,
        random_state=(
            RANDOM_STATE
            + len(audit_parts)
        )
    )

    audit_parts.append(
        sampled
    )


audit_df = pd.concat(
    audit_parts,
    ignore_index=True
)


print("=" * 70)
print("MULTI-CLIP MOUTH ROI AUDIT")
print("=" * 70)

print(
    f"Testing {len(audit_df)} clips"
)

print(
    f"N per split = {N_PER_SPLIT}"
)

print()


# ============================================================
# RUN EXTRACTION
# ============================================================

results = []

preview_sequences = []


for audit_index, row in tqdm(

    audit_df.iterrows(),

    total=len(audit_df),

    desc="Auditing mouth ROI"
):

    try:

        seq, debug = extract_mouth_sequence(

            row["video_clip_path"],

            return_debug=True
        )


        status = "OK"


        if (
            debug["detection_rate"]
            < MIN_ACCEPTABLE_DETECTION_RATE
        ):

            status = "LOW_DETECTION"


        results.append({

            "audit_index":
                audit_index,

            "split":
                row["split"],

            "word":
                row["word"],

            "class_id":
                int(
                    row["model_class_id"]
                ),

            "frames":
                debug["frames"],

            "detected_frames":
                debug["detected_frames"],

            "detection_rate":
                debug["detection_rate"],

            "full_frame":
                debug["full_video_count"],

            "fallback":
                debug["fallback_count"],

            "missing":
                debug["missing_count"],

            "status":
                status,

            "video_clip_path":
                row["video_clip_path"]
        })


        preview_sequences.append(
            seq
        )


    except Exception as e:

        results.append({

            "audit_index":
                audit_index,

            "split":
                row["split"],

            "word":
                row["word"],

            "class_id":
                int(
                    row["model_class_id"]
                ),

            "frames":
                np.nan,

            "detected_frames":
                0,

            "detection_rate":
                0.0,

            "full_frame":
                0,

            "fallback":
                0,

            "missing":
                np.nan,

            "status":
                "FAILED",

            "video_clip_path":
                row["video_clip_path"],

            "error":
                repr(e)
        })


        preview_sequences.append(
            None
        )


# ============================================================
# RESULTS TABLE
# ============================================================

audit_results = pd.DataFrame(
    results
)


print()
print("=" * 70)
print("AUDIT RESULTS")
print("=" * 70)

display(
    audit_results[
        [
            "split",
            "word",
            "class_id",
            "frames",
            "detected_frames",
            "detection_rate",
            "full_frame",
            "fallback",
            "missing",
            "status"
        ]
    ]
)


# ============================================================
# SUMMARY
# ============================================================

total_tested = len(
    audit_results
)

ok_count = int(
    (
        audit_results[
            "status"
        ] == "OK"
    ).sum()
)

low_detection_count = int(
    (
        audit_results[
            "status"
        ] == "LOW_DETECTION"
    ).sum()
)

failed_count = int(
    (
        audit_results[
            "status"
        ] == "FAILED"
    ).sum()
)


valid_detection = audit_results[
    audit_results["status"] != "FAILED"
]


if len(valid_detection) > 0:

    mean_detection_rate = float(
        valid_detection[
            "detection_rate"
        ].mean()
    )

    min_detection_rate = float(
        valid_detection[
            "detection_rate"
        ].min()
    )

else:

    mean_detection_rate = 0.0
    min_detection_rate = 0.0


print()
print("=" * 70)
print("SUMMARY")
print("=" * 70)

print(
    f"Total tested      : {total_tested}"
)

print(
    f"OK                : {ok_count}"
)

print(
    f"Low detection     : {low_detection_count}"
)

print(
    f"Failed            : {failed_count}"
)

print(
    f"Mean detect rate  : "
    f"{mean_detection_rate:.2%}"
)

print(
    f"Min detect rate   : "
    f"{min_detection_rate:.2%}"
)


# ============================================================
# FALLBACK STATISTICS
# ============================================================

if len(valid_detection) > 0:

    total_detected_frames = int(
        valid_detection[
            "detected_frames"
        ].sum()
    )

    total_full = int(
        valid_detection[
            "full_frame"
        ].sum()
    )

    total_fallback = int(
        valid_detection[
            "fallback"
        ].sum()
    )


    print()
    print("=" * 70)
    print("DETECTION METHOD SUMMARY")
    print("=" * 70)

    print(
        "Detected frames :",
        total_detected_frames
    )

    print(
        "Full-frame      :",
        total_full
    )

    print(
        "Fallback        :",
        total_fallback
    )


    if total_detected_frames > 0:

        print(
            "Fallback usage :",
            f"{total_fallback / total_detected_frames:.2%}"
        )


# ============================================================
# VISUAL MONTAGE
# ============================================================

n = len(
    preview_sequences
)

fig, axes = plt.subplots(

    nrows=n,

    ncols=3,

    figsize=(
        9,
        max(
            3,
            n * 2.6
        )
    )
)


if n == 1:

    axes = np.expand_dims(
        axes,
        axis=0
    )


preview_indices = [
    2,
    DISPLAY_FRAME_INDEX,
    13
]


for i in range(n):

    row = audit_df.iloc[i]

    seq = preview_sequences[i]


    for j, frame_idx in enumerate(
        preview_indices
    ):

        ax = axes[i, j]


        if seq is not None:

            frame_idx_safe = min(
                frame_idx,
                len(seq) - 1
            )

            ax.imshow(
                seq[
                    frame_idx_safe
                ]
            )

            ax.set_title(
                f'{row["split"]} | '
                f'{row["word"]} | '
                f'F{frame_idx_safe}'
            )

        else:

            ax.text(
                0.5,
                0.5,
                "FAILED",
                ha="center",
                va="center"
            )


        ax.axis(
            "off"
        )


plt.tight_layout()

plt.show()


# ============================================================
# FAILED DETAILS
# ============================================================

failed_rows = audit_results[
    audit_results["status"] == "FAILED"
]


if len(failed_rows) > 0:

    print()
    print("=" * 70)
    print("FAILED CLIPS")
    print("=" * 70)

    display(
        failed_rows[
            [
                "split",
                "word",
                "video_clip_path",
                "error"
            ]
        ]
    )


# ============================================================
# FINAL DECISION MESSAGE
# ============================================================

print()
print("=" * 70)
print("AUDIT DECISION")
print("=" * 70)


pass_rate = (
    ok_count
    / total_tested
    if total_tested > 0
    else 0.0
)


if (
    failed_count == 0
    and
    low_detection_count == 0
    and
    mean_detection_rate >= 0.90
):

    print(
        "✅ Automatic audit passed."
    )

    print(
        "If the displayed mouth ROIs also look correct,"
    )

    print(
        "the preprocessing pipeline is ready for Cell 7.3."
    )

else:

    print(
        "⚠️ Do NOT build the full cache yet."
    )

    print(
        "Inspect failed / low-detection clips and ROI images first."
    )


print()
print(
    "IMPORTANT:"
)

print(
    "Visual quality is still required."
)

print(
    "Detection rate alone cannot confirm that "
    "the correct person's mouth was cropped."
)

In [ ]:
# ============================================================
# CACHE VALIDATION — PREPROCESSING SAFE VERSION
# ไม่มี RuntimeError ระหว่างขั้นเตรียม cache
# ============================================================

from pathlib import Path
import numpy as np

# ------------------------------------------------------------
# CHECK CACHE FILES
# ------------------------------------------------------------

df["mouth_cache_exists"] = df["mouth_cache_path"].apply(
    lambda p: Path(str(p)).exists()
)

total = len(df)
ready = int(df["mouth_cache_exists"].sum())
missing = total - ready

print("=" * 70)
print("MOUTH CACHE STATUS")
print("=" * 70)

print(
    f"Cache ready : {ready:,}/{total:,} "
    f"({ready / total * 100:.2f}%)"
)

print(
    f"Missing     : {missing:,}"
)

print()


# ------------------------------------------------------------
# SPLIT STATUS
# ------------------------------------------------------------

print("Split status:")

for split_name in ["train", "val", "test"]:

    temp = df[
        df["split"] == split_name
    ]

    n_total = len(temp)

    n_ready = int(
        temp["mouth_cache_exists"].sum()
    )

    n_missing = n_total - n_ready

    print(
        f"{split_name:<5} "
        f"ready={n_ready:4d} "
        f"missing={n_missing:4d} "
        f"total={n_total:4d}"
    )


# ------------------------------------------------------------
# MISSING CACHE
# ------------------------------------------------------------

missing_cache = df[
    ~df["mouth_cache_exists"]
].copy()


print()
print("=" * 70)

if len(missing_cache) > 0:

    print("⚠️ CACHE IS NOT COMPLETE YET")
    print()
    print("This is normal before mouth preprocessing.")
    print()
    print("DO NOT TRAIN YET.")
    print()
    print("Continue with:")
    print("  Cell 7   -> load mouth extractor")
    print("  Cell 7.1 -> test one clip")
    print("  Cell 7.2 -> test multiple clips")
    print("  Cell 7.3 -> build full cache")
    print()

    print("Example missing rows:")

    cols = [
        c for c in [
            "split",
            "word",
            "model_class_id",
            "video_clip_path",
            "mouth_cache_path"
        ]
        if c in missing_cache.columns
    ]

    display(
        missing_cache[cols].head(20)
    )

else:

    print("✅ CACHE COMPLETE")
    print("All mouth cache files are available.")


# ------------------------------------------------------------
# CHECK ONE EXISTING CACHE ONLY IF AVAILABLE
# ------------------------------------------------------------

existing_df = df[
    df["mouth_cache_exists"]
].copy()


if len(existing_df) > 0:

    print()
    print("=" * 70)
    print("SAMPLE CACHE INSPECTION")
    print("=" * 70)

    sample_path = Path(
        str(
            existing_df.iloc[0][
                "mouth_cache_path"
            ]
        )
    )

    print("File:")
    print(sample_path)

    try:

        with np.load(
            sample_path,
            allow_pickle=False
        ) as data:

            print(
                "Keys:",
                list(data.files)
            )

            if "frames" in data.files:
                frames = data["frames"]
            else:
                frames = data[
                    data.files[0]
                ]

        print(
            "Shape:",
            frames.shape
        )

        print(
            "dtype:",
            frames.dtype
        )

        expected = (
            TARGET_FRAMES,
            ROI_SIZE,
            ROI_SIZE,
            3
        )

        if frames.shape == expected:
            print("✅ Shape correct")
        else:
            print("⚠️ Shape mismatch")
            print("Expected:", expected)

    except Exception as e:

        print(
            "⚠️ Cache inspection failed:"
        )

        print(
            repr(e)
        )


# ------------------------------------------------------------
# END
# ------------------------------------------------------------

print()
print("=" * 70)
print("CACHE STATUS CHECK COMPLETE")
print("=" * 70)

if len(missing_cache) > 0:

    print(
        "✅ Notebook may continue to preprocessing."
    )

else:

    print(
        "✅ Cache is ready."
    )

In [ ]:
# ============================================================
# CELL 7.3 : BUILD FULL MOUTH CACHE
# VocaLip Thai — Word-Level Baseline
#
# Purpose:
# - Build mouth ROI cache for all 2,253 samples
# - Resume-safe: skip existing cache automatically
# - Save failures to CSV
# - Show progress + final statistics
# ============================================================

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd

from tqdm.auto import tqdm


# ============================================================
# SUPPRESS MATPLOTLIB THAI GLYPH WARNINGS
# ไม่เกี่ยวกับ preprocessing/model
# ============================================================

warnings.filterwarnings(
    "ignore",
    message=r"Glyph .* missing from font"
)


# ============================================================
# CONFIG
# ============================================================

FAILURE_REPORT_PATH = (
    Path(ROOT)
    / "mouth_cache_v3_failures.csv"
)

PROGRESS_REPORT_PATH = (
    Path(ROOT)
    / "mouth_cache_v3_build_summary.csv"
)


print("=" * 72)
print("FULL MOUTH CACHE BUILD")
print("=" * 72)

print(
    f"Samples : {len(df):,}"
)

print(
    f"Cache root : {CACHE_ROOT}"
)

print(
    f"Target sequence : "
    f"{TARGET_FRAMES} x "
    f"{ROI_SIZE} x "
    f"{ROI_SIZE} x 3"
)

print()


# ============================================================
# REFRESH CACHE STATUS BEFORE START
# ============================================================

df["mouth_cache_exists"] = df[
    "mouth_cache_path"
].apply(
    lambda p: Path(str(p)).exists()
)


already_ready = int(
    df["mouth_cache_exists"].sum()
)

remaining = (
    len(df)
    - already_ready
)


print(
    f"Already cached : {already_ready:,}"
)

print(
    f"Need to build  : {remaining:,}"
)

print()


if remaining == 0:

    print(
        "✅ All cache files already exist."
    )

else:

    print(
        "Starting preprocessing..."
    )


# ============================================================
# BUILD CACHE
# ============================================================

start_time = time.time()


failures = []

new_cache_count = 0
skipped_count = 0


# Detection statistics
total_video_frames = 0
total_detected_frames = 0

total_full_frame = 0
total_fallback = 0
total_missing = 0


for row_index, row in tqdm(

    df.iterrows(),

    total=len(df),

    desc="Building mouth cache"
):

    cache_path = Path(
        str(
            row["mouth_cache_path"]
        )
    )


    # --------------------------------------------------------
    # RESUME SUPPORT
    # --------------------------------------------------------

    if cache_path.exists():

        skipped_count += 1

        continue


    cache_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )


    try:

        seq, debug = extract_mouth_sequence(

            row["video_clip_path"],

            return_debug=True
        )


        # ----------------------------------------------------
        # OUTPUT VALIDATION
        # ----------------------------------------------------

        expected_shape = (
            TARGET_FRAMES,
            ROI_SIZE,
            ROI_SIZE,
            3
        )


        if seq.shape != expected_shape:

            raise RuntimeError(
                f"BAD_CACHE_SHAPE: "
                f"{seq.shape}; "
                f"expected {expected_shape}"
            )


        if seq.dtype != np.uint8:

            seq = seq.astype(
                np.uint8
            )


        # ----------------------------------------------------
        # SAVE NPZ
        # ----------------------------------------------------

        np.savez_compressed(

            cache_path,

            frames=seq,

            word=str(
                row["word"]
            ),

            model_class_id=np.int16(
                row["model_class_id"]
            ),

            split=str(
                row["split"]
            ),

            detection_rate=np.float32(
                debug["detection_rate"]
            ),

            detected_frames=np.int16(
                debug["detected_frames"]
            ),

            video_frames=np.int16(
                debug["frames"]
            ),

            full_video_count=np.int16(
                debug["full_video_count"]
            ),

            fallback_count=np.int16(
                debug["fallback_count"]
            ),

            missing_count=np.int16(
                debug["missing_count"]
            )
        )


        # ----------------------------------------------------
        # STATISTICS
        # ----------------------------------------------------

        new_cache_count += 1


        total_video_frames += int(
            debug["frames"]
        )

        total_detected_frames += int(
            debug["detected_frames"]
        )

        total_full_frame += int(
            debug["full_video_count"]
        )

        total_fallback += int(
            debug["fallback_count"]
        )

        total_missing += int(
            debug["missing_count"]
        )


    except Exception as e:

        failures.append({

            "row_index":
                row_index,

            "split":
                row.get(
                    "split",
                    ""
                ),

            "word":
                row.get(
                    "word",
                    ""
                ),

            "model_class_id":
                row.get(
                    "model_class_id",
                    ""
                ),

            "video_clip_path":
                row.get(
                    "video_clip_path",
                    ""
                ),

            "mouth_cache_path":
                str(
                    cache_path
                ),

            "error":
                repr(e)
        })


# ============================================================
# SAVE FAILURE REPORT
# ============================================================

failure_df = pd.DataFrame(
    failures
)


if len(failure_df) > 0:

    failure_df.to_csv(

        FAILURE_REPORT_PATH,

        index=False,

        encoding="utf-8-sig"
    )


# ============================================================
# REFRESH FINAL CACHE STATUS
# ============================================================

df["mouth_cache_exists"] = df[
    "mouth_cache_path"
].apply(
    lambda p: Path(str(p)).exists()
)


final_ready = int(
    df["mouth_cache_exists"].sum()
)

final_missing = (
    len(df)
    - final_ready
)


elapsed = (
    time.time()
    - start_time
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 72)
print("FULL CACHE BUILD SUMMARY")
print("=" * 72)

print(
    f"New cache created : "
    f"{new_cache_count:,}"
)

print(
    f"Skipped existing  : "
    f"{skipped_count:,}"
)

print(
    f"Failures          : "
    f"{len(failure_df):,}"
)

print()

print(
    f"Final cache ready : "
    f"{final_ready:,}/{len(df):,}"
)

print(
    f"Final missing     : "
    f"{final_missing:,}"
)

print()

print(
    f"Elapsed time      : "
    f"{elapsed / 60:.2f} min"
)


# ============================================================
# DETECTION STATISTICS
# ============================================================

if total_video_frames > 0:

    overall_detection_rate = (
        total_detected_frames
        / total_video_frames
    )


    print()
    print("=" * 72)
    print("DETECTION STATISTICS — NEWLY BUILT CACHE")
    print("=" * 72)

    print(
        f"Video frames      : "
        f"{total_video_frames:,}"
    )

    print(
        f"Detected frames   : "
        f"{total_detected_frames:,}"
    )

    print(
        f"Overall detection : "
        f"{overall_detection_rate:.2%}"
    )

    print(
        f"Full-frame        : "
        f"{total_full_frame:,}"
    )

    print(
        f"Fallback          : "
        f"{total_fallback:,}"
    )

    print(
        f"Missing/interp.   : "
        f"{total_missing:,}"
    )


    if total_detected_frames > 0:

        fallback_usage = (
            total_fallback
            / total_detected_frames
        )

        print(
            f"Fallback usage    : "
            f"{fallback_usage:.2%}"
        )


# ============================================================
# SPLIT CACHE STATUS
# ============================================================

print()
print("=" * 72)
print("CACHE STATUS BY SPLIT")
print("=" * 72)


summary_rows = []


for split_name in [
    "train",
    "val",
    "test"
]:

    temp = df[
        df["split"] == split_name
    ]


    n_total = len(
        temp
    )

    n_ready = int(
        temp[
            "mouth_cache_exists"
        ].sum()
    )

    n_missing = (
        n_total
        - n_ready
    )


    pct = (
        n_ready
        / n_total
        * 100
        if n_total > 0
        else 0
    )


    print(
        f"{split_name:<5} "
        f"ready={n_ready:4d} "
        f"missing={n_missing:4d} "
        f"total={n_total:4d} "
        f"({pct:6.2f}%)"
    )


    summary_rows.append({

        "split":
            split_name,

        "ready":
            n_ready,

        "missing":
            n_missing,

        "total":
            n_total,

        "percent":
            pct
    })


# ============================================================
# SAVE BUILD SUMMARY
# ============================================================

pd.DataFrame(
    summary_rows
).to_csv(

    PROGRESS_REPORT_PATH,

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# FAILURE DETAILS
# ============================================================

if len(failure_df) > 0:

    print()
    print("=" * 72)
    print("FAILED CLIPS")
    print("=" * 72)

    display(
        failure_df[
            [
                "split",
                "word",
                "video_clip_path",
                "error"
            ]
        ].head(30)
    )

    print()

    print(
        "Failure report saved to:"
    )

    print(
        FAILURE_REPORT_PATH
    )


# ============================================================
# FINAL DECISION
# ============================================================

print()
print("=" * 72)
print("FINAL STATUS")
print("=" * 72)


if final_missing == 0:

    print(
        "✅ ALL 2,253 MOUTH CACHE FILES ARE READY"
    )

    print()

    print(
        "Next step: run Cell 8 "
        "Final Cache Validation."
    )


elif final_missing <= 10:

    print(
        f"⚠️ Almost complete: "
        f"{final_missing} cache file(s) failed."
    )

    print()

    print(
        "Inspect the failed clips before training."
    )


else:

    print(
        f"❌ Cache incomplete: "
        f"{final_missing:,} samples missing."
    )

    print()

    print(
        "Do NOT train yet."
    )

    print(
        "Inspect failure report first."
    )

In [ ]:
# ============================================================
# CELL 7.4 : RETRY FAILED CACHE ONLY
# VocaLip Thai — Word-Level Baseline
#
# Purpose:
# - Retry ONLY failed clips from Cell 7.3
# - Do not rebuild existing 2,245 caches
# - Use stronger fallback search for difficult clips
# - Save repaired cache if successful
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp

from tqdm.auto import tqdm

from mediapipe.tasks import python
from mediapipe.tasks.python import vision


# ============================================================
# LOAD FAILURE REPORT
# ============================================================

FAILURE_REPORT_PATH = (
    Path(ROOT)
    / "mouth_cache_v3_failures.csv"
)

assert FAILURE_REPORT_PATH.exists(), (
    f"Failure report not found:\n"
    f"{FAILURE_REPORT_PATH}"
)

failed_df = pd.read_csv(
    FAILURE_REPORT_PATH
)

print("=" * 72)
print("FAILED CACHE RETRY")
print("=" * 72)

print(
    f"Failed clips found : "
    f"{len(failed_df):,}"
)

display(
    failed_df[
        [
            "split",
            "word",
            "model_class_id",
            "video_clip_path",
            "error"
        ]
    ]
)


# ============================================================
# STRONG RETRY CONFIG
# ============================================================

RETRY_UPSCALES = [
    2.5,
    3.0,
    3.5,
    4.0
]

RETRY_MIN_DETECTION_CONF = 0.15
RETRY_MIN_PRESENCE_CONF = 0.15


# ============================================================
# CREATE RETRY IMAGE LANDMARKER
# ============================================================

def create_retry_landmarker():

    base_options = python.BaseOptions(
        model_asset_path=str(
            FACE_LANDMARKER_MODEL
        )
    )

    options = vision.FaceLandmarkerOptions(
        base_options=base_options,

        running_mode=vision.RunningMode.IMAGE,

        num_faces=1,

        min_face_detection_confidence=(
            RETRY_MIN_DETECTION_CONF
        ),

        min_face_presence_confidence=(
            RETRY_MIN_PRESENCE_CONF
        ),

        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False
    )

    return vision.FaceLandmarker.create_from_options(
        options
    )


# ============================================================
# MORE AGGRESSIVE SEARCH REGIONS
# ============================================================

def get_retry_regions(
    frame_width,
    frame_height
):

    w = frame_width
    h = frame_height

    return [

        # Full frame
        (
            0,
            0,
            w,
            h
        ),

        # Upper center
        (
            int(w * 0.10),
            0,
            int(w * 0.90),
            int(h * 0.80)
        ),

        # Upper left
        (
            0,
            0,
            int(w * 0.70),
            int(h * 0.80)
        ),

        # Upper right
        (
            int(w * 0.30),
            0,
            w,
            int(h * 0.80)
        ),

        # Center
        (
            int(w * 0.10),
            int(h * 0.05),
            int(w * 0.90),
            int(h * 0.95)
        ),

        # Left half
        (
            0,
            0,
            int(w * 0.55),
            h
        ),

        # Right half
        (
            int(w * 0.45),
            0,
            w,
            h
        ),

        # Central narrow
        (
            int(w * 0.20),
            0,
            int(w * 0.80),
            h
        )
    ]


# ============================================================
# RETRY ONE FRAME
# ============================================================

def retry_detect_mouth_box(
    retry_landmarker,
    rgb
):

    frame_h, frame_w = (
        rgb.shape[:2]
    )

    regions = get_retry_regions(
        frame_w,
        frame_h
    )


    for upscale in RETRY_UPSCALES:

        for region_index, region in enumerate(
            regions
        ):

            x1, y1, x2, y2 = region

            crop = rgb[
                y1:y2,
                x1:x2
            ]

            if crop.size == 0:
                continue

            crop_h, crop_w = (
                crop.shape[:2]
            )

            up_w = int(
                round(
                    crop_w
                    * upscale
                )
            )

            up_h = int(
                round(
                    crop_h
                    * upscale
                )
            )

            if (
                up_w < 10
                or up_h < 10
            ):
                continue


            crop_up = cv2.resize(

                crop,

                (
                    up_w,
                    up_h
                ),

                interpolation=cv2.INTER_CUBIC
            )


            mp_image = mp.Image(

                image_format=(
                    mp.ImageFormat.SRGB
                ),

                data=np.ascontiguousarray(
                    crop_up
                )
            )


            result = (
                retry_landmarker.detect(
                    mp_image
                )
            )


            if not result.face_landmarks:
                continue


            landmarks = (
                result.face_landmarks[0]
            )


            cx_up, cy_up, side_up = (
                mouth_box_from_landmarks(
                    landmarks,
                    up_w,
                    up_h
                )
            )


            cx_crop = (
                cx_up
                / upscale
            )

            cy_crop = (
                cy_up
                / upscale
            )

            side_crop = (
                side_up
                / upscale
            )


            cx_full = (
                x1
                + cx_crop
            )

            cy_full = (
                y1
                + cy_crop
            )


            return {

                "cx":
                    float(cx_full),

                "cy":
                    float(cy_full),

                "side":
                    float(side_crop),

                "method":
                    (
                        f"retry_r"
                        f"{region_index}"
                        f"_x{upscale}"
                    )
            }


    return None


# ============================================================
# RETRY VIDEO EXTRACTION
# ============================================================

def extract_mouth_sequence_retry(
    video_path,
    return_debug=False
):

    video_path = Path(
        str(video_path)
    )

    if not video_path.exists():

        raise FileNotFoundError(
            f"VIDEO_NOT_FOUND: "
            f"{video_path}"
        )


    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():

        raise RuntimeError(
            f"CANNOT_OPEN_VIDEO: "
            f"{video_path}"
        )


    frames_bgr = []


    while True:

        ok, frame = cap.read()

        if not ok:
            break

        frames_bgr.append(
            frame
        )


    cap.release()


    if len(frames_bgr) == 0:

        raise RuntimeError(
            "VIDEO_HAS_NO_FRAMES"
        )


    retry_landmarker = (
        create_retry_landmarker()
    )


    boxes = []
    methods = []


    try:

        for frame_bgr in frames_bgr:

            rgb = cv2.cvtColor(
                frame_bgr,
                cv2.COLOR_BGR2RGB
            )


            detection = (
                retry_detect_mouth_box(
                    retry_landmarker,
                    rgb
                )
            )


            if detection is None:

                boxes.append(
                    (
                        np.nan,
                        np.nan,
                        np.nan
                    )
                )

                methods.append(
                    "missing"
                )

                continue


            boxes.append(
                (
                    detection["cx"],
                    detection["cy"],
                    detection["side"]
                )
            )

            methods.append(
                detection["method"]
            )


    finally:

        retry_landmarker.close()


    boxes = np.asarray(
        boxes,
        dtype=np.float32
    )


    detected_mask = np.isfinite(
        boxes[:, 0]
    )

    detected_count = int(
        detected_mask.sum()
    )

    total_frames = len(
        frames_bgr
    )


    if detected_count == 0:

        raise RuntimeError(
            "ZERO_FACE_DETECTION_RETRY"
        )


    # --------------------------------------------------------
    # INTERPOLATE
    # --------------------------------------------------------

    cx = interpolate_missing(
        boxes[:, 0]
    )

    cy = interpolate_missing(
        boxes[:, 1]
    )

    side = interpolate_missing(
        boxes[:, 2]
    )


    # --------------------------------------------------------
    # SMOOTH
    # --------------------------------------------------------

    cx = ema_smooth(
        cx,
        SMOOTH_ALPHA
    )

    cy = ema_smooth(
        cy,
        SMOOTH_ALPHA
    )

    side = ema_smooth(
        side,
        SMOOTH_ALPHA
    )


    # --------------------------------------------------------
    # CROP
    # --------------------------------------------------------

    mouth_frames = []


    for i, frame_bgr in enumerate(
        frames_bgr
    ):

        roi = crop_mouth_roi(

            frame_bgr,

            cx[i],
            cy[i],
            side[i]
        )

        mouth_frames.append(
            roi
        )


    sequence = temporal_resample(

        mouth_frames,

        TARGET_FRAMES
    )


    sequence = sequence.astype(
        np.uint8
    )


    if return_debug:

        return sequence, {

            "frames":
                total_frames,

            "detected_frames":
                detected_count,

            "detection_rate":
                (
                    detected_count
                    / total_frames
                ),

            "missing_count":
                methods.count(
                    "missing"
                ),

            "methods":
                methods
        }


    return sequence


# ============================================================
# RETRY FAILED CLIPS
# ============================================================

retry_results = []


for _, failed_row in tqdm(

    failed_df.iterrows(),

    total=len(failed_df),

    desc="Retrying failed clips"
):

    video_path = Path(
        str(
            failed_row[
                "video_clip_path"
            ]
        )
    )

    cache_path = Path(
        str(
            failed_row[
                "mouth_cache_path"
            ]
        )
    )


    # --------------------------------------------------------
    # ALREADY REPAIRED?
    # --------------------------------------------------------

    if cache_path.exists():

        retry_results.append({

            "word":
                failed_row["word"],

            "video_clip_path":
                str(video_path),

            "status":
                "ALREADY_EXISTS",

            "detection_rate":
                np.nan,

            "missing":
                np.nan,

            "error":
                ""
        })

        continue


    try:

        seq, debug = (
            extract_mouth_sequence_retry(

                video_path,

                return_debug=True
            )
        )


        expected_shape = (
            TARGET_FRAMES,
            ROI_SIZE,
            ROI_SIZE,
            3
        )


        if seq.shape != expected_shape:

            raise RuntimeError(
                f"BAD_RETRY_SHAPE: "
                f"{seq.shape}"
            )


        cache_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )


        np.savez_compressed(

            cache_path,

            frames=seq,

            word=str(
                failed_row["word"]
            ),

            model_class_id=np.int16(
                failed_row[
                    "model_class_id"
                ]
            ),

            split=str(
                failed_row["split"]
            ),

            detection_rate=np.float32(
                debug[
                    "detection_rate"
                ]
            ),

            detected_frames=np.int16(
                debug[
                    "detected_frames"
                ]
            ),

            video_frames=np.int16(
                debug[
                    "frames"
                ]
            ),

            full_video_count=np.int16(
                0
            ),

            fallback_count=np.int16(
                debug[
                    "detected_frames"
                ]
            ),

            missing_count=np.int16(
                debug[
                    "missing_count"
                ]
            ),

            retry=np.uint8(
                1
            )
        )


        retry_results.append({

            "word":
                failed_row["word"],

            "video_clip_path":
                str(video_path),

            "status":
                "REPAIRED",

            "detection_rate":
                debug[
                    "detection_rate"
                ],

            "missing":
                debug[
                    "missing_count"
                ],

            "error":
                ""
        })


    except Exception as e:

        retry_results.append({

            "word":
                failed_row["word"],

            "video_clip_path":
                str(video_path),

            "status":
                "FAILED_AGAIN",

            "detection_rate":
                0.0,

            "missing":
                np.nan,

            "error":
                repr(e)
        })


# ============================================================
# RETRY RESULTS
# ============================================================

retry_results_df = pd.DataFrame(
    retry_results
)


print()
print("=" * 72)
print("RETRY RESULTS")
print("=" * 72)

display(
    retry_results_df
)


# ============================================================
# FINAL CACHE COUNT
# ============================================================

df["mouth_cache_exists"] = df[
    "mouth_cache_path"
].apply(
    lambda p: Path(
        str(p)
    ).exists()
)


final_ready = int(
    df[
        "mouth_cache_exists"
    ].sum()
)

final_missing = (
    len(df)
    - final_ready
)


print()
print("=" * 72)
print("FINAL CACHE STATUS")
print("=" * 72)

print(
    f"Ready   : "
    f"{final_ready:,}/{len(df):,}"
)

print(
    f"Missing : "
    f"{final_missing:,}"
)


# ============================================================
# SAVE REMAINING FAILURES
# ============================================================

still_failed = retry_results_df[
    retry_results_df[
        "status"
    ] == "FAILED_AGAIN"
].copy()


RETRY_FAILURE_PATH = (
    Path(ROOT)
    / "mouth_cache_v3_retry_failures.csv"
)


if len(still_failed) > 0:

    still_failed.to_csv(

        RETRY_FAILURE_PATH,

        index=False,

        encoding="utf-8-sig"
    )

    print()
    print(
        "Remaining failure report:"
    )

    print(
        RETRY_FAILURE_PATH
    )


# ============================================================
# DECISION
# ============================================================

print()
print("=" * 72)

if final_missing == 0:

    print(
        "✅ ALL CACHE FILES ARE NOW READY"
    )

    print()
    print(
        "Next step: Cell 8 Final Cache Validation"
    )

else:

    print(
        f"⚠️ Remaining failed clips: "
        f"{final_missing}"
    )

    print()
    print(
        "Do not rebuild the entire cache."
    )

    print(
        "Inspect only the remaining failed clips."
    )

print("=" * 72)

In [ ]:
# ============================================================
# CELL 7.5 : INSPECT REMAINING FAILED CLIPS
# VocaLip Thai — Word-Level Baseline
#
# Purpose:
# - Inspect ONLY the remaining 8 failed clips
# - Check whether video files are valid
# - Check frame count / FPS / resolution
# - Display beginning / middle / end frames
# - Help decide:
#       A) repair preprocessing
#       B) exclude unusable samples
#
# IMPORTANT:
# This cell DOES NOT delete or modify dataset files.
# ============================================================

from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# PATH
# ============================================================

RETRY_FAILURE_PATH = (
    Path(ROOT)
    / "mouth_cache_v3_retry_failures.csv"
)


assert RETRY_FAILURE_PATH.exists(), (
    f"Retry failure report not found:\n"
    f"{RETRY_FAILURE_PATH}"
)


# ============================================================
# LOAD FAILURES
# ============================================================

failed8 = pd.read_csv(
    RETRY_FAILURE_PATH
)


print("=" * 78)
print("REMAINING FAILED CLIP INSPECTION")
print("=" * 78)

print(
    f"Remaining failed clips : "
    f"{len(failed8):,}"
)

print()


display(
    failed8
)


# ============================================================
# VIDEO INSPECTION FUNCTION
# ============================================================

def inspect_video_file(
    video_path
):

    video_path = Path(
        str(video_path)
    )


    result = {

        "video_exists":
            video_path.exists(),

        "video_opened":
            False,

        "frame_count":
            0,

        "fps":
            np.nan,

        "width":
            0,

        "height":
            0,

        "frames_read":
            0,

        "status":
            "",

        "sample_frames":
            []
    }


    # --------------------------------------------------------
    # FILE DOES NOT EXIST
    # --------------------------------------------------------

    if not video_path.exists():

        result[
            "status"
        ] = "FILE_NOT_FOUND"

        return result


    # --------------------------------------------------------
    # OPEN VIDEO
    # --------------------------------------------------------

    cap = cv2.VideoCapture(
        str(video_path)
    )


    if not cap.isOpened():

        result[
            "status"
        ] = "CANNOT_OPEN_VIDEO"

        return result


    result[
        "video_opened"
    ] = True


    # --------------------------------------------------------
    # METADATA
    # --------------------------------------------------------

    frame_count_meta = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )


    fps = cap.get(
        cv2.CAP_PROP_FPS
    )


    width = int(
        cap.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )
    )


    height = int(
        cap.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )
    )


    result[
        "frame_count"
    ] = frame_count_meta


    result[
        "fps"
    ] = fps


    result[
        "width"
    ] = width


    result[
        "height"
    ] = height


    # --------------------------------------------------------
    # READ ALL FRAMES
    # Only 8 short clips, so this is safe.
    # --------------------------------------------------------

    frames = []


    while True:

        ok, frame = cap.read()

        if not ok:
            break


        frames.append(
            frame
        )


    cap.release()


    result[
        "frames_read"
    ] = len(
        frames
    )


    # --------------------------------------------------------
    # STATUS
    # --------------------------------------------------------

    if len(frames) == 0:

        result[
            "status"
        ] = "ZERO_READABLE_FRAMES"

        return result


    if width <= 0 or height <= 0:

        result[
            "status"
        ] = "BAD_RESOLUTION"

        return result


    # --------------------------------------------------------
    # SAMPLE 3 FRAMES
    # --------------------------------------------------------

    sample_ids = [

        0,

        len(frames) // 2,

        len(frames) - 1
    ]


    sample_frames = []


    for frame_idx in sample_ids:

        frame_bgr = frames[
            frame_idx
        ]


        frame_rgb = cv2.cvtColor(

            frame_bgr,

            cv2.COLOR_BGR2RGB
        )


        sample_frames.append(
            (
                frame_idx,
                frame_rgb
            )
        )


    result[
        "sample_frames"
    ] = sample_frames


    result[
        "status"
    ] = "VIDEO_OK"


    return result


# ============================================================
# RUN INSPECTION
# ============================================================

inspection_rows = []

inspection_visuals = []


for i, row in failed8.iterrows():

    video_path = row[
        "video_clip_path"
    ]


    info = inspect_video_file(
        video_path
    )


    inspection_rows.append({

        "index":
            i,

        "word":
            row.get(
                "word",
                ""
            ),

        "video_exists":
            info[
                "video_exists"
            ],

        "video_opened":
            info[
                "video_opened"
            ],

        "frame_count_meta":
            info[
                "frame_count"
            ],

        "frames_read":
            info[
                "frames_read"
            ],

        "fps":
            info[
                "fps"
            ],

        "width":
            info[
                "width"
            ],

        "height":
            info[
                "height"
            ],

        "video_status":
            info[
                "status"
            ],

        "previous_error":
            row.get(
                "error",
                ""
            ),

        "video_clip_path":
            video_path
    })


    inspection_visuals.append(
        info[
            "sample_frames"
        ]
    )


inspection_df = pd.DataFrame(
    inspection_rows
)


# ============================================================
# SUMMARY TABLE
# ============================================================

print()
print("=" * 78)
print("VIDEO HEALTH SUMMARY")
print("=" * 78)


display(
    inspection_df[
        [
            "index",
            "word",
            "video_exists",
            "video_opened",
            "frame_count_meta",
            "frames_read",
            "fps",
            "width",
            "height",
            "video_status",
            "previous_error"
        ]
    ]
)


# ============================================================
# STATUS COUNTS
# ============================================================

print()
print("=" * 78)
print("STATUS COUNTS")
print("=" * 78)


print(
    inspection_df[
        "video_status"
    ].value_counts(
        dropna=False
    )
)


# ============================================================
# DISPLAY FRAMES
# ============================================================

n = len(
    inspection_df
)


fig, axes = plt.subplots(

    nrows=n,

    ncols=3,

    figsize=(
        12,
        max(
            4,
            n * 3
        )
    )
)


if n == 1:

    axes = np.expand_dims(
        axes,
        axis=0
    )


for row_idx in range(n):

    visual_frames = (
        inspection_visuals[
            row_idx
        ]
    )


    word = inspection_df.iloc[
        row_idx
    ][
        "word"
    ]


    status = inspection_df.iloc[
        row_idx
    ][
        "video_status"
    ]


    # --------------------------------------------------------
    # NO READABLE FRAME
    # --------------------------------------------------------

    if len(
        visual_frames
    ) == 0:

        for col_idx in range(
            3
        ):

            ax = axes[
                row_idx,
                col_idx
            ]


            ax.text(

                0.5,
                0.5,

                status,

                ha="center",
                va="center",

                fontsize=12
            )


            ax.axis(
                "off"
            )


        continue


    # --------------------------------------------------------
    # DISPLAY START / MID / END
    # --------------------------------------------------------

    for col_idx, (
        frame_idx,
        frame_rgb

    ) in enumerate(
        visual_frames
    ):

        ax = axes[
            row_idx,
            col_idx
        ]


        ax.imshow(
            frame_rgb
        )


        # Avoid Thai glyph warning in title
        ax.set_title(
            f"Clip {row_idx} | Frame {frame_idx}"
        )


        ax.axis(
            "off"
        )


plt.tight_layout()

plt.show()


# ============================================================
# AUTOMATIC RECOMMENDATION
# ============================================================

print()
print("=" * 78)
print("AUTOMATIC DATA DECISION")
print("=" * 78)


bad_video_mask = (

    inspection_df[
        "video_status"
    ] != "VIDEO_OK"
)


bad_video_count = int(
    bad_video_mask.sum()
)


valid_video_count = (
    len(inspection_df)
    - bad_video_count
)


print(
    f"Valid video files    : "
    f"{valid_video_count}"
)

print(
    f"Invalid video files  : "
    f"{bad_video_count}"
)


print()


if bad_video_count > 0:

    print(
        "❌ Some failures are caused by invalid video files."
    )

    print(
        "Those samples should not be forced into the training set."
    )


if valid_video_count > 0:

    print(
        "ℹ️ VIDEO_OK clips still failed face landmark detection."
    )

    print(
        "Visually inspect the displayed frames."
    )

    print(
        "Check whether the target speaker's face is actually visible."
    )


print()
print("=" * 78)
print("WHAT TO CHECK IN THE 8 ROWS")
print("=" * 78)

print(
    "1. Is the target speaker visible?"
)

print(
    "2. Is the face large enough to see?"
)

print(
    "3. Is the mouth visible and not blocked?"
)

print(
    "4. Is the correct person present in the clip?"
)

print(
    "5. Is the frame blurred / black / transition?"
)

print(
    "6. Is the word clip temporally misaligned?"
)

print()
print(
    "Do NOT delete anything yet."
)

print(
    "Send VIDEO HEALTH SUMMARY + what the 8 displayed clips look like."
)

In [ ]:
# ============================================================
# CELL 7.6 : DENSE TILE RETRY FOR 8 HARD CLIPS
# VocaLip Thai — Word-Level Baseline
#
# Purpose:
# - Retry only the remaining VIDEO_OK clips
# - Search small faces using overlapping image tiles
# - Do NOT save cache yet
# - Visualize recovered mouth ROIs first
# ============================================================

from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mediapipe as mp

from tqdm.auto import tqdm

from mediapipe.tasks import python
from mediapipe.tasks.python import vision


# ============================================================
# LOAD REMAINING FAILURES
# ============================================================

RETRY_FAILURE_PATH = (
    Path(ROOT)
    / "mouth_cache_v3_retry_failures.csv"
)

assert RETRY_FAILURE_PATH.exists(), (
    f"Not found:\n{RETRY_FAILURE_PATH}"
)

hard_df = pd.read_csv(
    RETRY_FAILURE_PATH
)

print("=" * 76)
print("DENSE TILE RETRY")
print("=" * 76)

print(
    f"Hard clips : {len(hard_df)}"
)


# ============================================================
# CONFIG
# ============================================================

TILE_UPSCALE = 3.0

TILE_OVERLAP = 0.35

DENSE_MIN_DETECTION_CONF = 0.10

DENSE_MIN_PRESENCE_CONF = 0.10


# Tile sizes relative to full image
# Smaller tiles make a tiny face appear larger.
TILE_SCALES = [
    0.60,
    0.48,
    0.38,
    0.30
]


# ============================================================
# CREATE IMAGE LANDMARKER
# ============================================================

def create_dense_landmarker():

    base_options = python.BaseOptions(
        model_asset_path=str(
            FACE_LANDMARKER_MODEL
        )
    )

    options = vision.FaceLandmarkerOptions(
        base_options=base_options,

        running_mode=vision.RunningMode.IMAGE,

        num_faces=1,

        min_face_detection_confidence=(
            DENSE_MIN_DETECTION_CONF
        ),

        min_face_presence_confidence=(
            DENSE_MIN_PRESENCE_CONF
        ),

        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False
    )

    return vision.FaceLandmarker.create_from_options(
        options
    )


# ============================================================
# GENERATE OVERLAPPING TILES
# ============================================================

def generate_dense_tiles(
    frame_w,
    frame_h
):

    tiles = []


    for scale in TILE_SCALES:

        tile_w = int(
            round(
                frame_w * scale
            )
        )

        tile_h = int(
            round(
                frame_h * scale
            )
        )


        tile_w = min(
            tile_w,
            frame_w
        )

        tile_h = min(
            tile_h,
            frame_h
        )


        stride_x = max(
            1,
            int(
                tile_w
                * (
                    1.0
                    - TILE_OVERLAP
                )
            )
        )

        stride_y = max(
            1,
            int(
                tile_h
                * (
                    1.0
                    - TILE_OVERLAP
                )
            )
        )


        xs = list(
            range(
                0,
                max(
                    1,
                    frame_w - tile_w + 1
                ),
                stride_x
            )
        )

        ys = list(
            range(
                0,
                max(
                    1,
                    frame_h - tile_h + 1
                ),
                stride_y
            )
        )


        # Ensure right/bottom edge coverage
        last_x = max(
            0,
            frame_w - tile_w
        )

        last_y = max(
            0,
            frame_h - tile_h
        )


        if last_x not in xs:
            xs.append(
                last_x
            )

        if last_y not in ys:
            ys.append(
                last_y
            )


        for y1 in ys:

            for x1 in xs:

                x2 = min(
                    frame_w,
                    x1 + tile_w
                )

                y2 = min(
                    frame_h,
                    y1 + tile_h
                )


                tiles.append(
                    (
                        x1,
                        y1,
                        x2,
                        y2,
                        scale
                    )
                )


    # --------------------------------------------------------
    # Prefer upper/central tiles first
    # --------------------------------------------------------

    def tile_priority(
        tile
    ):

        x1, y1, x2, y2, scale = tile

        cx = (
            x1 + x2
        ) / 2.0

        cy = (
            y1 + y2
        ) / 2.0


        norm_x = (
            cx / frame_w
        )

        norm_y = (
            cy / frame_h
        )


        center_distance = (
            abs(
                norm_x - 0.50
            )
            +
            0.7
            * abs(
                norm_y - 0.35
            )
        )


        return center_distance


    tiles.sort(
        key=tile_priority
    )


    return tiles


# ============================================================
# DENSE DETECTION ON ONE FRAME
# ============================================================

def dense_detect_mouth(
    landmarker,
    rgb
):

    h, w = rgb.shape[:2]


    tiles = generate_dense_tiles(
        w,
        h
    )


    for tile_index, tile in enumerate(
        tiles
    ):

        x1, y1, x2, y2, scale = tile


        crop = rgb[
            y1:y2,
            x1:x2
        ]


        if crop.size == 0:
            continue


        ch, cw = crop.shape[:2]


        up_w = int(
            round(
                cw
                * TILE_UPSCALE
            )
        )

        up_h = int(
            round(
                ch
                * TILE_UPSCALE
            )
        )


        crop_up = cv2.resize(
            crop,
            (
                up_w,
                up_h
            ),
            interpolation=cv2.INTER_CUBIC
        )


        mp_image = mp.Image(
            image_format=(
                mp.ImageFormat.SRGB
            ),
            data=np.ascontiguousarray(
                crop_up
            )
        )


        result = landmarker.detect(
            mp_image
        )


        if not result.face_landmarks:
            continue


        landmarks = (
            result.face_landmarks[0]
        )


        cx_up, cy_up, side_up = (
            mouth_box_from_landmarks(
                landmarks,
                up_w,
                up_h
            )
        )


        # Upscaled tile -> original tile
        cx_tile = (
            cx_up
            / TILE_UPSCALE
        )

        cy_tile = (
            cy_up
            / TILE_UPSCALE
        )

        side_tile = (
            side_up
            / TILE_UPSCALE
        )


        # Tile -> full frame
        cx_full = (
            x1 + cx_tile
        )

        cy_full = (
            y1 + cy_tile
        )


        return {

            "cx":
                float(
                    cx_full
                ),

            "cy":
                float(
                    cy_full
                ),

            "side":
                float(
                    side_tile
                ),

            "tile_index":
                tile_index,

            "tile_scale":
                scale
        }


    return None


# ============================================================
# EXTRACT ONE HARD VIDEO
# ============================================================

def extract_dense_retry(
    video_path,
    return_debug=False
):

    video_path = Path(
        str(video_path)
    )


    cap = cv2.VideoCapture(
        str(video_path)
    )


    if not cap.isOpened():

        raise RuntimeError(
            "CANNOT_OPEN_VIDEO"
        )


    frames_bgr = []


    while True:

        ok, frame = cap.read()

        if not ok:
            break

        frames_bgr.append(
            frame
        )


    cap.release()


    if len(frames_bgr) == 0:

        raise RuntimeError(
            "NO_FRAMES"
        )


    landmarker = (
        create_dense_landmarker()
    )


    boxes = []

    detected_methods = []


    try:

        for frame_bgr in frames_bgr:

            rgb = cv2.cvtColor(
                frame_bgr,
                cv2.COLOR_BGR2RGB
            )


            result = dense_detect_mouth(
                landmarker,
                rgb
            )


            if result is None:

                boxes.append(
                    [
                        np.nan,
                        np.nan,
                        np.nan
                    ]
                )

                detected_methods.append(
                    None
                )

            else:

                boxes.append(
                    [
                        result["cx"],
                        result["cy"],
                        result["side"]
                    ]
                )

                detected_methods.append(
                    result
                )


    finally:

        landmarker.close()


    boxes = np.asarray(
        boxes,
        dtype=np.float32
    )


    detected_mask = np.isfinite(
        boxes[:, 0]
    )


    detected_count = int(
        detected_mask.sum()
    )


    total_frames = len(
        frames_bgr
    )


    if detected_count == 0:

        raise RuntimeError(
            "ZERO_FACE_DETECTION_DENSE_TILE"
        )


    # ========================================================
    # INTERPOLATE
    # ========================================================

    cx = interpolate_missing(
        boxes[:, 0]
    )

    cy = interpolate_missing(
        boxes[:, 1]
    )

    side = interpolate_missing(
        boxes[:, 2]
    )


    # ========================================================
    # SMOOTH
    # ========================================================

    cx = ema_smooth(
        cx,
        SMOOTH_ALPHA
    )

    cy = ema_smooth(
        cy,
        SMOOTH_ALPHA
    )

    side = ema_smooth(
        side,
        SMOOTH_ALPHA
    )


    # ========================================================
    # CROP
    # ========================================================

    mouth_frames = []


    for i, frame_bgr in enumerate(
        frames_bgr
    ):

        mouth_roi = crop_mouth_roi(

            frame_bgr,

            cx[i],
            cy[i],
            side[i]
        )


        mouth_frames.append(
            mouth_roi
        )


    sequence = temporal_resample(
        mouth_frames,
        TARGET_FRAMES
    )


    sequence = sequence.astype(
        np.uint8
    )


    if return_debug:

        return sequence, {

            "frames":
                total_frames,

            "detected_frames":
                detected_count,

            "detection_rate":
                detected_count
                / total_frames,

            "missing_count":
                total_frames
                - detected_count,

            "methods":
                detected_methods
        }


    return sequence


# ============================================================
# RUN ALL 8
# ============================================================

dense_results = []

dense_sequences = []


for idx, row in tqdm(

    hard_df.iterrows(),

    total=len(hard_df),

    desc="Dense tile retry"
):

    try:

        seq, debug = (
            extract_dense_retry(

                row[
                    "video_clip_path"
                ],

                return_debug=True
            )
        )


        dense_results.append({

            "index":
                idx,

            "word":
                row[
                    "word"
                ],

            "status":
                "RECOVERED",

            "frames":
                debug[
                    "frames"
                ],

            "detected_frames":
                debug[
                    "detected_frames"
                ],

            "detection_rate":
                debug[
                    "detection_rate"
                ],

            "missing":
                debug[
                    "missing_count"
                ],

            "video_clip_path":
                row[
                    "video_clip_path"
                ]
        })


        dense_sequences.append(
            seq
        )


    except Exception as e:

        dense_results.append({

            "index":
                idx,

            "word":
                row[
                    "word"
                ],

            "status":
                "FAILED",

            "frames":
                np.nan,

            "detected_frames":
                0,

            "detection_rate":
                0.0,

            "missing":
                np.nan,

            "video_clip_path":
                row[
                    "video_clip_path"
                ],

            "error":
                repr(e)
        })


        dense_sequences.append(
            None
        )


# ============================================================
# RESULT TABLE
# ============================================================

dense_results_df = pd.DataFrame(
    dense_results
)


print()
print("=" * 76)
print("DENSE TILE RESULTS")
print("=" * 76)


display(
    dense_results_df[
        [
            "index",
            "word",
            "status",
            "frames",
            "detected_frames",
            "detection_rate",
            "missing"
        ]
    ]
)


# ============================================================
# SUMMARY
# ============================================================

recovered_count = int(
    (
        dense_results_df[
            "status"
        ] == "RECOVERED"
    ).sum()
)


failed_count = int(
    (
        dense_results_df[
            "status"
        ] == "FAILED"
    ).sum()
)


print()
print("=" * 76)
print("SUMMARY")
print("=" * 76)

print(
    "Recovered :",
    recovered_count
)

print(
    "Failed    :",
    failed_count
)


# ============================================================
# VISUALIZE RECOVERED MOUTH ROI
# ============================================================

n = len(
    dense_sequences
)


fig, axes = plt.subplots(

    nrows=n,

    ncols=3,

    figsize=(
        9,
        max(
            3,
            n * 2.7
        )
    )
)


if n == 1:

    axes = np.expand_dims(
        axes,
        axis=0
    )


preview_indices = [
    2,
    8,
    13
]


for i in range(n):

    seq = dense_sequences[
        i
    ]


    status = dense_results_df.iloc[
        i
    ][
        "status"
    ]


    for j, frame_idx in enumerate(
        preview_indices
    ):

        ax = axes[
            i,
            j
        ]


        if seq is not None:

            ax.imshow(
                seq[
                    frame_idx
                ]
            )


            ax.set_title(
                f"Clip {i} | F{frame_idx}"
            )


        else:

            ax.text(

                0.5,
                0.5,

                "FAILED",

                ha="center",
                va="center"
            )


        ax.axis(
            "off"
        )


plt.tight_layout()

plt.show()


# ============================================================
# DECISION
# ============================================================

print()
print("=" * 76)
print("NEXT DECISION")
print("=" * 76)


if recovered_count > 0:

    print(
        f"✅ Dense tile search recovered "
        f"{recovered_count}/{len(hard_df)} clips."
    )

    print()
    print(
        "IMPORTANT:"
    )

    print(
        "Check the mouth ROI images carefully."
    )

    print(
        "Do not save these caches yet."
    )

    print(
        "We must confirm they belong to the correct speaker."
    )


if failed_count > 0:

    print()
    print(
        f"⚠️ {failed_count} clip(s) still failed."
    )


print()
print(
    "Send the SUMMARY and tell me whether "
    "the displayed ROIs are the correct speaker's mouth."
)

In [ ]:
# ============================================================
# CELL 7.7 : FINALIZE CLEAN BASELINE DATASET
# Exclude the 8 unrecoverable / subtitle-contaminated clips
#
# IMPORTANT:
# - Does NOT delete original videos
# - Does NOT delete rows from original manifest
# - Creates a NEW baseline manifest
# - Training will use only samples with valid mouth cache
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ============================================================
# OUTPUT PATHS
# ============================================================

BASELINE_MANIFEST_PATH = (
    Path(ROOT)
    / "word_train_manifest_v5_baseline_clean.csv"
)

EXCLUDED_MANIFEST_PATH = (
    Path(ROOT)
    / "word_train_manifest_v5_baseline_excluded.csv"
)


# ============================================================
# CHECK CACHE EXISTENCE
# ============================================================

work_df = df.copy()


work_df["mouth_cache_exists"] = (
    work_df["mouth_cache_path"]
    .apply(
        lambda p: Path(str(p)).exists()
    )
)


# ============================================================
# SPLIT VALID / EXCLUDED
# ============================================================

baseline_df = (
    work_df[
        work_df["mouth_cache_exists"]
    ]
    .copy()
    .reset_index(drop=True)
)


excluded_df = (
    work_df[
        ~work_df["mouth_cache_exists"]
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# BASIC COUNTS
# ============================================================

print("=" * 76)
print("FINAL CLEAN BASELINE DATASET")
print("=" * 76)

print(
    f"Original samples : {len(work_df):,}"
)

print(
    f"Baseline samples : {len(baseline_df):,}"
)

print(
    f"Excluded samples : {len(excluded_df):,}"
)


# ============================================================
# VERIFY CACHE
# ============================================================

assert baseline_df[
    "mouth_cache_exists"
].all(), (
    "Baseline manifest still contains missing cache."
)


# ============================================================
# CHECK CLASSES
# ============================================================

expected_classes = set(
    range(NUM_CLASSES)
)

actual_classes = set(
    baseline_df[
        "model_class_id"
    ].astype(int)
)


missing_classes = sorted(
    expected_classes
    - actual_classes
)


print()
print("=" * 76)
print("CLASS CHECK")
print("=" * 76)

print(
    f"Expected classes : {NUM_CLASSES}"
)

print(
    f"Present classes  : {len(actual_classes)}"
)

print(
    f"Missing classes  : {missing_classes}"
)


if missing_classes:

    raise RuntimeError(
        "Exclusion removed an entire class. "
        f"Missing classes: {missing_classes}"
    )


# ============================================================
# SPLIT SUMMARY
# ============================================================

print()
print("=" * 76)
print("SPLIT SUMMARY")
print("=" * 76)


split_rows = []


for split_name in [
    "train",
    "val",
    "test"
]:

    before = work_df[
        work_df["split"] == split_name
    ]

    after = baseline_df[
        baseline_df["split"] == split_name
    ]

    removed = (
        len(before)
        - len(after)
    )


    print(
        f"{split_name:<5} "
        f"before={len(before):4d} "
        f"after={len(after):4d} "
        f"excluded={removed:3d}"
    )


    split_rows.append({
        "split": split_name,
        "before": len(before),
        "after": len(after),
        "excluded": removed
    })


# ============================================================
# CHECK ALL 44 CLASSES IN EACH SPLIT
# ============================================================

print()
print("=" * 76)
print("CLASS COVERAGE BY SPLIT")
print("=" * 76)


split_class_ok = True


for split_name in [
    "train",
    "val",
    "test"
]:

    temp = baseline_df[
        baseline_df["split"] == split_name
    ]


    present = set(
        temp[
            "model_class_id"
        ].astype(int)
    )


    missing = sorted(
        expected_classes
        - present
    )


    print(
        f"{split_name:<5} "
        f"classes={len(present):2d}/{NUM_CLASSES} "
        f"missing={missing}"
    )


    if missing:
        split_class_ok = False


# ============================================================
# CLASS DISTRIBUTION
# ============================================================

class_distribution = (

    baseline_df
    .groupby(
        [
            "split",
            "model_class_id",
            "word"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
)


print()
print("=" * 76)
print("MIN / MAX SAMPLES PER CLASS")
print("=" * 76)


for split_name in [
    "train",
    "val",
    "test"
]:

    temp = class_distribution[
        class_distribution[
            "split"
        ] == split_name
    ]


    if len(temp) == 0:
        continue


    print(
        f"{split_name:<5} "
        f"min={temp['count'].min():3d} "
        f"max={temp['count'].max():3d} "
        f"mean={temp['count'].mean():6.2f}"
    )


# ============================================================
# EXCLUSION REPORT
# ============================================================

print()
print("=" * 76)
print("EXCLUDED SAMPLES")
print("=" * 76)


if len(excluded_df) > 0:

    columns_to_show = [
        c for c in [
            "split",
            "word",
            "model_class_id",
            "video_clip_path",
            "mouth_cache_path"
        ]
        if c in excluded_df.columns
    ]


    display(
        excluded_df[
            columns_to_show
        ]
    )

else:

    print(
        "No excluded samples."
    )


# ============================================================
# ADD EXCLUSION REASON
# ============================================================

if len(excluded_df) > 0:

    excluded_df[
        "baseline_exclusion_reason"
    ] = (
        "mouth_roi_detection_failed_"
        "subtitle_occlusion"
    )


# ============================================================
# DROP TEMP STATUS COLUMN FROM MAIN EXPORT
# ============================================================

baseline_export_df = (
    baseline_df
    .drop(
        columns=[
            "mouth_cache_exists"
        ],
        errors="ignore"
    )
)


excluded_export_df = (
    excluded_df
    .drop(
        columns=[
            "mouth_cache_exists"
        ],
        errors="ignore"
    )
)


# ============================================================
# SAVE
# ============================================================

baseline_export_df.to_csv(

    BASELINE_MANIFEST_PATH,

    index=False,

    encoding="utf-8-sig"
)


excluded_export_df.to_csv(

    EXCLUDED_MANIFEST_PATH,

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# FINAL VALIDATION
# ============================================================

saved_df = pd.read_csv(
    BASELINE_MANIFEST_PATH
)


assert len(saved_df) == len(
    baseline_export_df
)


assert saved_df[
    "mouth_cache_path"
].apply(
    lambda p: Path(str(p)).exists()
).all()


# ============================================================
# FINAL REPORT
# ============================================================

print()
print("=" * 76)
print("FINAL BASELINE MANIFEST")
print("=" * 76)

print(
    f"Training-ready samples : "
    f"{len(saved_df):,}"
)

print(
    f"Classes                : "
    f"{saved_df['model_class_id'].nunique()}"
)

print(
    f"Words                  : "
    f"{saved_df['word'].nunique()}"
)

print()

print(
    "Baseline manifest:"
)

print(
    BASELINE_MANIFEST_PATH
)

print()

print(
    "Excluded samples:"
)

print(
    EXCLUDED_MANIFEST_PATH
)


print()
print("=" * 76)

if (
    len(actual_classes) == NUM_CLASSES
    and split_class_ok
):

    print(
        "✅ CLEAN BASELINE DATASET READY"
    )

    print(
        "✅ All 44 classes retained"
    )

    print(
        "✅ All training samples have valid mouth cache"
    )

    print()
    print(
        "Next: Cell 8 Final Cache Validation"
    )

else:

    print(
        "⚠️ Dataset created, but class coverage "
        "needs inspection before training."
    )

print("=" * 76)

In [ ]:
# ============================================================
# CELL 8 : FINAL CACHE VALIDATION
# VocaLip Thai — Word-Level Baseline
#
# Purpose:
# - Load the clean baseline manifest
# - Validate every mouth cache file
# - Check shape / dtype / labels / split coverage
# - Confirm dataset is safe to continue to training
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

from tqdm.auto import tqdm


# ============================================================
# CLEAN BASELINE MANIFEST
# ============================================================

BASELINE_MANIFEST_PATH = (
    Path(ROOT)
    / "word_train_manifest_v5_baseline_clean.csv"
)

assert BASELINE_MANIFEST_PATH.exists(), (
    f"Baseline manifest not found:\n"
    f"{BASELINE_MANIFEST_PATH}"
)


# ============================================================
# LOAD CLEAN MANIFEST
# ============================================================

df = pd.read_csv(
    BASELINE_MANIFEST_PATH
)


print("=" * 76)
print("FINAL CACHE VALIDATION")
print("=" * 76)

print(
    f"Manifest : {BASELINE_MANIFEST_PATH}"
)

print(
    f"Samples  : {len(df):,}"
)

print(
    f"Classes  : {df['model_class_id'].nunique()}"
)

print(
    f"Words    : {df['word'].nunique()}"
)

print()


# ============================================================
# EXPECTED CACHE FORMAT
# ============================================================

EXPECTED_SHAPE = (
    TARGET_FRAMES,
    ROI_SIZE,
    ROI_SIZE,
    3
)

EXPECTED_DTYPE = np.uint8


# ============================================================
# VALIDATE EVERY CACHE
# ============================================================

validation_rows = []


for row_index, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Validating mouth cache"
):

    cache_path = Path(
        str(
            row["mouth_cache_path"]
        )
    )


    result = {

        "row_index":
            row_index,

        "split":
            row["split"],

        "word":
            row["word"],

        "model_class_id":
            int(
                row["model_class_id"]
            ),

        "mouth_cache_path":
            str(
                cache_path
            ),

        "exists":
            False,

        "load_ok":
            False,

        "shape_ok":
            False,

        "dtype_ok":
            False,

        "label_ok":
            False,

        "finite_ok":
            False,

        "status":
            "",

        "error":
            ""
    }


    # --------------------------------------------------------
    # EXISTS
    # --------------------------------------------------------

    if not cache_path.exists():

        result[
            "status"
        ] = "MISSING"

        validation_rows.append(
            result
        )

        continue


    result[
        "exists"
    ] = True


    # --------------------------------------------------------
    # LOAD NPZ
    # --------------------------------------------------------

    try:

        with np.load(
            cache_path,
            allow_pickle=False
        ) as cache:

            if "frames" not in cache:

                raise RuntimeError(
                    "frames key missing"
                )


            frames = cache[
                "frames"
            ]


            # ------------------------------------------------
            # SHAPE
            # ------------------------------------------------

            result[
                "shape_ok"
            ] = (
                frames.shape
                == EXPECTED_SHAPE
            )


            # ------------------------------------------------
            # DTYPE
            # ------------------------------------------------

            result[
                "dtype_ok"
            ] = (
                frames.dtype
                == EXPECTED_DTYPE
            )


            # ------------------------------------------------
            # FINITE
            # uint8 should always be finite,
            # but keep this as integrity check.
            # ------------------------------------------------

            result[
                "finite_ok"
            ] = bool(
                np.isfinite(
                    frames
                ).all()
            )


            # ------------------------------------------------
            # LABEL CHECK
            # ------------------------------------------------

            label_ok = True


            if "model_class_id" in cache:

                cached_class = int(
                    np.asarray(
                        cache[
                            "model_class_id"
                        ]
                    ).item()
                )


                if cached_class != int(
                    row[
                        "model_class_id"
                    ]
                ):

                    label_ok = False


            if "word" in cache:

                cached_word = str(
                    np.asarray(
                        cache["word"]
                    ).item()
                )


                if cached_word != str(
                    row["word"]
                ):

                    label_ok = False


            result[
                "label_ok"
            ] = label_ok


            result[
                "load_ok"
            ] = True


            # ------------------------------------------------
            # FINAL STATUS
            # ------------------------------------------------

            if (
                result[
                    "shape_ok"
                ]
                and
                result[
                    "dtype_ok"
                ]
                and
                result[
                    "finite_ok"
                ]
                and
                result[
                    "label_ok"
                ]
            ):

                result[
                    "status"
                ] = "OK"

            else:

                result[
                    "status"
                ] = "INVALID"


    except Exception as e:

        result[
            "status"
        ] = "LOAD_ERROR"

        result[
            "error"
        ] = repr(
            e
        )


    validation_rows.append(
        result
    )


# ============================================================
# VALIDATION DATAFRAME
# ============================================================

validation_df = pd.DataFrame(
    validation_rows
)


# ============================================================
# SUMMARY COUNTS
# ============================================================

total_samples = len(
    validation_df
)

ok_count = int(
    (
        validation_df[
            "status"
        ] == "OK"
    ).sum()
)

missing_count = int(
    (
        validation_df[
            "status"
        ] == "MISSING"
    ).sum()
)

invalid_count = int(
    (
        validation_df[
            "status"
        ] == "INVALID"
    ).sum()
)

load_error_count = int(
    (
        validation_df[
            "status"
        ] == "LOAD_ERROR"
    ).sum()
)


print()
print("=" * 76)
print("CACHE INTEGRITY SUMMARY")
print("=" * 76)

print(
    f"Total samples : {total_samples:,}"
)

print(
    f"OK            : {ok_count:,}"
)

print(
    f"Missing       : {missing_count:,}"
)

print(
    f"Invalid       : {invalid_count:,}"
)

print(
    f"Load error    : {load_error_count:,}"
)


# ============================================================
# FIELD CHECK SUMMARY
# ============================================================

print()
print("=" * 76)
print("FIELD VALIDATION")
print("=" * 76)

print(
    f"Exists    : "
    f"{int(validation_df['exists'].sum()):,}"
    f"/{total_samples:,}"
)

print(
    f"Load OK   : "
    f"{int(validation_df['load_ok'].sum()):,}"
    f"/{total_samples:,}"
)

print(
    f"Shape OK  : "
    f"{int(validation_df['shape_ok'].sum()):,}"
    f"/{total_samples:,}"
)

print(
    f"Dtype OK  : "
    f"{int(validation_df['dtype_ok'].sum()):,}"
    f"/{total_samples:,}"
)

print(
    f"Label OK  : "
    f"{int(validation_df['label_ok'].sum()):,}"
    f"/{total_samples:,}"
)

print(
    f"Finite OK : "
    f"{int(validation_df['finite_ok'].sum()):,}"
    f"/{total_samples:,}"
)


# ============================================================
# SPLIT VALIDATION
# ============================================================

print()
print("=" * 76)
print("CACHE STATUS BY SPLIT")
print("=" * 76)


for split_name in [
    "train",
    "val",
    "test"
]:

    temp = validation_df[
        validation_df[
            "split"
        ] == split_name
    ]


    split_total = len(
        temp
    )

    split_ok = int(
        (
            temp[
                "status"
            ] == "OK"
        ).sum()
    )


    print(
        f"{split_name:<5} "
        f"OK={split_ok:4d}/"
        f"{split_total:4d}"
    )


# ============================================================
# CLASS COVERAGE
# ============================================================

print()
print("=" * 76)
print("CLASS COVERAGE")
print("=" * 76)


expected_classes = set(
    range(
        NUM_CLASSES
    )
)


all_classes = set(
    df[
        "model_class_id"
    ].astype(int)
)


print(
    f"Overall : "
    f"{len(all_classes)}/{NUM_CLASSES}"
)


overall_missing_classes = sorted(
    expected_classes
    - all_classes
)


print(
    f"Missing overall : "
    f"{overall_missing_classes}"
)


split_class_ok = True


for split_name in [
    "train",
    "val",
    "test"
]:

    temp = df[
        df[
            "split"
        ] == split_name
    ]


    classes = set(
        temp[
            "model_class_id"
        ].astype(int)
    )


    missing = sorted(
        expected_classes
        - classes
    )


    print(
        f"{split_name:<5} : "
        f"{len(classes):2d}/{NUM_CLASSES} "
        f"missing={missing}"
    )


    if len(
        missing
    ) > 0:

        split_class_ok = False


# ============================================================
# LABEL MAP CHECK
# ============================================================

print()
print("=" * 76)
print("LABEL MAP CHECK")
print("=" * 76)


assert LABEL_MAP_PATH.exists(), (
    f"Label map not found:\n"
    f"{LABEL_MAP_PATH}"
)


with open(
    LABEL_MAP_PATH,
    "r",
    encoding="utf-8"
) as f:

    label_map = json.load(
        f
    )


print(
    f"Label map entries : "
    f"{len(label_map)}"
)


if len(label_map) != NUM_CLASSES:

    print(
        "⚠️ Label map size "
        "does not match NUM_CLASSES."
    )

else:

    print(
        "✅ Label map size correct."
    )


# ============================================================
# BAD SAMPLE REPORT
# ============================================================

bad_df = validation_df[
    validation_df[
        "status"
    ] != "OK"
].copy()


if len(
    bad_df
) > 0:

    print()
    print("=" * 76)
    print("PROBLEMATIC CACHE FILES")
    print("=" * 76)


    display(
        bad_df[
            [
                "split",
                "word",
                "model_class_id",
                "status",
                "mouth_cache_path",
                "error"
            ]
        ].head(
            50
        )
    )


    BAD_CACHE_REPORT_PATH = (
        Path(ROOT)
        / "mouth_cache_v3_final_validation_errors.csv"
    )


    bad_df.to_csv(
        BAD_CACHE_REPORT_PATH,
        index=False,
        encoding="utf-8-sig"
    )


    print(
        "\nReport saved to:"
    )

    print(
        BAD_CACHE_REPORT_PATH
    )


# ============================================================
# FINAL HARD GATE
# ============================================================

print()
print("=" * 76)
print("FINAL DECISION")
print("=" * 76)


all_cache_ok = (
    ok_count
    == total_samples
)


if (
    all_cache_ok
    and
    len(all_classes)
        == NUM_CLASSES
    and
    split_class_ok
):

    print(
        "✅ FINAL CACHE VALIDATION PASSED"
    )

    print(
        f"✅ {total_samples:,}/{total_samples:,} "
        "samples valid"
    )

    print(
        f"✅ All {NUM_CLASSES} classes retained"
    )

    print(
        "✅ Train / val / test class coverage complete"
    )

    print()
    print(
        "Dataset preprocessing is complete."
    )

    print(
        "Next step: Cell 9 "
        "Dataset + Augmentation."
    )

else:

    print(
        "❌ FINAL CACHE VALIDATION FAILED"
    )

    print()

    print(
        "Do NOT continue to training yet."
    )


    if not all_cache_ok:

        print(
            f"- Invalid cache samples: "
            f"{total_samples - ok_count}"
        )


    if len(
        overall_missing_classes
    ) > 0:

        print(
            "- Missing classes overall:",
            overall_missing_classes
        )


    if not split_class_ok:

        print(
            "- One or more splits "
            "are missing classes."
        )


    raise RuntimeError(
        "Final cache validation failed."
    )

In [ ]:
# ============================================================
# CELL 9 : DATASET + AUGMENTATION
# VocaLip Thai — Word-Level Baseline
#
# Purpose:
# - Load precomputed mouth ROI cache
# - Apply mild augmentation to TRAIN only
# - Normalize RGB frames
# - Return tensors:
#       video : (T, C, H, W)
#       label : int64
#
# Baseline policy:
# - NO horizontal flip
# - NO aggressive geometric transforms
# - NO temporal reversal
# - Mild photometric augmentation only
# ============================================================

from pathlib import Path
import random

import cv2
import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset


# ============================================================
# NORMALIZATION
# ImageNet mean/std
# Suitable for RGB CNN input
# ============================================================

RGB_MEAN = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32
)

RGB_STD = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32
)


# ============================================================
# AUGMENTATION CONFIG
# ============================================================

TRAIN_AUGMENT = True

BRIGHTNESS_PROB = 0.30
CONTRAST_PROB = 0.30

BRIGHTNESS_RANGE = (
    0.90,
    1.10
)

CONTRAST_RANGE = (
    0.90,
    1.10
)

GAUSSIAN_NOISE_PROB = 0.15
GAUSSIAN_NOISE_STD = 2.0

BLUR_PROB = 0.10

TEMPORAL_JITTER_PROB = 0.20
TEMPORAL_JITTER_MAX_SHIFT = 1


# ============================================================
# PHOTOMETRIC AUGMENTATION
# Apply SAME transform across whole sequence
# to preserve temporal consistency.
# ============================================================

def apply_sequence_augmentation(
    frames
):
    """
    frames:
        uint8 numpy array
        shape = (T, H, W, C)

    returns:
        uint8 numpy array
        same shape
    """

    x = frames.astype(
        np.float32
    )


    # --------------------------------------------------------
    # BRIGHTNESS
    # --------------------------------------------------------

    if random.random() < BRIGHTNESS_PROB:

        factor = random.uniform(
            *BRIGHTNESS_RANGE
        )

        x *= factor


    # --------------------------------------------------------
    # CONTRAST
    # --------------------------------------------------------

    if random.random() < CONTRAST_PROB:

        factor = random.uniform(
            *CONTRAST_RANGE
        )

        mean = x.mean(
            axis=(1, 2),
            keepdims=True
        )

        x = (
            mean
            + factor
            * (
                x - mean
            )
        )


    # --------------------------------------------------------
    # GAUSSIAN NOISE
    # --------------------------------------------------------

    if random.random() < GAUSSIAN_NOISE_PROB:

        noise = np.random.normal(
            loc=0.0,
            scale=GAUSSIAN_NOISE_STD,
            size=x.shape
        ).astype(
            np.float32
        )

        x += noise


    # --------------------------------------------------------
    # MILD BLUR
    # Same kernel across all frames
    # --------------------------------------------------------

    if random.random() < BLUR_PROB:

        blurred = []

        for frame in x:

            frame_blur = cv2.GaussianBlur(
                frame,
                (3, 3),
                sigmaX=0
            )

            blurred.append(
                frame_blur
            )

        x = np.stack(
            blurred,
            axis=0
        )


    # --------------------------------------------------------
    # CLIP
    # --------------------------------------------------------

    x = np.clip(
        x,
        0,
        255
    )


    return x.astype(
        np.uint8
    )


# ============================================================
# TEMPORAL JITTER
# Small ±1 frame shift only.
#
# Because cache already has TARGET_FRAMES frames,
# we use edge replication.
# ============================================================

def apply_temporal_jitter(
    frames
):
    """
    frames:
        (T, H, W, C)
    """

    if (
        random.random()
        >= TEMPORAL_JITTER_PROB
    ):

        return frames


    shift = random.randint(
        -TEMPORAL_JITTER_MAX_SHIFT,
        TEMPORAL_JITTER_MAX_SHIFT
    )


    if shift == 0:

        return frames


    T = len(
        frames
    )


    if shift > 0:

        pad = np.repeat(
            frames[:1],
            shift,
            axis=0
        )

        output = np.concatenate(
            [
                pad,
                frames[
                    :T-shift
                ]
            ],
            axis=0
        )


    else:

        shift_abs = abs(
            shift
        )

        pad = np.repeat(
            frames[-1:],
            shift_abs,
            axis=0
        )

        output = np.concatenate(
            [
                frames[
                    shift_abs:
                ],
                pad
            ],
            axis=0
        )


    return output


# ============================================================
# NORMALIZE
# ============================================================

def normalize_frames(
    frames
):
    """
    uint8:
        (T, H, W, C)

    returns float32:
        (T, C, H, W)
    """

    x = frames.astype(
        np.float32
    ) / 255.0


    x = (
        x
        - RGB_MEAN[
            None,
            None,
            None,
            :
        ]
    ) / RGB_STD[
        None,
        None,
        None,
        :
    ]


    # T H W C -> T C H W
    x = np.transpose(
        x,
        (
            0,
            3,
            1,
            2
        )
    )


    return np.ascontiguousarray(
        x,
        dtype=np.float32
    )


# ============================================================
# DATASET
# ============================================================

class VocaLipWordDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        training=False
    ):

        self.df = (
            dataframe
            .reset_index(
                drop=True
            )
            .copy()
        )

        self.training = bool(
            training
        )


    def __len__(
        self
    ):

        return len(
            self.df
        )


    def __getitem__(
        self,
        index
    ):

        row = self.df.iloc[
            index
        ]


        cache_path = Path(
            str(
                row[
                    "mouth_cache_path"
                ]
            )
        )


        # ----------------------------------------------------
        # LOAD CACHE
        # ----------------------------------------------------

        with np.load(
            cache_path,
            allow_pickle=False
        ) as cache:

            frames = cache[
                "frames"
            ].copy()


        # ----------------------------------------------------
        # SAFETY CHECK
        # ----------------------------------------------------

        expected_shape = (
            TARGET_FRAMES,
            ROI_SIZE,
            ROI_SIZE,
            3
        )


        if frames.shape != expected_shape:

            raise RuntimeError(
                f"Bad cache shape at index {index}: "
                f"{frames.shape} "
                f"expected {expected_shape}"
            )


        # ----------------------------------------------------
        # TRAIN AUGMENTATION ONLY
        # ----------------------------------------------------

        if (
            self.training
            and TRAIN_AUGMENT
        ):

            frames = (
                apply_temporal_jitter(
                    frames
                )
            )

            frames = (
                apply_sequence_augmentation(
                    frames
                )
            )


        # ----------------------------------------------------
        # NORMALIZE
        # ----------------------------------------------------

        frames = normalize_frames(
            frames
        )


        # ----------------------------------------------------
        # TO TENSOR
        # ----------------------------------------------------

        video_tensor = torch.from_numpy(
            frames
        ).float()


        label_tensor = torch.tensor(
            int(
                row[
                    "model_class_id"
                ]
            ),
            dtype=torch.long
        )


        return {

            "video":
                video_tensor,

            "label":
                label_tensor,

            "word":
                str(
                    row[
                        "word"
                    ]
                ),

            "split":
                str(
                    row[
                        "split"
                    ]
                ),

            "cache_path":
                str(
                    cache_path
                )
        }


# ============================================================
# SPLIT DATAFRAMES
# ============================================================

train_df = (
    df[
        df["split"] == "train"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

val_df = (
    df[
        df["split"] == "val"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

test_df = (
    df[
        df["split"] == "test"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ============================================================
# CREATE DATASETS
# ============================================================

train_dataset = VocaLipWordDataset(
    train_df,
    training=True
)

val_dataset = VocaLipWordDataset(
    val_df,
    training=False
)

test_dataset = VocaLipWordDataset(
    test_df,
    training=False
)


# ============================================================
# BASIC SUMMARY
# ============================================================

print("=" * 76)
print("DATASET + AUGMENTATION")
print("=" * 76)

print(
    f"Train : {len(train_dataset):,}"
)

print(
    f"Val   : {len(val_dataset):,}"
)

print(
    f"Test  : {len(test_dataset):,}"
)

print()

print(
    f"TRAIN_AUGMENT : {TRAIN_AUGMENT}"
)

print(
    "Horizontal flip : DISABLED"
)

print(
    "Heavy geometry  : DISABLED"
)

print(
    "Temporal reverse: DISABLED"
)


# ============================================================
# TEST ONE SAMPLE
# ============================================================

sample = train_dataset[
    0
]


print()
print("=" * 76)
print("SINGLE SAMPLE CHECK")
print("=" * 76)

print(
    f"Video shape : "
    f"{tuple(sample['video'].shape)}"
)

print(
    f"Video dtype : "
    f"{sample['video'].dtype}"
)

print(
    f"Label       : "
    f"{sample['label'].item()}"
)

print(
    f"Word        : "
    f"{sample['word']}"
)

print(
    f"Split       : "
    f"{sample['split']}"
)

print(
    f"Min value   : "
    f"{sample['video'].min().item():.4f}"
)

print(
    f"Max value   : "
    f"{sample['video'].max().item():.4f}"
)


# ============================================================
# HARD CHECKS
# ============================================================

assert tuple(
    sample[
        "video"
    ].shape
) == (
    TARGET_FRAMES,
    3,
    ROI_SIZE,
    ROI_SIZE
)


assert (
    sample[
        "video"
    ].dtype
    == torch.float32
)


assert (
    sample[
        "label"
    ].dtype
    == torch.long
)


assert 0 <= int(
    sample[
        "label"
    ]
) < NUM_CLASSES


print()
print("=" * 76)
print("✅ CELL 9 DATASET PIPELINE READY")
print("=" * 76)

print(
    "Output tensor format:"
)

print(
    "    (T, C, H, W)"
)

print(
    f"    ({TARGET_FRAMES}, 3, "
    f"{ROI_SIZE}, {ROI_SIZE})"
)

print()

print(
    "Next: Cell 10 "
    "Class Weights + DataLoaders"
)

In [ ]:
# ============================================================
# CELL 10 : CLASS WEIGHTS + DATALOADERS
# VocaLip Thai — Word-Level Baseline
#
# Purpose:
# - Compute class weights from TRAIN split only
# - Build train / val / test DataLoaders
# - Keep validation/test deterministic
# - Use weighted CE later in Cell 12
#
# Baseline policy:
# - USE_CLASS_WEIGHTED_LOSS = True
# - USE_WEIGHTED_SAMPLER = False
# ============================================================

import os
import numpy as np
import pandas as pd

import torch
from torch.utils.data import DataLoader
from torch.utils.data import WeightedRandomSampler


# ============================================================
# CLASS COUNTS — TRAIN ONLY
# ============================================================

train_class_counts = (
    train_df[
        "model_class_id"
    ]
    .astype(int)
    .value_counts()
    .sort_index()
)


# Ensure all classes are represented
expected_class_ids = list(
    range(NUM_CLASSES)
)


missing_train_classes = [
    class_id
    for class_id in expected_class_ids
    if class_id not in train_class_counts.index
]


assert len(missing_train_classes) == 0, (
    f"Train split missing classes: "
    f"{missing_train_classes}"
)


# Reindex explicitly 0..43
train_class_counts = (
    train_class_counts
    .reindex(
        expected_class_ids,
        fill_value=0
    )
)


print("=" * 76)
print("TRAIN CLASS DISTRIBUTION")
print("=" * 76)

print(
    f"Train samples : {len(train_df):,}"
)

print(
    f"Classes       : {NUM_CLASSES}"
)

print(
    f"Min/class     : "
    f"{train_class_counts.min()}"
)

print(
    f"Max/class     : "
    f"{train_class_counts.max()}"
)

print(
    f"Mean/class    : "
    f"{train_class_counts.mean():.2f}"
)


imbalance_ratio = (
    train_class_counts.max()
    / train_class_counts.min()
)


print(
    f"Max/min ratio : "
    f"{imbalance_ratio:.2f}x"
)


# ============================================================
# CLASS WEIGHTS
#
# Formula:
#     weight_c = N / (K * count_c)
#
# Mean class weight ~= 1 in balanced setting.
# ============================================================

total_train = len(
    train_df
)


class_weights_np = (
    total_train
    /
    (
        NUM_CLASSES
        * train_class_counts.values.astype(
            np.float32
        )
    )
)


# Optional mild clipping:
# Avoid extremely large gradients for rare classes.
#
# Current dataset is imbalanced but not so extreme that
# aggressive weighting is desirable for baseline.
# ============================================================

CLASS_WEIGHT_MIN = 0.25
CLASS_WEIGHT_MAX = 4.00


class_weights_np = np.clip(
    class_weights_np,
    CLASS_WEIGHT_MIN,
    CLASS_WEIGHT_MAX
)


class_weights = torch.tensor(
    class_weights_np,
    dtype=torch.float32
)


# ============================================================
# CLASS WEIGHT TABLE
# ============================================================

class_weight_table = pd.DataFrame({

    "model_class_id":
        expected_class_ids,

    "train_count":
        train_class_counts.values,

    "class_weight":
        class_weights_np
})


# Add word names
class_id_to_word = (
    train_df[
        [
            "model_class_id",
            "word"
        ]
    ]
    .drop_duplicates(
        subset=[
            "model_class_id"
        ]
    )
    .set_index(
        "model_class_id"
    )[
        "word"
    ]
    .to_dict()
)


class_weight_table[
    "word"
] = class_weight_table[
    "model_class_id"
].map(
    class_id_to_word
)


class_weight_table = class_weight_table[
    [
        "model_class_id",
        "word",
        "train_count",
        "class_weight"
    ]
]


print()
print("=" * 76)
print("CLASS WEIGHTS")
print("=" * 76)

display(
    class_weight_table
)


print(
    f"\nWeight min  : "
    f"{class_weights_np.min():.4f}"
)

print(
    f"Weight max  : "
    f"{class_weights_np.max():.4f}"
)

print(
    f"Weight mean : "
    f"{class_weights_np.mean():.4f}"
)


# ============================================================
# OPTIONAL WEIGHTED SAMPLER
#
# Default = OFF
# We should not combine sampler + weighted CE in baseline
# unless intentionally running an ablation.
# ============================================================

train_sampler = None


if USE_WEIGHTED_SAMPLER:

    sample_weights = train_df[
        "model_class_id"
    ].astype(int).map(
        {
            class_id:
                class_weights_np[class_id]
            for class_id in expected_class_ids
        }
    ).values


    train_sampler = WeightedRandomSampler(

        weights=torch.tensor(
            sample_weights,
            dtype=torch.double
        ),

        num_samples=len(
            sample_weights
        ),

        replacement=True
    )


# ============================================================
# DATALOADER SETTINGS
# ============================================================

PIN_MEMORY = bool(
    torch.cuda.is_available()
)


# NUM_WORKERS already defined in config.
# Current Windows setup uses NUM_WORKERS = 0
# for maximum notebook stability.
# ============================================================

persistent_workers = (
    NUM_WORKERS > 0
)


# ============================================================
# TRAIN LOADER
# ============================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=(
        train_sampler is None
    ),

    sampler=train_sampler,

    num_workers=NUM_WORKERS,

    pin_memory=PIN_MEMORY,

    drop_last=False,

    persistent_workers=(
        persistent_workers
    )
)


# ============================================================
# VALIDATION LOADER
# ============================================================

val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=PIN_MEMORY,

    drop_last=False,

    persistent_workers=(
        persistent_workers
    )
)


# ============================================================
# TEST LOADER
# ============================================================

test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=PIN_MEMORY,

    drop_last=False,

    persistent_workers=(
        persistent_workers
    )
)


# ============================================================
# LOADER SUMMARY
# ============================================================

print()
print("=" * 76)
print("DATALOADER SUMMARY")
print("=" * 76)

print(
    f"Batch size     : "
    f"{BATCH_SIZE}"
)

print(
    f"Num workers    : "
    f"{NUM_WORKERS}"
)

print(
    f"Pin memory     : "
    f"{PIN_MEMORY}"
)

print(
    f"Weighted loss  : "
    f"{USE_CLASS_WEIGHTED_LOSS}"
)

print(
    f"Weighted sampler : "
    f"{USE_WEIGHTED_SAMPLER}"
)

print()

print(
    f"Train batches  : "
    f"{len(train_loader)}"
)

print(
    f"Val batches    : "
    f"{len(val_loader)}"
)

print(
    f"Test batches   : "
    f"{len(test_loader)}"
)


# ============================================================
# TEST ONE TRAIN BATCH
# ============================================================

batch = next(
    iter(
        train_loader
    )
)


print()
print("=" * 76)
print("BATCH CHECK")
print("=" * 76)

print(
    f"Video batch shape : "
    f"{tuple(batch['video'].shape)}"
)

print(
    f"Video dtype       : "
    f"{batch['video'].dtype}"
)

print(
    f"Label shape       : "
    f"{tuple(batch['label'].shape)}"
)

print(
    f"Label dtype       : "
    f"{batch['label'].dtype}"
)

print(
    f"Label min/max     : "
    f"{batch['label'].min().item()} / "
    f"{batch['label'].max().item()}"
)


# ============================================================
# EXPECTED SHAPE
#
# B, T, C, H, W
# ============================================================

assert batch[
    "video"
].ndim == 5


assert batch[
    "video"
].shape[1:] == (
    TARGET_FRAMES,
    3,
    ROI_SIZE,
    ROI_SIZE
)


assert batch[
    "video"
].dtype == torch.float32


assert batch[
    "label"
].dtype == torch.long


assert batch[
    "label"
].min().item() >= 0


assert batch[
    "label"
].max().item() < NUM_CLASSES


# ============================================================
# POLICY CHECK
# ============================================================

if (
    USE_CLASS_WEIGHTED_LOSS
    and
    USE_WEIGHTED_SAMPLER
):

    print()
    print(
        "⚠️ WARNING:"
    )

    print(
        "Both weighted loss and weighted sampler are enabled."
    )

    print(
        "For Experiment 0, use only one imbalance correction."
    )


else:

    print()
    print(
        "✅ Imbalance policy is clean."
    )


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 76)
print("✅ CELL 10 DATALOADERS READY")
print("=" * 76)

print(
    "Expected input to model:"
)

print(
    "    (B, T, C, H, W)"
)

print(
    f"    (B, {TARGET_FRAMES}, 3, "
    f"{ROI_SIZE}, {ROI_SIZE})"
)

print()

print(
    "Next: Cell 11 Model"
)

In [ ]:
# ============================================================
# CELL 11 : MODEL
# VocaLip Thai — Word-Level Baseline
#
# Architecture:
#   Input:
#       (B, T, C, H, W)
#
#   Frame CNN Encoder:
#       each frame -> FEATURE_DIM
#
#   Temporal Encoder:
#       2-layer Bidirectional GRU
#
#   Temporal Attention:
#       weighted pooling across T
#
#   Classifier:
#       -> 44 Thai word classes
#
# Experiment 0:
# - Mouth-only
# - Word-level classification
# - No CTC
# - No SSL
# - No face stream
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# MODEL CONFIG
# ============================================================

FEATURE_DIM = 128
GRU_HIDDEN = 256
GRU_LAYERS = 2
MODEL_DROPOUT = 0.20


# ============================================================
# FRAME CNN ENCODER
# ============================================================

class FrameCNNEncoder(nn.Module):

    def __init__(
        self,
        feature_dim=128
    ):

        super().__init__()


        self.features = nn.Sequential(

            # ------------------------------------------------
            # 112 x 112
            # ------------------------------------------------

            nn.Conv2d(
                3,
                32,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                32
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            ),

            # -> 56 x 56


            # ------------------------------------------------
            # 56 x 56
            # ------------------------------------------------

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                64
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            ),

            # -> 28 x 28


            # ------------------------------------------------
            # 28 x 28
            # ------------------------------------------------

            nn.Conv2d(
                64,
                96,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                96
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            ),

            # -> 14 x 14


            # ------------------------------------------------
            # 14 x 14
            # ------------------------------------------------

            nn.Conv2d(
                96,
                128,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                128
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.AdaptiveAvgPool2d(
                (
                    1,
                    1
                )
            )
        )


        self.projection = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128,
                feature_dim
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Dropout(
                p=0.10
            )
        )


    def forward(
        self,
        x
    ):

        x = self.features(
            x
        )

        x = self.projection(
            x
        )

        return x


# ============================================================
# TEMPORAL ATTENTION
# ============================================================

class TemporalAttention(nn.Module):

    def __init__(
        self,
        input_dim,
        attention_dim=128
    ):

        super().__init__()


        self.attention = nn.Sequential(

            nn.Linear(
                input_dim,
                attention_dim
            ),

            nn.Tanh(),

            nn.Linear(
                attention_dim,
                1
            )
        )


    def forward(
        self,
        x
    ):
        """
        x:
            (B, T, D)

        returns:
            pooled:
                (B, D)

            attention_weights:
                (B, T)
        """


        scores = self.attention(
            x
        ).squeeze(
            -1
        )

        # (B, T)


        weights = torch.softmax(
            scores,
            dim=1
        )

        # (B, T)


        pooled = torch.sum(

            x
            * weights.unsqueeze(
                -1
            ),

            dim=1
        )

        # (B, D)


        return (
            pooled,
            weights
        )


# ============================================================
# VOCALIP WORD MODEL
# ============================================================

class VocaLipWordBaseline(nn.Module):

    def __init__(
        self,
        num_classes,
        feature_dim=128,
        gru_hidden=256,
        gru_layers=2,
        dropout=0.20
    ):

        super().__init__()


        # ----------------------------------------------------
        # FRAME ENCODER
        # ----------------------------------------------------

        self.frame_encoder = FrameCNNEncoder(
            feature_dim=feature_dim
        )


        # ----------------------------------------------------
        # TEMPORAL ENCODER
        # ----------------------------------------------------

        self.temporal_encoder = nn.GRU(

            input_size=feature_dim,

            hidden_size=gru_hidden,

            num_layers=gru_layers,

            batch_first=True,

            bidirectional=True,

            dropout=(
                dropout
                if gru_layers > 1
                else 0.0
            )
        )


        temporal_dim = (
            gru_hidden
            * 2
        )


        # ----------------------------------------------------
        # TEMPORAL ATTENTION
        # ----------------------------------------------------

        self.temporal_attention = TemporalAttention(

            input_dim=temporal_dim,

            attention_dim=128
        )


        # ----------------------------------------------------
        # CLASSIFIER
        # ----------------------------------------------------

        self.classifier = nn.Sequential(

            nn.LayerNorm(
                temporal_dim
            ),

            nn.Dropout(
                p=dropout
            ),

            nn.Linear(
                temporal_dim,
                256
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Dropout(
                p=dropout
            ),

            nn.Linear(
                256,
                num_classes
            )
        )


    def forward(
        self,
        video,
        return_attention=False
    ):
        """
        video:
            (B, T, C, H, W)

        returns:
            logits:
                (B, NUM_CLASSES)
        """


        B, T, C, H, W = (
            video.shape
        )


        # ----------------------------------------------------
        # MERGE BATCH + TIME
        #
        # (B,T,C,H,W)
        # ->
        # (B*T,C,H,W)
        # ----------------------------------------------------

        x = video.reshape(

            B * T,
            C,
            H,
            W
        )


        # ----------------------------------------------------
        # FRAME FEATURES
        #
        # (B*T, FEATURE_DIM)
        # ----------------------------------------------------

        frame_features = (
            self.frame_encoder(
                x
            )
        )


        # ----------------------------------------------------
        # RESTORE TIME
        #
        # (B,T,FEATURE_DIM)
        # ----------------------------------------------------

        frame_features = (
            frame_features.reshape(

                B,
                T,
                -1
            )
        )


        # ----------------------------------------------------
        # BiGRU
        #
        # output:
        # (B,T,2*GRU_HIDDEN)
        # ----------------------------------------------------

        temporal_features, _ = (
            self.temporal_encoder(
                frame_features
            )
        )


        # ----------------------------------------------------
        # ATTENTION POOLING
        # ----------------------------------------------------

        pooled, attention_weights = (
            self.temporal_attention(
                temporal_features
            )
        )


        # ----------------------------------------------------
        # CLASSIFICATION
        # ----------------------------------------------------

        logits = self.classifier(
            pooled
        )


        if return_attention:

            return (
                logits,
                attention_weights
            )


        return logits


# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(

    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 76)
print("MODEL SETUP")
print("=" * 76)

print(
    f"Device       : {DEVICE}"
)

if torch.cuda.is_available():

    print(
        f"GPU          : "
        f"{torch.cuda.get_device_name(0)}"
    )


# ============================================================
# CREATE MODEL
# ============================================================

model = VocaLipWordBaseline(

    num_classes=NUM_CLASSES,

    feature_dim=FEATURE_DIM,

    gru_hidden=GRU_HIDDEN,

    gru_layers=GRU_LAYERS,

    dropout=MODEL_DROPOUT
)


model = model.to(
    DEVICE
)


# ============================================================
# PARAMETER COUNT
# ============================================================

total_params = sum(
    p.numel()
    for p in model.parameters()
)


trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print()
print("=" * 76)
print("MODEL SUMMARY")
print("=" * 76)

print(
    f"Feature dim    : {FEATURE_DIM}"
)

print(
    f"GRU hidden     : {GRU_HIDDEN}"
)

print(
    f"GRU layers     : {GRU_LAYERS}"
)

print(
    f"Bidirectional  : True"
)

print(
    f"Dropout        : {MODEL_DROPOUT}"
)

print(
    f"Classes        : {NUM_CLASSES}"
)

print()

print(
    f"Total params   : "
    f"{total_params:,}"
)

print(
    f"Trainable      : "
    f"{trainable_params:,}"
)


# ============================================================
# FORWARD PASS TEST
# ============================================================

model.eval()


with torch.no_grad():

    test_video = batch[
        "video"
    ][:2].to(
        DEVICE,
        non_blocking=True
    )


    logits, attention = model(

        test_video,

        return_attention=True
    )


print()
print("=" * 76)
print("FORWARD PASS CHECK")
print("=" * 76)

print(
    f"Input shape     : "
    f"{tuple(test_video.shape)}"
)

print(
    f"Logits shape    : "
    f"{tuple(logits.shape)}"
)

print(
    f"Attention shape : "
    f"{tuple(attention.shape)}"
)

print(
    f"Logits dtype    : "
    f"{logits.dtype}"
)

print(
    f"Attention min   : "
    f"{attention.min().item():.6f}"
)

print(
    f"Attention max   : "
    f"{attention.max().item():.6f}"
)

print(
    f"Attention sums  : "
    f"{attention.sum(dim=1).detach().cpu().numpy()}"
)


# ============================================================
# HARD CHECKS
# ============================================================

assert logits.shape == (
    2,
    NUM_CLASSES
)


assert attention.shape == (
    2,
    TARGET_FRAMES
)


assert torch.isfinite(
    logits
).all()


assert torch.isfinite(
    attention
).all()


assert torch.allclose(

    attention.sum(
        dim=1
    ),

    torch.ones(
        attention.shape[0],
        device=attention.device
    ),

    atol=1e-5
)


# ============================================================
# GPU MEMORY CHECK
# ============================================================

if torch.cuda.is_available():

    print()
    print("=" * 76)
    print("GPU MEMORY")
    print("=" * 76)

    print(
        f"Allocated : "
        f"{torch.cuda.memory_allocated() / 1024**2:.1f} MB"
    )

    print(
        f"Reserved  : "
        f"{torch.cuda.memory_reserved() / 1024**2:.1f} MB"
    )


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 76)
print("✅ CELL 11 MODEL READY")
print("=" * 76)

print(
    "Architecture:"
)

print(
    "Mouth ROI -> CNN -> BiGRU -> "
    "Temporal Attention -> 44 classes"
)

print()

print(
    "Next: Cell 12 "
    "Loss + Optimizer + Scheduler + AMP"
)

In [ ]:
# ============================================================
# CELL 12 : LOSS + OPTIMIZER + SCHEDULER + AMP
# VocaLip Thai — Word-Level Baseline
#
# Purpose:
# - Weighted CrossEntropyLoss
# - AdamW optimizer
# - ReduceLROnPlateau scheduler
# - Mixed precision training (AMP) on CUDA
# ============================================================

import torch
import torch.nn as nn


# ============================================================
# LOSS
# ============================================================

if USE_CLASS_WEIGHTED_LOSS:

    loss_class_weights = (
        class_weights
        .to(
            DEVICE
        )
    )

    criterion = nn.CrossEntropyLoss(
        weight=loss_class_weights
    )

else:

    criterion = nn.CrossEntropyLoss()


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LR,

    weight_decay=WEIGHT_DECAY
)


# ============================================================
# LR SCHEDULER
#
# Monitor validation Macro F1.
# Higher is better -> mode="max"
# ============================================================

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="max",

    factor=0.5,

    patience=3,

    min_lr=1e-6
)


# ============================================================
# AMP
# ============================================================

USE_AMP = (
    DEVICE.type == "cuda"
)


# New PyTorch AMP API when available
try:

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP
    )

    AMP_API = "torch.amp"

except Exception:

    scaler = torch.cuda.amp.GradScaler(
        enabled=USE_AMP
    )

    AMP_API = "torch.cuda.amp"


# ============================================================
# AUTOCAST HELPER
# ============================================================

def autocast_context():

    if DEVICE.type == "cuda":

        try:

            return torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=USE_AMP
            )

        except Exception:

            return torch.cuda.amp.autocast(
                enabled=USE_AMP
            )

    else:

        return torch.amp.autocast(
            device_type="cpu",
            enabled=False
        )


# ============================================================
# TRAINING STABILITY CONFIG
# ============================================================

GRAD_CLIP_NORM = 5.0


# ============================================================
# SUMMARY
# ============================================================

print("=" * 76)
print("TRAINING SETUP")
print("=" * 76)

print(
    f"Device               : {DEVICE}"
)

print(
    f"Loss                 : "
    f"{criterion.__class__.__name__}"
)

print(
    f"Class weighted loss  : "
    f"{USE_CLASS_WEIGHTED_LOSS}"
)

print(
    f"Optimizer            : AdamW"
)

print(
    f"Learning rate        : "
    f"{LR}"
)

print(
    f"Weight decay         : "
    f"{WEIGHT_DECAY}"
)

print(
    f"Scheduler            : "
    f"ReduceLROnPlateau"
)

print(
    f"Scheduler monitor    : "
    f"Val Macro F1"
)

print(
    f"Scheduler factor     : "
    f"0.5"
)

print(
    f"Scheduler patience   : "
    f"3"
)

print(
    f"AMP enabled          : "
    f"{USE_AMP}"
)

print(
    f"AMP API              : "
    f"{AMP_API}"
)

print(
    f"Gradient clipping    : "
    f"{GRAD_CLIP_NORM}"
)


# ============================================================
# SANITY LOSS TEST
# ============================================================

model.eval()


with torch.no_grad():

    sanity_video = (
        batch["video"][:4]
        .to(
            DEVICE,
            non_blocking=True
        )
    )

    sanity_labels = (
        batch["label"][:4]
        .to(
            DEVICE,
            non_blocking=True
        )
    )


    with autocast_context():

        sanity_logits = model(
            sanity_video
        )

        sanity_loss = criterion(
            sanity_logits,
            sanity_labels
        )


print()
print("=" * 76)
print("LOSS CHECK")
print("=" * 76)

print(
    f"Logits shape : "
    f"{tuple(sanity_logits.shape)}"
)

print(
    f"Labels shape : "
    f"{tuple(sanity_labels.shape)}"
)

print(
    f"Loss         : "
    f"{sanity_loss.item():.6f}"
)


# ============================================================
# HARD CHECKS
# ============================================================

assert sanity_logits.shape == (
    len(sanity_labels),
    NUM_CLASSES
)

assert torch.isfinite(
    sanity_loss
)

assert sanity_loss.item() > 0


# ============================================================
# OPTIMIZER CHECK
# ============================================================

current_lr = optimizer.param_groups[
    0
][
    "lr"
]


assert abs(
    current_lr - LR
) < 1e-12


# ============================================================
# CLASS WEIGHT CHECK
# ============================================================

if USE_CLASS_WEIGHTED_LOSS:

    assert criterion.weight is not None

    assert criterion.weight.shape[
        0
    ] == NUM_CLASSES

    assert torch.isfinite(
        criterion.weight
    ).all()


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 76)
print("✅ CELL 12 TRAINING SETUP READY")
print("=" * 76)

print(
    "Next: Cell 13 "
    "Metrics + Train/Eval Functions"
)

In [ ]:
# ============================================================
# CELL 13 : METRICS + TRAIN / EVAL FUNCTIONS
# VocaLip Thai — Word-Level Baseline
#
# Metrics:
# - Loss
# - Accuracy
# - Macro F1
# - Top-3 Accuracy
#
# Training features:
# - AMP
# - Gradient clipping
# - tqdm progress bar
#
# Important:
# - Macro F1 is computed over all 44 classes
# - Validation/test have small counts for some classes,
#   so Macro F1 may fluctuate more than Accuracy
# ============================================================

from contextlib import nullcontext

import numpy as np
import torch

from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    f1_score
)


# ============================================================
# TOP-K ACCURACY
# ============================================================

def compute_topk_accuracy(
    logits,
    targets,
    k=3
):
    """
    logits:
        torch.Tensor (B, C)

    targets:
        torch.Tensor (B,)

    returns:
        number of correct Top-k predictions
    """

    k = min(
        int(k),
        logits.shape[1]
    )

    topk_indices = torch.topk(
        logits,
        k=k,
        dim=1
    ).indices

    correct = topk_indices.eq(
        targets.unsqueeze(1)
    ).any(
        dim=1
    )

    return int(
        correct.sum().item()
    )


# ============================================================
# FINAL METRIC CALCULATION
# ============================================================

def calculate_epoch_metrics(
    targets,
    predictions,
    top3_correct,
    total_samples
):
    """
    Compute classification metrics after one epoch.
    """

    targets = np.asarray(
        targets,
        dtype=np.int64
    )

    predictions = np.asarray(
        predictions,
        dtype=np.int64
    )


    accuracy = accuracy_score(
        targets,
        predictions
    )


    macro_f1 = f1_score(
        targets,
        predictions,
        labels=list(
            range(NUM_CLASSES)
        ),
        average="macro",
        zero_division=0
    )


    top3_accuracy = (
        float(top3_correct)
        / float(total_samples)
        if total_samples > 0
        else 0.0
    )


    return {
        "accuracy":
            float(accuracy),

        "macro_f1":
            float(macro_f1),

        "top3_accuracy":
            float(top3_accuracy)
    }


# ============================================================
# TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    scaler,
    device,
    epoch=None,
    show_progress=True
):
    """
    Train model for one epoch.

    Returns dict:
        loss
        accuracy
        macro_f1
        top3_accuracy
        lr
        grad_norm_mean
        grad_norm_max
    """

    model.train()


    running_loss = 0.0
    total_samples = 0

    all_targets = []
    all_predictions = []

    top3_correct = 0

    grad_norm_values = []


    # --------------------------------------------------------
    # PROGRESS BAR
    # --------------------------------------------------------

    description = (
        f"Train {epoch}"
        if epoch is not None
        else "Train"
    )


    iterator = tqdm(
        loader,
        desc=description,
        leave=False,
        disable=not show_progress
    )


    # --------------------------------------------------------
    # LOOP
    # --------------------------------------------------------

    for batch_data in iterator:

        videos = batch_data[
            "video"
        ].to(
            device,
            non_blocking=True
        )


        targets = batch_data[
            "label"
        ].to(
            device,
            non_blocking=True
        )


        batch_size_now = targets.size(
            0
        )


        # ----------------------------------------------------
        # RESET GRADIENTS
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )


        # ----------------------------------------------------
        # FORWARD + LOSS
        # ----------------------------------------------------

        with autocast_context():

            logits = model(
                videos
            )

            loss = criterion(
                logits,
                targets
            )


        # ----------------------------------------------------
        # SAFETY
        # ----------------------------------------------------

        if not torch.isfinite(
            loss
        ):

            raise RuntimeError(
                "Non-finite training loss detected: "
                f"{loss.item()}"
            )


        # ----------------------------------------------------
        # BACKPROP
        # ----------------------------------------------------

        if USE_AMP:

            scaler.scale(
                loss
            ).backward()


            # Unscale before gradient clipping
            scaler.unscale_(
                optimizer
            )


            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=GRAD_CLIP_NORM
            )


            scaler.step(
                optimizer
            )


            scaler.update()


        else:

            loss.backward()


            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=GRAD_CLIP_NORM
            )


            optimizer.step()


        # ----------------------------------------------------
        # GRADIENT NORM
        # ----------------------------------------------------

        if torch.is_tensor(
            grad_norm
        ):

            grad_norm_value = float(
                grad_norm.detach().item()
            )

        else:

            grad_norm_value = float(
                grad_norm
            )


        if np.isfinite(
            grad_norm_value
        ):

            grad_norm_values.append(
                grad_norm_value
            )


        # ----------------------------------------------------
        # METRICS
        # ----------------------------------------------------

        predictions = torch.argmax(
            logits,
            dim=1
        )


        top3_correct += compute_topk_accuracy(
            logits,
            targets,
            k=3
        )


        running_loss += (
            loss.item()
            * batch_size_now
        )


        total_samples += batch_size_now


        all_targets.extend(
            targets.detach()
            .cpu()
            .numpy()
            .tolist()
        )


        all_predictions.extend(
            predictions.detach()
            .cpu()
            .numpy()
            .tolist()
        )


        # ----------------------------------------------------
        # LIVE PROGRESS
        # ----------------------------------------------------

        current_loss = (
            running_loss
            / total_samples
        )


        current_accuracy = (
            np.mean(
                np.asarray(
                    all_targets
                )
                ==
                np.asarray(
                    all_predictions
                )
            )
        )


        iterator.set_postfix(
            loss=f"{current_loss:.4f}",
            acc=f"{current_accuracy:.3f}"
        )


    # --------------------------------------------------------
    # FINAL EPOCH METRICS
    # --------------------------------------------------------

    epoch_loss = (
        running_loss
        / max(
            total_samples,
            1
        )
    )


    metrics = calculate_epoch_metrics(
        targets=all_targets,
        predictions=all_predictions,
        top3_correct=top3_correct,
        total_samples=total_samples
    )


    metrics[
        "loss"
    ] = float(
        epoch_loss
    )


    metrics[
        "lr"
    ] = float(
        optimizer.param_groups[
            0
        ][
            "lr"
        ]
    )


    metrics[
        "grad_norm_mean"
    ] = (
        float(
            np.mean(
                grad_norm_values
            )
        )
        if len(
            grad_norm_values
        ) > 0
        else 0.0
    )


    metrics[
        "grad_norm_max"
    ] = (
        float(
            np.max(
                grad_norm_values
            )
        )
        if len(
            grad_norm_values
        ) > 0
        else 0.0
    )


    return metrics


# ============================================================
# EVALUATE
# ============================================================

@torch.no_grad()
def evaluate_model(
    model,
    loader,
    criterion,
    device,
    split_name="Val",
    show_progress=True,
    return_predictions=False
):
    """
    Evaluate model.

    Returns metrics dict.

    Optional:
        predictions
        targets
        probabilities
    """

    model.eval()


    running_loss = 0.0
    total_samples = 0

    all_targets = []
    all_predictions = []
    all_probabilities = []

    top3_correct = 0


    # --------------------------------------------------------
    # PROGRESS
    # --------------------------------------------------------

    iterator = tqdm(
        loader,
        desc=split_name,
        leave=False,
        disable=not show_progress
    )


    # --------------------------------------------------------
    # LOOP
    # --------------------------------------------------------

    for batch_data in iterator:

        videos = batch_data[
            "video"
        ].to(
            device,
            non_blocking=True
        )


        targets = batch_data[
            "label"
        ].to(
            device,
            non_blocking=True
        )


        batch_size_now = targets.size(
            0
        )


        # ----------------------------------------------------
        # FORWARD
        # ----------------------------------------------------

        with autocast_context():

            logits = model(
                videos
            )

            loss = criterion(
                logits,
                targets
            )


        # ----------------------------------------------------
        # SAFETY
        # ----------------------------------------------------

        if not torch.isfinite(
            loss
        ):

            raise RuntimeError(
                f"Non-finite {split_name} loss detected: "
                f"{loss.item()}"
            )


        probabilities = torch.softmax(
            logits.float(),
            dim=1
        )


        predictions = torch.argmax(
            probabilities,
            dim=1
        )


        # ----------------------------------------------------
        # ACCUMULATE
        # ----------------------------------------------------

        running_loss += (
            loss.item()
            * batch_size_now
        )


        total_samples += batch_size_now


        top3_correct += compute_topk_accuracy(
            probabilities,
            targets,
            k=3
        )


        all_targets.extend(
            targets.detach()
            .cpu()
            .numpy()
            .tolist()
        )


        all_predictions.extend(
            predictions.detach()
            .cpu()
            .numpy()
            .tolist()
        )


        if return_predictions:

            all_probabilities.append(
                probabilities.detach()
                .cpu()
                .numpy()
            )


        # ----------------------------------------------------
        # LIVE PROGRESS
        # ----------------------------------------------------

        current_loss = (
            running_loss
            / total_samples
        )


        current_accuracy = (
            np.mean(
                np.asarray(
                    all_targets
                )
                ==
                np.asarray(
                    all_predictions
                )
            )
        )


        iterator.set_postfix(
            loss=f"{current_loss:.4f}",
            acc=f"{current_accuracy:.3f}"
        )


    # --------------------------------------------------------
    # FINAL METRICS
    # --------------------------------------------------------

    epoch_loss = (
        running_loss
        / max(
            total_samples,
            1
        )
    )


    metrics = calculate_epoch_metrics(
        targets=all_targets,
        predictions=all_predictions,
        top3_correct=top3_correct,
        total_samples=total_samples
    )


    metrics[
        "loss"
    ] = float(
        epoch_loss
    )


    # --------------------------------------------------------
    # RETURN
    # --------------------------------------------------------

    if return_predictions:

        if len(
            all_probabilities
        ) > 0:

            all_probabilities = np.concatenate(
                all_probabilities,
                axis=0
            )

        else:

            all_probabilities = np.empty(
                (
                    0,
                    NUM_CLASSES
                ),
                dtype=np.float32
            )


        return (
            metrics,
            np.asarray(
                all_targets,
                dtype=np.int64
            ),
            np.asarray(
                all_predictions,
                dtype=np.int64
            ),
            all_probabilities
        )


    return metrics


# ============================================================
# METRIC FORMATTER
# ============================================================

def format_metrics(
    metrics
):
    """
    Compact human-readable metrics.
    """

    text = (
        f"loss={metrics['loss']:.4f} | "
        f"acc={metrics['accuracy']:.4f} | "
        f"macroF1={metrics['macro_f1']:.4f} | "
        f"top3={metrics['top3_accuracy']:.4f}"
    )


    if "lr" in metrics:

        text += (
            f" | lr={metrics['lr']:.2e}"
        )


    return text


# ============================================================
# EVALUATION SMOKE TEST
# Use one small subset of the existing validation loader.
# No training performed here.
# ============================================================

print("=" * 76)
print("METRICS + TRAIN/EVAL FUNCTIONS")
print("=" * 76)


print(
    "Metrics:"
)

print(
    "  - Accuracy"
)

print(
    "  - Macro F1"
)

print(
    "  - Top-3 Accuracy"
)

print()

print(
    f"AMP              : {USE_AMP}"
)

print(
    f"Gradient clipping: {GRAD_CLIP_NORM}"
)


# ============================================================
# ONE-BATCH METRIC FUNCTION TEST
# ============================================================

model.eval()


smoke_batch = next(
    iter(
        val_loader
    )
)


with torch.no_grad():

    smoke_videos = smoke_batch[
        "video"
    ].to(
        DEVICE,
        non_blocking=True
    )


    smoke_targets = smoke_batch[
        "label"
    ].to(
        DEVICE,
        non_blocking=True
    )


    with autocast_context():

        smoke_logits = model(
            smoke_videos
        )

        smoke_loss = criterion(
            smoke_logits,
            smoke_targets
        )


smoke_predictions = torch.argmax(
    smoke_logits,
    dim=1
)


smoke_top3 = compute_topk_accuracy(
    smoke_logits,
    smoke_targets,
    k=3
)


smoke_metrics = calculate_epoch_metrics(

    targets=(
        smoke_targets
        .detach()
        .cpu()
        .numpy()
    ),

    predictions=(
        smoke_predictions
        .detach()
        .cpu()
        .numpy()
    ),

    top3_correct=smoke_top3,

    total_samples=len(
        smoke_targets
    )
)


smoke_metrics[
    "loss"
] = float(
    smoke_loss.item()
)


print()
print("=" * 76)
print("ONE-BATCH SMOKE TEST")
print("=" * 76)

print(
    f"Batch samples : "
    f"{len(smoke_targets)}"
)

print(
    format_metrics(
        smoke_metrics
    )
)


# ============================================================
# HARD CHECKS
# ============================================================

assert np.isfinite(
    smoke_metrics[
        "loss"
    ]
)


assert (
    0.0
    <= smoke_metrics[
        "accuracy"
    ]
    <= 1.0
)


assert (
    0.0
    <= smoke_metrics[
        "macro_f1"
    ]
    <= 1.0
)


assert (
    0.0
    <= smoke_metrics[
        "top3_accuracy"
    ]
    <= 1.0
)


assert (
    smoke_metrics[
        "top3_accuracy"
    ]
    + 1e-12
    >= smoke_metrics[
        "accuracy"
    ]
)


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 76)
print("✅ CELL 13 TRAIN / EVAL PIPELINE READY")
print("=" * 76)

print(
    "No model training was performed in this cell."
)

print()

print(
    "Next: Cell 14 "
    "Sanity Overfit Test"
)

In [ ]:
# ============================================================
# CELL 14 : CONTROLLED SANITY OVERFIT TEST — V2
# VocaLip Thai — Word-Level Baseline
#
# Goal:
# Test whether the complete training pipeline can memorize
# a tiny, FIXED and NON-AUGMENTED dataset.
#
# Design:
# - 4 classes
# - 8 samples / class
# - 32 samples total
# - NO augmentation
# - Fresh temporary model
# - Ordinary CrossEntropyLoss
# - No weight decay
# - Up to 100 epochs
#
# IMPORTANT:
# Full training must NOT start unless this test passes.
# ============================================================

import gc
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    f1_score
)


# ============================================================
# SANITY CONFIG
# ============================================================

SANITY_NUM_CLASSES = 4
SANITY_SAMPLES_PER_CLASS = 8

SANITY_BATCH_SIZE = 16
SANITY_EPOCHS = 100

SANITY_LR = 1e-3

SANITY_PASS_ACC = 0.95
SANITY_STRONG_PASS_ACC = 0.999


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )


# ============================================================
# SELECT CLASSES
#
# Pick classes with enough samples.
# Prefer classes near median frequency rather than only
# the largest classes.
# ============================================================

class_counts = (
    train_df[
        "model_class_id"
    ]
    .astype(int)
    .value_counts()
    .sort_index()
)


eligible_classes = class_counts[
    class_counts
    >= SANITY_SAMPLES_PER_CLASS
].index.tolist()


assert (
    len(eligible_classes)
    >= SANITY_NUM_CLASSES
), (
    "Not enough eligible classes for sanity test."
)


# ------------------------------------------------------------
# Sort eligible classes by frequency
# and choose evenly distributed positions
# ------------------------------------------------------------

eligible_sorted = sorted(
    eligible_classes,
    key=lambda x: class_counts.loc[x]
)


positions = np.linspace(
    0,
    len(eligible_sorted) - 1,
    SANITY_NUM_CLASSES,
    dtype=int
)


selected_classes = [
    int(
        eligible_sorted[pos]
    )
    for pos in positions
]


# ============================================================
# BUILD BALANCED FIXED SANITY DATAFRAME
# ============================================================

sanity_parts = []


for class_id in selected_classes:

    class_rows = train_df[
        train_df[
            "model_class_id"
        ].astype(int)
        == class_id
    ]


    class_sample = class_rows.sample(
        n=SANITY_SAMPLES_PER_CLASS,
        random_state=(
            SEED
            + class_id
        )
    )


    sanity_parts.append(
        class_sample
    )


sanity_df = pd.concat(
    sanity_parts,
    axis=0,
    ignore_index=True
)


# Shuffle once only
sanity_df = sanity_df.sample(
    frac=1.0,
    random_state=SEED
).reset_index(
    drop=True
)


# ============================================================
# IMPORTANT:
# Create a NEW dataset with training=False.
#
# This guarantees:
# - no brightness augmentation
# - no contrast augmentation
# - no noise
# - no blur
# - no temporal jitter
#
# The model sees the exact same cached samples every epoch.
# ============================================================

sanity_dataset = VocaLipWordDataset(
    sanity_df,
    training=False
)


sanity_loader = DataLoader(

    sanity_dataset,

    batch_size=SANITY_BATCH_SIZE,

    shuffle=True,

    num_workers=0,

    pin_memory=(
        DEVICE.type == "cuda"
    ),

    drop_last=False
)


sanity_eval_loader = DataLoader(

    sanity_dataset,

    batch_size=SANITY_BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=(
        DEVICE.type == "cuda"
    ),

    drop_last=False
)


# ============================================================
# SHOW DATASET
# ============================================================

print("=" * 76)
print("CONTROLLED SANITY DATASET")
print("=" * 76)

print(
    f"Samples          : {len(sanity_df)}"
)

print(
    f"Classes          : "
    f"{sanity_df['model_class_id'].nunique()}"
)

print(
    f"Samples / class  : "
    f"{SANITY_SAMPLES_PER_CLASS}"
)

print(
    f"Augmentation     : DISABLED"
)

print()


sanity_class_table = (
    sanity_df[
        [
            "model_class_id",
            "word"
        ]
    ]
    .value_counts()
    .reset_index(
        name="samples"
    )
    .sort_values(
        "model_class_id"
    )
)


display(
    sanity_class_table
)


# ============================================================
# VERIFY INPUT DETERMINISM
#
# Same dataset index should produce exactly same tensor.
# ============================================================

check_a = sanity_dataset[
    0
][
    "video"
]


check_b = sanity_dataset[
    0
][
    "video"
]


deterministic_input = torch.equal(
    check_a,
    check_b
)


print()
print("=" * 76)
print("INPUT DETERMINISM CHECK")
print("=" * 76)

print(
    f"Same sample identical : "
    f"{deterministic_input}"
)


assert deterministic_input, (
    "Sanity dataset is not deterministic."
)


# ============================================================
# VERIFY LABELS
# ============================================================

unique_sanity_labels = sorted(
    sanity_df[
        "model_class_id"
    ]
    .astype(int)
    .unique()
    .tolist()
)


assert len(
    unique_sanity_labels
) == SANITY_NUM_CLASSES


print(
    f"Selected class IDs    : "
    f"{unique_sanity_labels}"
)


# ============================================================
# FRESH TEMP MODEL
# ============================================================

sanity_model = VocaLipWordBaseline(

    num_classes=NUM_CLASSES,

    feature_dim=FEATURE_DIM,

    gru_hidden=GRU_HIDDEN,

    gru_layers=GRU_LAYERS,

    dropout=MODEL_DROPOUT

).to(
    DEVICE
)


# ============================================================
# LOSS
#
# Use plain CE.
# Class imbalance does not exist in this sanity subset.
# ============================================================

sanity_criterion = nn.CrossEntropyLoss()


# ============================================================
# OPTIMIZER
# ============================================================

sanity_optimizer = torch.optim.AdamW(

    sanity_model.parameters(),

    lr=SANITY_LR,

    weight_decay=0.0
)


# ============================================================
# AMP
# ============================================================

try:

    sanity_scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP
    )

except Exception:

    sanity_scaler = (
        torch.cuda.amp.GradScaler(
            enabled=USE_AMP
        )
    )


# ============================================================
# EVALUATION FUNCTION
# ============================================================

@torch.no_grad()
def evaluate_sanity(
    model,
    loader
):

    model.eval()


    total_loss = 0.0
    total_count = 0

    targets_all = []
    predictions_all = []


    for batch_data in loader:

        videos = batch_data[
            "video"
        ].to(
            DEVICE,
            non_blocking=True
        )


        targets = batch_data[
            "label"
        ].to(
            DEVICE,
            non_blocking=True
        )


        with autocast_context():

            logits = model(
                videos
            )

            loss = sanity_criterion(
                logits,
                targets
            )


        predictions = torch.argmax(
            logits,
            dim=1
        )


        n = targets.size(
            0
        )


        total_loss += (
            loss.item()
            * n
        )

        total_count += n


        targets_all.extend(
            targets.detach()
            .cpu()
            .numpy()
            .tolist()
        )


        predictions_all.extend(
            predictions.detach()
            .cpu()
            .numpy()
            .tolist()
        )


    targets_all = np.asarray(
        targets_all,
        dtype=np.int64
    )


    predictions_all = np.asarray(
        predictions_all,
        dtype=np.int64
    )


    accuracy = accuracy_score(
        targets_all,
        predictions_all
    )


    macro_f1 = f1_score(

        targets_all,

        predictions_all,

        labels=unique_sanity_labels,

        average="macro",

        zero_division=0
    )


    return {

        "loss":
            float(
                total_loss
                / max(
                    total_count,
                    1
                )
            ),

        "accuracy":
            float(
                accuracy
            ),

        "macro_f1":
            float(
                macro_f1
            )
    }


# ============================================================
# INITIAL PERFORMANCE
# ============================================================

initial_metrics = evaluate_sanity(
    sanity_model,
    sanity_eval_loader
)


print()
print("=" * 76)
print("INITIAL SANITY PERFORMANCE")
print("=" * 76)

print(
    f"Loss     : "
    f"{initial_metrics['loss']:.4f}"
)

print(
    f"Accuracy : "
    f"{initial_metrics['accuracy']:.4f}"
)

print(
    f"Macro F1 : "
    f"{initial_metrics['macro_f1']:.4f}"
)


# ============================================================
# TRAIN LOOP
# ============================================================

history = []


for epoch in range(
    1,
    SANITY_EPOCHS + 1
):

    sanity_model.train()


    running_loss = 0.0
    running_correct = 0
    running_count = 0


    for batch_data in sanity_loader:

        videos = batch_data[
            "video"
        ].to(
            DEVICE,
            non_blocking=True
        )


        targets = batch_data[
            "label"
        ].to(
            DEVICE,
            non_blocking=True
        )


        sanity_optimizer.zero_grad(
            set_to_none=True
        )


        # ----------------------------------------------------
        # FORWARD
        # ----------------------------------------------------

        with autocast_context():

            logits = sanity_model(
                videos
            )

            loss = sanity_criterion(
                logits,
                targets
            )


        if not torch.isfinite(
            loss
        ):

            raise RuntimeError(
                f"Non-finite loss at "
                f"epoch {epoch}"
            )


        # ----------------------------------------------------
        # BACKWARD
        # ----------------------------------------------------

        if USE_AMP:

            sanity_scaler.scale(
                loss
            ).backward()


            sanity_scaler.unscale_(
                sanity_optimizer
            )


            torch.nn.utils.clip_grad_norm_(
                sanity_model.parameters(),
                GRAD_CLIP_NORM
            )


            sanity_scaler.step(
                sanity_optimizer
            )


            sanity_scaler.update()


        else:

            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                sanity_model.parameters(),
                GRAD_CLIP_NORM
            )


            sanity_optimizer.step()


        # ----------------------------------------------------
        # TRAIN METRICS
        # ----------------------------------------------------

        predictions = torch.argmax(
            logits,
            dim=1
        )


        n = targets.size(
            0
        )


        running_loss += (
            loss.item()
            * n
        )


        running_correct += int(
            (
                predictions
                == targets
            )
            .sum()
            .item()
        )


        running_count += n


    train_loss = (
        running_loss
        / running_count
    )


    train_acc = (
        running_correct
        / running_count
    )


    # --------------------------------------------------------
    # DETERMINISTIC EVAL
    # --------------------------------------------------------

    eval_metrics = evaluate_sanity(
        sanity_model,
        sanity_eval_loader
    )


    history.append({

        "epoch":
            epoch,

        "train_loss":
            train_loss,

        "train_acc":
            train_acc,

        "eval_loss":
            eval_metrics[
                "loss"
            ],

        "eval_acc":
            eval_metrics[
                "accuracy"
            ],

        "macro_f1":
            eval_metrics[
                "macro_f1"
            ]
    })


    # --------------------------------------------------------
    # PRINT
    # --------------------------------------------------------

    if (
        epoch == 1
        or epoch % 5 == 0
        or eval_metrics[
            "accuracy"
        ] >= SANITY_PASS_ACC
    ):

        print(

            f"Epoch {epoch:03d}/"
            f"{SANITY_EPOCHS} | "

            f"train_loss="
            f"{train_loss:.4f} | "

            f"train_acc="
            f"{train_acc:.4f} | "

            f"eval_loss="
            f"{eval_metrics['loss']:.4f} | "

            f"eval_acc="
            f"{eval_metrics['accuracy']:.4f} | "

            f"F1="
            f"{eval_metrics['macro_f1']:.4f}"
        )


    # --------------------------------------------------------
    # STRONG EARLY SUCCESS
    # --------------------------------------------------------

    if (
        eval_metrics[
            "accuracy"
        ] >= SANITY_STRONG_PASS_ACC
        and epoch >= 5
    ):

        print()
        print(
            f"✅ 100% memorization reached "
            f"at epoch {epoch}."
        )

        break


# ============================================================
# FINAL EVALUATION
# ============================================================

final_metrics = evaluate_sanity(
    sanity_model,
    sanity_eval_loader
)


print()
print("=" * 76)
print("CONTROLLED SANITY OVERFIT RESULT")
print("=" * 76)

print(
    f"Samples        : "
    f"{len(sanity_dataset)}"
)

print(
    f"Classes        : "
    f"{SANITY_NUM_CLASSES}"
)

print(
    f"Final loss     : "
    f"{final_metrics['loss']:.6f}"
)

print(
    f"Final accuracy : "
    f"{final_metrics['accuracy']:.4f}"
)

print(
    f"Final Macro F1 : "
    f"{final_metrics['macro_f1']:.4f}"
)


# ============================================================
# LEARNING PROGRESS
# ============================================================

first_eval_loss = history[
    0
][
    "eval_loss"
]


last_eval_loss = history[
    -1
][
    "eval_loss"
]


loss_reduction = (
    first_eval_loss
    - last_eval_loss
)


print()
print(
    f"Eval loss reduction : "
    f"{loss_reduction:.6f}"
)


# ============================================================
# DECISION
# ============================================================

print()
print("=" * 76)
print("SANITY DECISION")
print("=" * 76)


SANITY_OVERFIT_PASSED = (
    final_metrics[
        "accuracy"
    ]
    >= SANITY_PASS_ACC
)


if (
    final_metrics[
        "accuracy"
    ]
    >= 0.999
):

    print(
        "✅ STRONG PASS"
    )

    print(
        "Model memorized the fixed tiny dataset "
        "essentially perfectly."
    )

    print(
        "Forward / labels / loss / backward / "
        "optimizer pipeline is functioning."
    )


elif SANITY_OVERFIT_PASSED:

    print(
        "✅ SANITY OVERFIT PASSED"
    )

    print(
        "The controlled pipeline can memorize "
        "the tiny fixed dataset."
    )


else:

    print(
        "❌ SANITY OVERFIT STILL FAILED"
    )

    print(
        "Do NOT run Cell 15."
    )

    print(
        "Because augmentation has now been removed, "
        "the next investigation should focus on:"
    )

    print(
        "1. frame encoder architecture"
    )

    print(
        "2. feature collapse"
    )

    print(
        "3. gradient/update behavior"
    )

    print(
        "4. duplicate or visually identical samples"
    )

    print(
        "5. label / cache consistency"
    )


# ============================================================
# SAVE HISTORY IN MEMORY
# ============================================================

sanity_history_v2 = pd.DataFrame(
    history
)


# ============================================================
# CLEAN TEMP GPU OBJECTS
# ============================================================

del sanity_model
del sanity_optimizer
del sanity_scaler

gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ============================================================
# FINAL GATE
# ============================================================

print()
print("=" * 76)

if SANITY_OVERFIT_PASSED:

    print(
        "✅ SANITY GATE OPEN"
    )

    print(
        "Next step can proceed to Cell 15."
    )

else:

    print(
        "⛔ SANITY GATE CLOSED"
    )

    print(
        "Stop here and diagnose before full training."
    )

print("=" * 76)

In [ ]:
# ============================================================
# CELL 14.1 : CORRECT SANITY OVERFIT DECISION
# VocaLip Thai — Word-Level Baseline
#
# Important correction:
# Sanity overfit should be judged from BEST achieved
# deterministic accuracy, not only the final epoch.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# CHECK HISTORY
# ============================================================

assert "sanity_history_v2" in globals(), (
    "sanity_history_v2 not found. "
    "Run Cell 14 V2 first."
)

assert len(sanity_history_v2) > 0


required_columns = [
    "epoch",
    "train_loss",
    "train_acc",
    "eval_loss",
    "eval_acc",
    "macro_f1"
]


for col in required_columns:

    assert col in sanity_history_v2.columns, (
        f"Missing column: {col}"
    )


# ============================================================
# BEST EPOCH BY DETERMINISTIC EVAL ACCURACY
#
# Tie-break:
# 1. highest eval accuracy
# 2. highest Macro F1
# 3. lowest eval loss
# ============================================================

best_rows = (
    sanity_history_v2
    .sort_values(
        by=[
            "eval_acc",
            "macro_f1",
            "eval_loss"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


best_row = best_rows.iloc[
    0
]


best_epoch = int(
    best_row[
        "epoch"
    ]
)

best_train_acc = float(
    best_row[
        "train_acc"
    ]
)

best_eval_acc = float(
    best_row[
        "eval_acc"
    ]
)

best_macro_f1 = float(
    best_row[
        "macro_f1"
    ]
)

best_eval_loss = float(
    best_row[
        "eval_loss"
    ]
)


# ============================================================
# PEAK TRAIN ACCURACY
# ============================================================

max_train_acc = float(
    sanity_history_v2[
        "train_acc"
    ].max()
)


# ============================================================
# FIRST EPOCH CROSSING 95%
# ============================================================

pass_rows = sanity_history_v2[
    sanity_history_v2[
        "eval_acc"
    ] >= SANITY_PASS_ACC
]


if len(pass_rows) > 0:

    first_pass_epoch = int(
        pass_rows.iloc[
            0
        ][
            "epoch"
        ]
    )

else:

    first_pass_epoch = None


# ============================================================
# CORRECT SANITY DECISION
# ============================================================

SANITY_OVERFIT_PASSED = (
    best_eval_acc
    >= SANITY_PASS_ACC
)


# ============================================================
# SUMMARY
# ============================================================

print("=" * 76)
print("CORRECTED SANITY OVERFIT ASSESSMENT")
print("=" * 76)

print(
    f"Peak train accuracy : "
    f"{max_train_acc:.4f}"
)

print(
    f"Best eval epoch     : "
    f"{best_epoch}"
)

print(
    f"Best eval loss      : "
    f"{best_eval_loss:.6f}"
)

print(
    f"Best eval accuracy  : "
    f"{best_eval_acc:.4f}"
)

print(
    f"Best Macro F1       : "
    f"{best_macro_f1:.4f}"
)

print(
    f"Pass threshold      : "
    f"{SANITY_PASS_ACC:.4f}"
)

print(
    f"First >=95% epoch   : "
    f"{first_pass_epoch}"
)


# ============================================================
# DECISION
# ============================================================

print()
print("=" * 76)
print("FINAL SANITY DECISION")
print("=" * 76)


if SANITY_OVERFIT_PASSED:

    print(
        "✅ SANITY OVERFIT PASSED"
    )

    print(
        "The model successfully memorized "
        "the controlled tiny dataset."
    )

    print(
        "The training pipeline is functional."
    )

    print(
        "The previous FAIL message was caused "
        "by evaluating only the final epoch."
    )

else:

    print(
        "❌ SANITY OVERFIT FAILED"
    )

    print(
        "Do not start full training."
    )


# ============================================================
# INSTABILITY INFORMATION
# ============================================================

last_row = sanity_history_v2.iloc[
    -1
]


last_eval_acc = float(
    last_row[
        "eval_acc"
    ]
)


drop_from_best = (
    best_eval_acc
    - last_eval_acc
)


print()
print("=" * 76)
print("SANITY STABILITY NOTE")
print("=" * 76)

print(
    f"Best eval accuracy : "
    f"{best_eval_acc:.4f}"
)

print(
    f"Final eval accuracy: "
    f"{last_eval_acc:.4f}"
)

print(
    f"Drop from best     : "
    f"{drop_from_best:.4f}"
)


if drop_from_best >= 0.10:

    print()
    print(
        "⚠️ Tiny-set training became unstable "
        "after reaching its best solution."
    )

    print(
        "This does NOT invalidate the sanity test."
    )

    print(
        "For full training we will use:"
    )

    print(
        "  - validation monitoring"
    )

    print(
        "  - best-checkpoint saving"
    )

    print(
        "  - ReduceLROnPlateau"
    )

    print(
        "  - early stopping"
    )


# ============================================================
# FINAL GATE
# ============================================================

print()
print("=" * 76)

if SANITY_OVERFIT_PASSED:

    print(
        "✅ SANITY GATE OPEN"
    )

    print(
        "Ready to prepare Cell 15 Full Training."
    )

else:

    print(
        "⛔ SANITY GATE CLOSED"
    )

print("=" * 76)

In [ ]:
# ============================================================
# CELL 14.2 : MOUTH ROI VISUAL LEAKAGE AUDIT
# VocaLip Thai — Word-Level Baseline
#
# Purpose:
# - Audit cached mouth ROI before full training
# - Look for subtitles / burned-in text
# - Look for severe mouth occlusion
# - Look for wrong ROI / face / crop
#
# Sampling:
# - 48 clips total
# - stratified across train / val / test
# - deterministic
#
# Each sample shows:
# - early frame
# - middle frame
# - late frame
#
# IMPORTANT:
# This cell does NOT modify the dataset.
# ============================================================

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path


# ============================================================
# CONFIG
# ============================================================

AUDIT_TOTAL = 48

AUDIT_PER_SPLIT = {
    "train": 24,
    "val": 12,
    "test": 12
}

AUDIT_RANDOM_STATE = SEED

SAMPLES_PER_PAGE = 8


# ============================================================
# CHECK CLEAN DATAFRAME
# ============================================================

assert "df" in globals(), (
    "Clean dataframe `df` not found. "
    "Run Cell 8 first."
)


required_columns = [
    "split",
    "word",
    "model_class_id",
    "mouth_cache_path"
]


for col in required_columns:

    assert col in df.columns, (
        f"Missing required column: {col}"
    )


print("=" * 76)
print("MOUTH ROI VISUAL LEAKAGE AUDIT")
print("=" * 76)

print(
    f"Dataset samples : {len(df):,}"
)

print(
    f"Audit samples   : {AUDIT_TOTAL}"
)

print(
    f"Train           : {AUDIT_PER_SPLIT['train']}"
)

print(
    f"Val             : {AUDIT_PER_SPLIT['val']}"
)

print(
    f"Test            : {AUDIT_PER_SPLIT['test']}"
)


# ============================================================
# BALANCED CLASS-AWARE SAMPLING
#
# We try to maximize class diversity rather than simply
# randomly selecting many clips from frequent classes.
# ============================================================

def sample_split_for_audit(
    split_df,
    n_samples,
    seed
):

    split_df = split_df.copy()

    available_classes = (
        split_df[
            "model_class_id"
        ]
        .astype(int)
        .unique()
        .tolist()
    )

    rng = np.random.default_rng(
        seed
    )

    rng.shuffle(
        available_classes
    )


    selected_indices = []


    # --------------------------------------------------------
    # PASS 1:
    # one sample from as many different classes as possible
    # --------------------------------------------------------

    for class_id in available_classes:

        class_rows = split_df[
            split_df[
                "model_class_id"
            ].astype(int)
            == int(class_id)
        ]


        if len(class_rows) == 0:

            continue


        random_state = int(
            seed
            + int(class_id)
            + len(selected_indices)
        )


        selected = class_rows.sample(
            n=1,
            random_state=random_state
        )


        selected_indices.append(
            selected.index[0]
        )


        if len(selected_indices) >= n_samples:

            break


    # --------------------------------------------------------
    # PASS 2:
    # fill remaining slots randomly
    # --------------------------------------------------------

    if len(selected_indices) < n_samples:

        remaining = split_df[
            ~split_df.index.isin(
                selected_indices
            )
        ]


        remaining_needed = min(
            n_samples - len(selected_indices),
            len(remaining)
        )


        if remaining_needed > 0:

            extra = remaining.sample(
                n=remaining_needed,
                random_state=seed + 999
            )


            selected_indices.extend(
                extra.index.tolist()
            )


    return split_df.loc[
        selected_indices
    ].copy()


# ============================================================
# SAMPLE EACH SPLIT
# ============================================================

audit_parts = []


for split_idx, split_name in enumerate(
    [
        "train",
        "val",
        "test"
    ]
):

    split_df = df[
        df[
            "split"
        ].astype(str)
        == split_name
    ]


    n_requested = AUDIT_PER_SPLIT[
        split_name
    ]


    sampled = sample_split_for_audit(

        split_df=split_df,

        n_samples=n_requested,

        seed=(
            AUDIT_RANDOM_STATE
            + split_idx * 1000
        )
    )


    audit_parts.append(
        sampled
    )


audit_df = pd.concat(
    audit_parts,
    axis=0
).reset_index(
    drop=True
)


# ============================================================
# CHECK SAMPLE
# ============================================================

print()
print("=" * 76)
print("AUDIT SAMPLE DISTRIBUTION")
print("=" * 76)


for split_name in [
    "train",
    "val",
    "test"
]:

    subset = audit_df[
        audit_df[
            "split"
        ] == split_name
    ]


    print(
        f"{split_name:<5} | "
        f"samples={len(subset):>2} | "
        f"classes="
        f"{subset['model_class_id'].nunique():>2}"
    )


print()

print(
    f"Total unique classes in audit : "
    f"{audit_df['model_class_id'].nunique()}/"
    f"{NUM_CLASSES}"
)


# ============================================================
# CACHE LOADER
# ============================================================

def load_audit_cache(
    cache_path
):

    cache_path = Path(
        str(
            cache_path
        )
    )


    if not cache_path.exists():

        raise FileNotFoundError(
            str(
                cache_path
            )
        )


    with np.load(
        cache_path,
        allow_pickle=False
    ) as npz:

        frames = npz[
            "frames"
        ]


    assert (
        frames.ndim == 4
    ), (
        f"Unexpected cache shape: "
        f"{frames.shape}"
    )


    return frames


# ============================================================
# FRAME POSITIONS
# ============================================================

def get_three_frames(
    frames
):

    T = len(
        frames
    )


    positions = [
        int(
            round(
                (T - 1) * 0.20
            )
        ),

        int(
            round(
                (T - 1) * 0.50
            )
        ),

        int(
            round(
                (T - 1) * 0.80
            )
        )
    ]


    return (
        positions,
        [
            frames[
                idx
            ]
            for idx in positions
        ]
    )


# ============================================================
# DISPLAY AUDIT PAGES
#
# One row = one clip
# Three columns = early / middle / late
# ============================================================

num_pages = math.ceil(
    len(audit_df)
    / SAMPLES_PER_PAGE
)


for page_idx in range(
    num_pages
):

    page_start = (
        page_idx
        * SAMPLES_PER_PAGE
    )

    page_end = min(
        page_start
        + SAMPLES_PER_PAGE,

        len(
            audit_df
        )
    )


    page_df = audit_df.iloc[
        page_start:page_end
    ]


    n_rows = len(
        page_df
    )


    fig, axes = plt.subplots(

        n_rows,

        3,

        figsize=(
            11,
            2.5 * n_rows
        )
    )


    if n_rows == 1:

        axes = np.expand_dims(
            axes,
            axis=0
        )


    for row_plot, (
        audit_index,
        row
    ) in enumerate(
        page_df.iterrows()
    ):

        try:

            frames = load_audit_cache(
                row[
                    "mouth_cache_path"
                ]
            )


            positions, selected_frames = (
                get_three_frames(
                    frames
                )
            )


            for col_idx in range(
                3
            ):

                ax = axes[
                    row_plot,
                    col_idx
                ]


                frame = selected_frames[
                    col_idx
                ]


                ax.imshow(
                    frame
                )


                if col_idx == 0:

                    title = (
                        f"#{audit_index:02d} | "
                        f"{row['split']} | "
                        f"ID {int(row['model_class_id']):02d}\n"
                        f"{row['word']} | "
                        f"frame {positions[col_idx]}"
                    )

                else:

                    title = (
                        f"frame "
                        f"{positions[col_idx]}"
                    )


                ax.set_title(
                    title,
                    fontsize=9
                )


                ax.axis(
                    "off"
                )


        except Exception as e:

            for col_idx in range(
                3
            ):

                ax = axes[
                    row_plot,
                    col_idx
                ]


                ax.text(

                    0.5,
                    0.5,

                    f"LOAD ERROR\n"
                    f"{type(e).__name__}",

                    ha="center",
                    va="center"
                )


                ax.axis(
                    "off"
                )


    fig.suptitle(

        (
            "VocaLip Thai — "
            "Mouth ROI Leakage Audit\n"
            f"Page {page_idx + 1}/{num_pages}"
        ),

        fontsize=14
    )


    plt.tight_layout()

    plt.show()


# ============================================================
# CREATE MANUAL AUDIT TABLE
#
# We intentionally do not try to automatically detect Thai
# subtitles here because a weak heuristic could give false
# confidence.
#
# User can mark suspicious rows after visual inspection.
# ============================================================

audit_review = audit_df[
    [
        "split",
        "model_class_id",
        "word",
        "mouth_cache_path"
    ]
].copy()


audit_review.insert(
    0,
    "audit_id",
    np.arange(
        len(
            audit_review
        )
    )
)


audit_review[
    "subtitle_visible"
] = ""


audit_review[
    "mouth_occluded"
] = ""


audit_review[
    "wrong_roi"
] = ""


audit_review[
    "notes"
] = ""


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 76)
print("VISUAL AUDIT CHECKLIST")
print("=" * 76)

print(
    "Inspect all displayed samples for:"
)

print(
    "  1. Thai/English subtitles inside mouth ROI"
)

print(
    "  2. Text crossing or covering the lips"
)

print(
    "  3. Wrong person's mouth"
)

print(
    "  4. ROI not centered on the mouth"
)

print(
    "  5. Severe blur / obstruction"
)

print(
    "  6. Large static graphical overlays"
)

print()

print(
    "Important:"
)

print(
    "Small natural background details are acceptable."
)

print(
    "Text overlapping or consistently appearing "
    "near the lips is the main leakage risk."
)


# ============================================================
# DATA TABLE PREVIEW
# ============================================================

print()
print("=" * 76)
print("AUDIT INDEX TABLE")
print("=" * 76)


display(
    audit_review[
        [
            "audit_id",
            "split",
            "model_class_id",
            "word"
        ]
    ]
)


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 76)
print("✅ CELL 14.2 VISUAL AUDIT GENERATED")
print("=" * 76)

print(
    f"Please visually inspect "
    f"{len(audit_review)} clips."
)

print()

print(
    "Do NOT start Cell 15 yet if subtitles/text "
    "are repeatedly visible inside the mouth ROI."
)

print()

print(
    "If the sampled ROIs are clean, "
    "we can proceed to Cell 15 Full Training."
)

In [ ]:
# ============================================================
# CELL 15 : FULL TRAINING
# VocaLip Thai — Word-Level Baseline
#
# Training:
# - Mouth-only
# - 44 Thai word classes
# - Weighted CrossEntropyLoss
# - AdamW
# - AMP
# - Gradient clipping
# - ReduceLROnPlateau
# - Early stopping
#
# Model selection:
# - BEST checkpoint by Validation Macro F1
#
# IMPORTANT:
# - Test set is NOT used during training
# - Best model is saved automatically
# ============================================================

import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch


# ============================================================
# PRE-TRAINING SAFETY CHECKS
# ============================================================

assert "model" in globals(), (
    "Main model not found. Run Cell 11 first."
)

assert "criterion" in globals(), (
    "Loss function not found. Run Cell 12 first."
)

assert "optimizer" in globals(), (
    "Optimizer not found. Run Cell 12 first."
)

assert "scheduler" in globals(), (
    "Scheduler not found. Run Cell 12 first."
)

assert "scaler" in globals(), (
    "AMP scaler not found. Run Cell 12 first."
)

assert "train_loader" in globals()
assert "val_loader" in globals()

assert "train_one_epoch" in globals()
assert "evaluate_model" in globals()

assert "SANITY_OVERFIT_PASSED" in globals(), (
    "Sanity gate variable not found. "
    "Run Cell 14.1 first."
)

assert SANITY_OVERFIT_PASSED is True, (
    "Sanity overfit gate is not open. "
    "Do not start full training."
)


# ============================================================
# TRAINING CONFIG
# ============================================================

FULL_TRAIN_EPOCHS = EPOCHS
EARLY_STOP_PATIENCE = PATIENCE

MIN_IMPROVEMENT = 1e-4

CHECKPOINT_METRIC = "macro_f1"


# ============================================================
# MODEL DIRECTORY
# ============================================================

MODEL_DIR = Path(
    MODEL_DIR
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


BEST_MODEL_PATH = Path(
    BEST_MODEL_PATH
)


LAST_MODEL_PATH = (
    MODEL_DIR
    / "01_word_level_baseline_last.pt"
)


HISTORY_CSV_PATH = (
    MODEL_DIR
    / "01_word_level_baseline_history.csv"
)


TRAINING_SUMMARY_PATH = (
    MODEL_DIR
    / "01_word_level_baseline_training_summary.json"
)


# ============================================================
# CHECK DATASET
# ============================================================

print("=" * 80)
print("FULL TRAINING PRE-CHECK")
print("=" * 80)

print(
    f"Device              : {DEVICE}"
)

if torch.cuda.is_available():

    print(
        f"GPU                 : "
        f"{torch.cuda.get_device_name(0)}"
    )


print(
    f"Train samples       : "
    f"{len(train_dataset):,}"
)

print(
    f"Validation samples  : "
    f"{len(val_dataset):,}"
)

print(
    f"Classes             : "
    f"{NUM_CLASSES}"
)

print(
    f"Batch size          : "
    f"{BATCH_SIZE}"
)

print(
    f"Max epochs          : "
    f"{FULL_TRAIN_EPOCHS}"
)

print(
    f"Early-stop patience : "
    f"{EARLY_STOP_PATIENCE}"
)

print(
    f"Checkpoint metric   : "
    f"Validation Macro F1"
)

print(
    f"Weighted CE         : "
    f"{USE_CLASS_WEIGHTED_LOSS}"
)

print(
    f"AMP                 : "
    f"{USE_AMP}"
)

print(
    f"Initial LR          : "
    f"{optimizer.param_groups[0]['lr']:.6f}"
)

print(
    f"Best checkpoint     : "
    f"{BEST_MODEL_PATH}"
)


# ============================================================
# IMPORTANT:
# Confirm main model is in training-ready state.
# ============================================================

model = model.to(
    DEVICE
)


# ============================================================
# TRAINING STATE
# ============================================================

training_history = []

best_val_f1 = -np.inf
best_val_acc = -np.inf
best_val_loss = np.inf

best_epoch = 0

epochs_without_improvement = 0

training_start_time = time.time()


# ============================================================
# HELPER: SAVE CHECKPOINT
# ============================================================

def save_training_checkpoint(
    path,
    epoch,
    model,
    optimizer,
    scheduler,
    scaler,
    train_metrics,
    val_metrics,
    best_val_f1
):

    checkpoint = {

        "epoch":
            int(epoch),

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "scaler_state_dict":
            scaler.state_dict()
            if scaler is not None
            else None,

        "train_metrics":
            train_metrics,

        "val_metrics":
            val_metrics,

        "best_val_macro_f1":
            float(best_val_f1),

        "num_classes":
            int(NUM_CLASSES),

        "target_frames":
            int(TARGET_FRAMES),

        "roi_size":
            int(ROI_SIZE),

        "feature_dim":
            int(FEATURE_DIM),

        "gru_hidden":
            int(GRU_HIDDEN),

        "gru_layers":
            int(GRU_LAYERS),

        "dropout":
            float(MODEL_DROPOUT),

        "class_weighted_loss":
            bool(
                USE_CLASS_WEIGHTED_LOSS
            ),

        "weighted_sampler":
            bool(
                USE_WEIGHTED_SAMPLER
            ),

        "learning_rate":
            float(
                optimizer.param_groups[
                    0
                ][
                    "lr"
                ]
            )
    }


    torch.save(
        checkpoint,
        path
    )


# ============================================================
# FULL TRAIN LOOP
# ============================================================

print()
print("=" * 80)
print("FULL TRAINING START")
print("=" * 80)


for epoch in range(
    1,
    FULL_TRAIN_EPOCHS + 1
):

    epoch_start_time = time.time()


    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    train_metrics = train_one_epoch(

        model=model,

        loader=train_loader,

        optimizer=optimizer,

        criterion=criterion,

        scaler=scaler,

        device=DEVICE,

        epoch=epoch,

        show_progress=True
    )


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    val_metrics = evaluate_model(

        model=model,

        loader=val_loader,

        criterion=criterion,

        device=DEVICE,

        split_name="Val",

        show_progress=True,

        return_predictions=False
    )


    # --------------------------------------------------------
    # SCHEDULER
    #
    # Monitor validation Macro F1
    # --------------------------------------------------------

    old_lr = float(
        optimizer.param_groups[
            0
        ][
            "lr"
        ]
    )


    scheduler.step(
        val_metrics[
            "macro_f1"
        ]
    )


    new_lr = float(
        optimizer.param_groups[
            0
        ][
            "lr"
        ]
    )


    # --------------------------------------------------------
    # BEST CHECKPOINT LOGIC
    # --------------------------------------------------------

    current_val_f1 = float(
        val_metrics[
            "macro_f1"
        ]
    )


    improved = (
        current_val_f1
        >
        best_val_f1
        + MIN_IMPROVEMENT
    )


    if improved:

        best_val_f1 = (
            current_val_f1
        )

        best_val_acc = float(
            val_metrics[
                "accuracy"
            ]
        )

        best_val_loss = float(
            val_metrics[
                "loss"
            ]
        )

        best_epoch = int(
            epoch
        )

        epochs_without_improvement = 0


        save_training_checkpoint(

            path=BEST_MODEL_PATH,

            epoch=epoch,

            model=model,

            optimizer=optimizer,

            scheduler=scheduler,

            scaler=scaler,

            train_metrics=train_metrics,

            val_metrics=val_metrics,

            best_val_f1=best_val_f1
        )


    else:

        epochs_without_improvement += 1


    # --------------------------------------------------------
    # SAVE LAST CHECKPOINT EVERY EPOCH
    # --------------------------------------------------------

    save_training_checkpoint(

        path=LAST_MODEL_PATH,

        epoch=epoch,

        model=model,

        optimizer=optimizer,

        scheduler=scheduler,

        scaler=scaler,

        train_metrics=train_metrics,

        val_metrics=val_metrics,

        best_val_f1=best_val_f1
    )


    # --------------------------------------------------------
    # EPOCH HISTORY
    # --------------------------------------------------------

    epoch_seconds = (
        time.time()
        - epoch_start_time
    )


    history_row = {

        "epoch":
            int(epoch),

        "train_loss":
            float(
                train_metrics[
                    "loss"
                ]
            ),

        "train_accuracy":
            float(
                train_metrics[
                    "accuracy"
                ]
            ),

        "train_macro_f1":
            float(
                train_metrics[
                    "macro_f1"
                ]
            ),

        "train_top3":
            float(
                train_metrics[
                    "top3_accuracy"
                ]
            ),

        "val_loss":
            float(
                val_metrics[
                    "loss"
                ]
            ),

        "val_accuracy":
            float(
                val_metrics[
                    "accuracy"
                ]
            ),

        "val_macro_f1":
            float(
                val_metrics[
                    "macro_f1"
                ]
            ),

        "val_top3":
            float(
                val_metrics[
                    "top3_accuracy"
                ]
            ),

        "learning_rate":
            float(
                new_lr
            ),

        "grad_norm_mean":
            float(
                train_metrics.get(
                    "grad_norm_mean",
                    np.nan
                )
            ),

        "grad_norm_max":
            float(
                train_metrics.get(
                    "grad_norm_max",
                    np.nan
                )
            ),

        "epoch_seconds":
            float(
                epoch_seconds
            ),

        "best_val_macro_f1":
            float(
                best_val_f1
            ),

        "epochs_without_improvement":
            int(
                epochs_without_improvement
            )
    }


    training_history.append(
        history_row
    )


    # --------------------------------------------------------
    # SAVE HISTORY CSV AFTER EACH EPOCH
    # --------------------------------------------------------

    pd.DataFrame(
        training_history
    ).to_csv(
        HISTORY_CSV_PATH,
        index=False,
        encoding="utf-8-sig"
    )


    # --------------------------------------------------------
    # PRINT SUMMARY
    # --------------------------------------------------------

    print()
    print("-" * 80)

    print(
        f"EPOCH {epoch:02d}/"
        f"{FULL_TRAIN_EPOCHS}"
    )

    print(
        f"Train | "
        f"loss={train_metrics['loss']:.4f} | "
        f"acc={train_metrics['accuracy']:.4f} | "
        f"F1={train_metrics['macro_f1']:.4f} | "
        f"top3={train_metrics['top3_accuracy']:.4f}"
    )

    print(
        f"Val   | "
        f"loss={val_metrics['loss']:.4f} | "
        f"acc={val_metrics['accuracy']:.4f} | "
        f"F1={val_metrics['macro_f1']:.4f} | "
        f"top3={val_metrics['top3_accuracy']:.4f}"
    )

    print(
        f"LR    | "
        f"{old_lr:.2e} -> "
        f"{new_lr:.2e}"
    )

    print(
        f"Best  | "
        f"epoch={best_epoch} | "
        f"val_F1={best_val_f1:.4f} | "
        f"val_acc={best_val_acc:.4f}"
    )

    print(
        f"No improvement     : "
        f"{epochs_without_improvement}/"
        f"{EARLY_STOP_PATIENCE}"
    )

    print(
        f"Epoch time         : "
        f"{epoch_seconds / 60:.2f} min"
    )


    if improved:

        print(
            "✅ New best checkpoint saved"
        )


    # --------------------------------------------------------
    # LR REDUCTION NOTICE
    # --------------------------------------------------------

    if new_lr < old_lr:

        print(
            f"📉 Learning rate reduced: "
            f"{old_lr:.2e} -> {new_lr:.2e}"
        )


    # --------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------

    if (
        epochs_without_improvement
        >= EARLY_STOP_PATIENCE
    ):

        print()
        print("=" * 80)
        print("EARLY STOPPING")
        print("=" * 80)

        print(
            f"No Validation Macro F1 improvement "
            f"for {EARLY_STOP_PATIENCE} epochs."
        )

        print(
            f"Best epoch    : {best_epoch}"
        )

        print(
            f"Best Val F1   : {best_val_f1:.4f}"
        )

        print(
            f"Best Val Acc  : {best_val_acc:.4f}"
        )

        break


# ============================================================
# TRAINING COMPLETE
# ============================================================

training_end_time = time.time()

total_training_seconds = (
    training_end_time
    - training_start_time
)

epochs_completed = len(
    training_history
)


# ============================================================
# FINAL HISTORY DATAFRAME
# ============================================================

training_history_df = pd.DataFrame(
    training_history
)


training_history_df.to_csv(
    HISTORY_CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# VERIFY BEST CHECKPOINT
# ============================================================

assert BEST_MODEL_PATH.exists(), (
    "Best checkpoint was not created."
)


best_checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location="cpu",
    weights_only=False
)


# ============================================================
# TRAINING SUMMARY
# ============================================================

training_summary = {

    "experiment":
        "Experiment 0 - Supervised Mouth-only Word-Level Baseline",

    "num_classes":
        int(NUM_CLASSES),

    "train_samples":
        int(
            len(
                train_dataset
            )
        ),

    "val_samples":
        int(
            len(
                val_dataset
            )
        ),

    "epochs_requested":
        int(
            FULL_TRAIN_EPOCHS
        ),

    "epochs_completed":
        int(
            epochs_completed
        ),

    "early_stop_patience":
        int(
            EARLY_STOP_PATIENCE
        ),

    "best_epoch":
        int(
            best_epoch
        ),

    "best_val_macro_f1":
        float(
            best_val_f1
        ),

    "best_val_accuracy":
        float(
            best_val_acc
        ),

    "best_val_loss":
        float(
            best_val_loss
        ),

    "final_learning_rate":
        float(
            optimizer.param_groups[
                0
            ][
                "lr"
            ]
        ),

    "total_training_seconds":
        float(
            total_training_seconds
        ),

    "total_training_minutes":
        float(
            total_training_seconds
            / 60.0
        ),

    "best_checkpoint":
        str(
            BEST_MODEL_PATH
        ),

    "last_checkpoint":
        str(
            LAST_MODEL_PATH
        ),

    "history_csv":
        str(
            HISTORY_CSV_PATH
        )
}


with open(
    TRAINING_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        training_summary,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# FINAL CONSOLE SUMMARY
# ============================================================

print()
print("=" * 80)
print("✅ FULL TRAINING COMPLETE")
print("=" * 80)

print(
    f"Epochs completed : "
    f"{epochs_completed}"
)

print(
    f"Best epoch       : "
    f"{best_epoch}"
)

print(
    f"Best Val loss    : "
    f"{best_val_loss:.6f}"
)

print(
    f"Best Val Acc     : "
    f"{best_val_acc:.4f}"
)

print(
    f"Best Val MacroF1 : "
    f"{best_val_f1:.4f}"
)

print(
    f"Training time    : "
    f"{total_training_seconds / 60:.2f} min"
)

print()

print(
    f"Best checkpoint  : "
    f"{BEST_MODEL_PATH}"
)

print(
    f"Last checkpoint  : "
    f"{LAST_MODEL_PATH}"
)

print(
    f"History CSV      : "
    f"{HISTORY_CSV_PATH}"
)

print(
    f"Summary JSON     : "
    f"{TRAINING_SUMMARY_PATH}"
)


# ============================================================
# GPU CLEANUP
# ============================================================

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ============================================================
# NEXT STEP
# ============================================================

print()
print("=" * 80)
print("NEXT")
print("=" * 80)

print(
    "Cell 16: Load BEST checkpoint "
    "and evaluate once on the TEST set."
)

print(
    "Do not use the LAST checkpoint "
    "for final test reporting."
)

print("=" * 80)

In [ ]:
# ============================================================
# CELL 16 : LOAD BEST CHECKPOINT + FINAL TEST EVALUATION
# VocaLip Thai — Word-Level Baseline
#
# IMPORTANT:
# - Load BEST validation checkpoint
# - Evaluate TEST set ONCE
# - Do not select model based on test result
#
# Outputs kept for later cells:
#   test_metrics
#   test_targets
#   test_predictions
#   test_probabilities
#   test_top3_predictions
#   test_confidences
# ============================================================

import gc
from pathlib import Path

import numpy as np
import torch


# ============================================================
# SAFETY CHECKS
# ============================================================

assert "BEST_MODEL_PATH" in globals(), (
    "BEST_MODEL_PATH not found. "
    "Run Cell 15 first."
)

assert "test_loader" in globals(), (
    "test_loader not found. "
    "Run Cell 10 first."
)

assert "evaluate_model" in globals(), (
    "evaluate_model not found. "
    "Run Cell 13 first."
)


BEST_MODEL_PATH = Path(
    BEST_MODEL_PATH
)


assert BEST_MODEL_PATH.exists(), (
    f"Best checkpoint not found:\n"
    f"{BEST_MODEL_PATH}"
)


# ============================================================
# LOAD CHECKPOINT
# ============================================================

print("=" * 80)
print("LOAD BEST BASELINE CHECKPOINT")
print("=" * 80)

print(
    f"Checkpoint : {BEST_MODEL_PATH}"
)


try:

    best_checkpoint = torch.load(
        BEST_MODEL_PATH,
        map_location=DEVICE,
        weights_only=False
    )

except TypeError:

    # Compatibility with older PyTorch
    best_checkpoint = torch.load(
        BEST_MODEL_PATH,
        map_location=DEVICE
    )


# ============================================================
# VERIFY CHECKPOINT METADATA
# ============================================================

checkpoint_epoch = int(
    best_checkpoint.get(
        "epoch",
        -1
    )
)


checkpoint_val_metrics = (
    best_checkpoint.get(
        "val_metrics",
        {}
    )
)


checkpoint_best_f1 = float(
    best_checkpoint.get(
        "best_val_macro_f1",
        np.nan
    )
)


print(
    f"Best epoch       : "
    f"{checkpoint_epoch}"
)

print(
    f"Checkpoint Val F1: "
    f"{checkpoint_best_f1:.4f}"
)


if checkpoint_val_metrics:

    print(
        f"Checkpoint Val Acc: "
        f"{checkpoint_val_metrics.get('accuracy', np.nan):.4f}"
    )

    print(
        f"Checkpoint Val Loss: "
        f"{checkpoint_val_metrics.get('loss', np.nan):.4f}"
    )


# ============================================================
# ARCHITECTURE CONSISTENCY CHECKS
# ============================================================

if "num_classes" in best_checkpoint:

    assert int(
        best_checkpoint[
            "num_classes"
        ]
    ) == NUM_CLASSES


if "target_frames" in best_checkpoint:

    assert int(
        best_checkpoint[
            "target_frames"
        ]
    ) == TARGET_FRAMES


if "roi_size" in best_checkpoint:

    assert int(
        best_checkpoint[
            "roi_size"
        ]
    ) == ROI_SIZE


# ============================================================
# LOAD BEST MODEL WEIGHTS
# ============================================================

model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ],
    strict=True
)


model = model.to(
    DEVICE
)


model.eval()


print()
print(
    "✅ Best model weights loaded."
)


# ============================================================
# TEST SET INFORMATION
# ============================================================

print()
print("=" * 80)
print("FINAL TEST SET")
print("=" * 80)

print(
    f"Samples : {len(test_dataset):,}"
)

print(
    f"Classes : "
    f"{test_df['model_class_id'].nunique()}/"
    f"{NUM_CLASSES}"
)

print()

print(
    "The test set has not been used "
    "for checkpoint selection."
)


# ============================================================
# FINAL TEST EVALUATION
# ============================================================

print()
print("=" * 80)
print("RUNNING FINAL TEST EVALUATION")
print("=" * 80)


(
    test_metrics,
    test_targets,
    test_predictions,
    test_probabilities

) = evaluate_model(

    model=model,

    loader=test_loader,

    criterion=criterion,

    device=DEVICE,

    split_name="TEST",

    show_progress=True,

    return_predictions=True
)


# ============================================================
# DERIVED TEST INFORMATION
# ============================================================

assert (
    len(test_targets)
    == len(test_dataset)
)


assert (
    test_probabilities.shape
    ==
    (
        len(test_dataset),
        NUM_CLASSES
    )
)


# ------------------------------------------------------------
# Confidence
# ------------------------------------------------------------

test_confidences = np.max(
    test_probabilities,
    axis=1
)


# ------------------------------------------------------------
# Top-3 predicted class IDs
# ------------------------------------------------------------

test_top3_predictions = np.argsort(
    test_probabilities,
    axis=1
)[:, -3:][:, ::-1]


# ============================================================
# BASIC TEST SANITY
# ============================================================

assert np.isfinite(
    test_metrics[
        "loss"
    ]
)


assert np.all(
    np.isfinite(
        test_probabilities
    )
)


assert (
    0.0
    <= test_metrics[
        "accuracy"
    ]
    <= 1.0
)


assert (
    0.0
    <= test_metrics[
        "macro_f1"
    ]
    <= 1.0
)


assert (
    test_metrics[
        "top3_accuracy"
    ]
    + 1e-12
    >= test_metrics[
        "accuracy"
    ]
)


# ============================================================
# RANDOM-CHANCE REFERENCE
# ============================================================

random_top1 = (
    1.0
    / NUM_CLASSES
)


random_top3 = (
    min(
        3,
        NUM_CLASSES
    )
    / NUM_CLASSES
)


# ============================================================
# FINAL RESULTS
# ============================================================

print()
print("=" * 80)
print("FINAL TEST RESULTS")
print("=" * 80)

print(
    f"Test Loss      : "
    f"{test_metrics['loss']:.6f}"
)

print(
    f"Test Accuracy  : "
    f"{test_metrics['accuracy']:.4f} "
    f"({test_metrics['accuracy'] * 100:.2f}%)"
)

print(
    f"Test Macro F1  : "
    f"{test_metrics['macro_f1']:.4f}"
)

print(
    f"Test Top-3 Acc : "
    f"{test_metrics['top3_accuracy']:.4f} "
    f"({test_metrics['top3_accuracy'] * 100:.2f}%)"
)


print()
print("=" * 80)
print("REFERENCE")
print("=" * 80)

print(
    f"Random Top-1   : "
    f"{random_top1:.4f} "
    f"({random_top1 * 100:.2f}%)"
)

print(
    f"Random Top-3   : "
    f"{random_top3:.4f} "
    f"({random_top3 * 100:.2f}%)"
)


# ============================================================
# VALIDATION ↔ TEST COMPARISON
# ============================================================

best_val_acc = float(
    checkpoint_val_metrics.get(
        "accuracy",
        np.nan
    )
)


best_val_f1 = float(
    checkpoint_val_metrics.get(
        "macro_f1",
        checkpoint_best_f1
    )
)


print()
print("=" * 80)
print("VALIDATION vs TEST")
print("=" * 80)

print(
    f"Best Val Accuracy : "
    f"{best_val_acc:.4f}"
)

print(
    f"Test Accuracy     : "
    f"{test_metrics['accuracy']:.4f}"
)

print(
    f"Difference        : "
    f"{test_metrics['accuracy'] - best_val_acc:+.4f}"
)


print()

print(
    f"Best Val Macro F1 : "
    f"{best_val_f1:.4f}"
)

print(
    f"Test Macro F1     : "
    f"{test_metrics['macro_f1']:.4f}"
)

print(
    f"Difference        : "
    f"{test_metrics['macro_f1'] - best_val_f1:+.4f}"
)


# ============================================================
# CONFIDENCE SUMMARY
# ============================================================

correct_mask = (
    test_predictions
    == test_targets
)


incorrect_mask = (
    ~correct_mask
)


mean_confidence = float(
    np.mean(
        test_confidences
    )
)


mean_correct_confidence = (
    float(
        np.mean(
            test_confidences[
                correct_mask
            ]
        )
    )
    if np.any(
        correct_mask
    )
    else np.nan
)


mean_incorrect_confidence = (
    float(
        np.mean(
            test_confidences[
                incorrect_mask
            ]
        )
    )
    if np.any(
        incorrect_mask
    )
    else np.nan
)


print()
print("=" * 80)
print("PREDICTION CONFIDENCE")
print("=" * 80)

print(
    f"Mean confidence           : "
    f"{mean_confidence:.4f}"
)

print(
    f"Mean confidence — correct : "
    f"{mean_correct_confidence:.4f}"
)

print(
    f"Mean confidence — wrong   : "
    f"{mean_incorrect_confidence:.4f}"
)


# ============================================================
# COUNTS
# ============================================================

num_correct = int(
    correct_mask.sum()
)


num_wrong = int(
    incorrect_mask.sum()
)


print()
print("=" * 80)
print("PREDICTION COUNTS")
print("=" * 80)

print(
    f"Correct : "
    f"{num_correct}/{len(test_targets)}"
)

print(
    f"Wrong   : "
    f"{num_wrong}/{len(test_targets)}"
)


# ============================================================
# KEEP RESULTS IN MEMORY FOR CELLS 17–20
# ============================================================

baseline_test_results = {

    "checkpoint_epoch":
        checkpoint_epoch,

    "val_accuracy":
        best_val_acc,

    "val_macro_f1":
        best_val_f1,

    "test_loss":
        float(
            test_metrics[
                "loss"
            ]
        ),

    "test_accuracy":
        float(
            test_metrics[
                "accuracy"
            ]
        ),

    "test_macro_f1":
        float(
            test_metrics[
                "macro_f1"
            ]
        ),

    "test_top3_accuracy":
        float(
            test_metrics[
                "top3_accuracy"
            ]
        ),

    "mean_confidence":
        mean_confidence,

    "mean_correct_confidence":
        mean_correct_confidence,

    "mean_incorrect_confidence":
        mean_incorrect_confidence,

    "num_correct":
        num_correct,

    "num_wrong":
        num_wrong,

    "num_test_samples":
        int(
            len(
                test_targets
            )
        )
}


# ============================================================
# GPU CLEANUP
# ============================================================

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 80)
print("✅ CELL 16 FINAL TEST EVALUATION COMPLETE")
print("=" * 80)

print(
    "Test predictions are retained in memory "
    "for detailed error analysis."
)

print()

print(
    "Next:"
)

print(
    "Cell 17 — Per-Class Classification Report"
)

print(
    "Cell 18 — Confusion Matrix"
)

print(
    "Cell 19 — Most Confused Word Pairs"
)

print(
    "Cell 20 — Prediction Inspection"
)

print("=" * 80)

In [ ]:
# ============================================================
# CELL 17 : PER-CLASS CLASSIFICATION REPORT
# VocaLip Thai — Word-Level Baseline
#
# Purpose:
# - Inspect performance for each Thai word
# - Precision / Recall / F1 / Support
# - Sort classes by F1
# - Identify strongest and weakest words
#
# Uses outputs from Cell 16:
#   test_targets
#   test_predictions
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    classification_report,
    precision_recall_fscore_support
)


# ============================================================
# SAFETY CHECKS
# ============================================================

assert "test_targets" in globals()
assert "test_predictions" in globals()
assert "test_df" in globals()

assert len(test_targets) == len(test_predictions)


# ============================================================
# BUILD CLASS ID -> WORD MAP
# ============================================================

class_word_table = (
    df[
        [
            "model_class_id",
            "word"
        ]
    ]
    .drop_duplicates()
    .copy()
)


class_word_table[
    "model_class_id"
] = class_word_table[
    "model_class_id"
].astype(int)


class_word_table = (
    class_word_table
    .sort_values(
        "model_class_id"
    )
    .reset_index(
        drop=True
    )
)


assert len(
    class_word_table
) == NUM_CLASSES


id_to_word = dict(
    zip(
        class_word_table[
            "model_class_id"
        ],
        class_word_table[
            "word"
        ]
    )
)


# ============================================================
# PER-CLASS METRICS
# ============================================================

labels = list(
    range(
        NUM_CLASSES
    )
)


precision, recall, f1, support = (
    precision_recall_fscore_support(

        test_targets,

        test_predictions,

        labels=labels,

        zero_division=0
    )
)


report_df = pd.DataFrame({

    "class_id":
        labels,

    "word":
        [
            id_to_word[
                class_id
            ]
            for class_id in labels
        ],

    "precision":
        precision,

    "recall":
        recall,

    "f1":
        f1,

    "support":
        support
})


# ============================================================
# CORRECT PREDICTION COUNT
# ============================================================

correct_counts = []


for class_id in labels:

    mask = (
        test_targets
        == class_id
    )


    class_correct = int(
        np.sum(
            test_predictions[
                mask
            ]
            == class_id
        )
    )


    correct_counts.append(
        class_correct
    )


report_df[
    "correct"
] = correct_counts


report_df[
    "accuracy_within_class"
] = (
    report_df[
        "correct"
    ]
    / report_df[
        "support"
    ].replace(
        0,
        np.nan
    )
)


# ============================================================
# SORTED VIEWS
# ============================================================

report_by_f1 = (
    report_df
    .sort_values(
        [
            "f1",
            "support"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


report_by_recall = (
    report_df
    .sort_values(
        [
            "recall",
            "support"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# PRINT OVERALL
# ============================================================

print("=" * 80)
print("PER-CLASS CLASSIFICATION REPORT")
print("=" * 80)

print(
    f"Test samples : "
    f"{len(test_targets)}"
)

print(
    f"Classes      : "
    f"{NUM_CLASSES}"
)

print(
    f"Test Acc     : "
    f"{test_metrics['accuracy']:.4f}"
)

print(
    f"Test MacroF1 : "
    f"{test_metrics['macro_f1']:.4f}"
)


# ============================================================
# FULL TABLE
# ============================================================

print()
print("=" * 80)
print("ALL CLASSES — SORTED BY CLASS ID")
print("=" * 80)


display(
    report_df.round(
        {
            "precision": 4,
            "recall": 4,
            "f1": 4,
            "accuracy_within_class": 4
        }
    )
)


# ============================================================
# BEST CLASSES
# ============================================================

TOP_N = 10


print()
print("=" * 80)
print(f"TOP {TOP_N} CLASSES BY F1")
print("=" * 80)


display(
    report_by_f1.head(
        TOP_N
    ).round(
        {
            "precision": 4,
            "recall": 4,
            "f1": 4,
            "accuracy_within_class": 4
        }
    )
)


# ============================================================
# WEAKEST CLASSES
# ============================================================

print()
print("=" * 80)
print(f"BOTTOM {TOP_N} CLASSES BY F1")
print("=" * 80)


display(
    report_by_f1.tail(
        TOP_N
    ).sort_values(
        "f1"
    ).round(
        {
            "precision": 4,
            "recall": 4,
            "f1": 4,
            "accuracy_within_class": 4
        }
    )
)


# ============================================================
# ZERO-RECALL / ZERO-F1 CLASSES
# ============================================================

zero_recall_df = report_df[
    report_df[
        "recall"
    ] == 0
].copy()


zero_f1_df = report_df[
    report_df[
        "f1"
    ] == 0
].copy()


print()
print("=" * 80)
print("FAILED CLASSES")
print("=" * 80)

print(
    f"Zero Recall classes : "
    f"{len(zero_recall_df)}"
)

print(
    f"Zero F1 classes     : "
    f"{len(zero_f1_df)}"
)


if len(
    zero_recall_df
) > 0:

    print()
    print(
        "Zero Recall:"
    )

    display(
        zero_recall_df[
            [
                "class_id",
                "word",
                "support",
                "precision",
                "recall",
                "f1"
            ]
        ]
    )


# ============================================================
# SUPPORT ANALYSIS
# ============================================================

print()
print("=" * 80)
print("TEST SUPPORT SUMMARY")
print("=" * 80)

print(
    f"Min support  : "
    f"{int(report_df['support'].min())}"
)

print(
    f"Max support  : "
    f"{int(report_df['support'].max())}"
)

print(
    f"Mean support : "
    f"{report_df['support'].mean():.2f}"
)

print(
    f"Median       : "
    f"{report_df['support'].median():.2f}"
)


# ============================================================
# SUPPORT vs F1 CORRELATION
#
# Interpret cautiously because test supports are small.
# ============================================================

if (
    report_df[
        "support"
    ].nunique()
    > 1
):

    support_f1_corr = float(

        report_df[
            [
                "support",
                "f1"
            ]
        ]
        .corr(
            method="spearman"
        )
        .iloc[
            0,
            1
        ]
    )

else:

    support_f1_corr = np.nan


print(
    f"Spearman support↔F1 : "
    f"{support_f1_corr:.4f}"
)


# ============================================================
# SAVE FOR LATER CELLS
# ============================================================

class_report_df = report_df.copy()

class_report_by_f1 = (
    report_by_f1.copy()
)


# ============================================================
# OPTIONAL CSV EXPORT
# ============================================================

CLASS_REPORT_PATH = (
    MODEL_DIR
    / "01_word_level_baseline_test_class_report.csv"
)


class_report_df.to_csv(
    CLASS_REPORT_PATH,
    index=False,
    encoding="utf-8-sig"
)


print()
print("=" * 80)
print("✅ CELL 17 COMPLETE")
print("=" * 80)

print(
    f"Saved report : "
    f"{CLASS_REPORT_PATH}"
)

print()

print(
    "Next:"
)

print(
    "Cell 18 — Confusion Matrix"
)

print("=" * 80)

In [ ]:
# ============================================================
# CELL 18 : CONFUSION MATRIX
# VocaLip Thai — Word-Level Baseline
#
# Purpose:
# - Visualize prediction errors across all 44 Thai words
# - Raw confusion matrix
# - Row-normalized confusion matrix
# - Identify prediction bias
#
# Uses:
#   test_targets
#   test_predictions
#   id_to_word
#
# IMPORTANT:
# Rows    = TRUE word
# Columns = PREDICTED word
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix


# ============================================================
# SAFETY CHECKS
# ============================================================

assert "test_targets" in globals(), (
    "test_targets not found. Run Cell 16 first."
)

assert "test_predictions" in globals(), (
    "test_predictions not found. Run Cell 16 first."
)

assert "id_to_word" in globals(), (
    "id_to_word not found. Run Cell 17 first."
)

assert len(test_targets) == len(test_predictions)


# ============================================================
# LABELS
# ============================================================

class_ids = list(
    range(NUM_CLASSES)
)

class_words = [
    id_to_word[class_id]
    for class_id in class_ids
]


# ============================================================
# RAW CONFUSION MATRIX
# ============================================================

cm_raw = confusion_matrix(
    test_targets,
    test_predictions,
    labels=class_ids
)


assert cm_raw.shape == (
    NUM_CLASSES,
    NUM_CLASSES
)


assert int(
    cm_raw.sum()
) == len(test_targets)


# ============================================================
# ROW-NORMALIZED CONFUSION MATRIX
#
# Each row sums to 1.
# This answers:
# "For this TRUE word, what did the model predict?"
# ============================================================

row_sum = cm_raw.sum(
    axis=1,
    keepdims=True
)


cm_normalized = np.divide(
    cm_raw,
    row_sum,
    out=np.zeros_like(
        cm_raw,
        dtype=np.float64
    ),
    where=row_sum != 0
)


# ============================================================
# DATAFRAMES
# ============================================================

cm_raw_df = pd.DataFrame(
    cm_raw,
    index=class_words,
    columns=class_words
)


cm_normalized_df = pd.DataFrame(
    cm_normalized,
    index=class_words,
    columns=class_words
)


# ============================================================
# BASIC SUMMARY
# ============================================================

diagonal_correct = int(
    np.trace(cm_raw)
)


total_samples = int(
    cm_raw.sum()
)


matrix_accuracy = (
    diagonal_correct
    / total_samples
)


print("=" * 80)
print("CONFUSION MATRIX SUMMARY")
print("=" * 80)

print(
    f"Classes          : {NUM_CLASSES}"
)

print(
    f"Test samples     : {total_samples}"
)

print(
    f"Diagonal correct : {diagonal_correct}"
)

print(
    f"Matrix accuracy  : "
    f"{matrix_accuracy:.4f}"
)

print(
    f"Cell 16 accuracy : "
    f"{test_metrics['accuracy']:.4f}"
)


assert np.isclose(
    matrix_accuracy,
    test_metrics["accuracy"]
)


# ============================================================
# FIGURE 1 — RAW CONFUSION MATRIX
# ============================================================

fig, ax = plt.subplots(
    figsize=(18, 16)
)


im = ax.imshow(
    cm_raw,
    interpolation="nearest",
    aspect="auto"
)


ax.set_title(
    "VocaLip Thai — Raw Confusion Matrix\n"
    "Mouth-only Word-Level Baseline",
    fontsize=16,
    pad=18
)


ax.set_xlabel(
    "Predicted Word",
    fontsize=12
)


ax.set_ylabel(
    "True Word",
    fontsize=12
)


ax.set_xticks(
    np.arange(NUM_CLASSES)
)

ax.set_yticks(
    np.arange(NUM_CLASSES)
)


ax.set_xticklabels(
    class_words,
    rotation=90,
    fontsize=8
)


ax.set_yticklabels(
    class_words,
    fontsize=8
)


fig.colorbar(
    im,
    ax=ax,
    fraction=0.046,
    pad=0.04,
    label="Number of samples"
)


plt.tight_layout()
plt.show()


# ============================================================
# FIGURE 2 — NORMALIZED CONFUSION MATRIX
#
# More useful for imbalanced classes.
# ============================================================

fig, ax = plt.subplots(
    figsize=(18, 16)
)


im = ax.imshow(
    cm_normalized,
    interpolation="nearest",
    aspect="auto",
    vmin=0.0,
    vmax=1.0
)


ax.set_title(
    "VocaLip Thai — Row-Normalized Confusion Matrix\n"
    "Rows = True Word | Columns = Predicted Word",
    fontsize=16,
    pad=18
)


ax.set_xlabel(
    "Predicted Word",
    fontsize=12
)


ax.set_ylabel(
    "True Word",
    fontsize=12
)


ax.set_xticks(
    np.arange(NUM_CLASSES)
)

ax.set_yticks(
    np.arange(NUM_CLASSES)
)


ax.set_xticklabels(
    class_words,
    rotation=90,
    fontsize=8
)


ax.set_yticklabels(
    class_words,
    fontsize=8
)


fig.colorbar(
    im,
    ax=ax,
    fraction=0.046,
    pad=0.04,
    label="Fraction of true class"
)


plt.tight_layout()
plt.show()


# ============================================================
# PREDICTION FREQUENCY
#
# Detect classes the model predicts too frequently.
# ============================================================

true_counts = np.bincount(
    test_targets,
    minlength=NUM_CLASSES
)


pred_counts = np.bincount(
    test_predictions,
    minlength=NUM_CLASSES
)


correct_counts = np.diag(
    cm_raw
)


prediction_frequency_df = pd.DataFrame({

    "class_id":
        class_ids,

    "word":
        class_words,

    "true_count":
        true_counts,

    "predicted_count":
        pred_counts,

    "correct":
        correct_counts
})


prediction_frequency_df[
    "prediction_minus_true"
] = (
    prediction_frequency_df[
        "predicted_count"
    ]
    -
    prediction_frequency_df[
        "true_count"
    ]
)


# ============================================================
# OVER-PREDICTED CLASSES
# ============================================================

overpredicted_df = (
    prediction_frequency_df
    .sort_values(
        [
            "prediction_minus_true",
            "predicted_count"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


print()
print("=" * 80)
print("MOST OVER-PREDICTED WORDS")
print("=" * 80)


display(
    overpredicted_df.head(
        10
    )
)


# ============================================================
# UNDER-PREDICTED CLASSES
# ============================================================

underpredicted_df = (
    prediction_frequency_df
    .sort_values(
        [
            "prediction_minus_true",
            "predicted_count"
        ],
        ascending=[
            True,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


print()
print("=" * 80)
print("MOST UNDER-PREDICTED WORDS")
print("=" * 80)


display(
    underpredicted_df.head(
        10
    )
)


# ============================================================
# NEVER-PREDICTED CLASSES
# ============================================================

never_predicted_df = (
    prediction_frequency_df[
        prediction_frequency_df[
            "predicted_count"
        ] == 0
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print()
print("=" * 80)
print("NEVER-PREDICTED CLASSES")
print("=" * 80)

print(
    f"Count : {len(never_predicted_df)}"
)


if len(never_predicted_df) > 0:

    display(
        never_predicted_df
    )

else:

    print(
        "Every class was predicted at least once."
    )


# ============================================================
# MOST COMMON WRONG PREDICTION FOR EACH TRUE CLASS
# ============================================================

class_error_rows = []


for true_id in class_ids:

    row = cm_raw[
        true_id
    ].copy()


    # Remove correct diagonal prediction
    row[
        true_id
    ] = 0


    wrong_total = int(
        row.sum()
    )


    if wrong_total > 0:

        top_wrong_id = int(
            np.argmax(
                row
            )
        )


        top_wrong_count = int(
            row[
                top_wrong_id
            ]
        )


        top_wrong_rate = (
            top_wrong_count
            / int(
                cm_raw[
                    true_id
                ].sum()
            )
        )


        class_error_rows.append({

            "true_class_id":
                true_id,

            "true_word":
                id_to_word[
                    true_id
                ],

            "top_wrong_class_id":
                top_wrong_id,

            "top_wrong_word":
                id_to_word[
                    top_wrong_id
                ],

            "wrong_count":
                top_wrong_count,

            "true_support":
                int(
                    cm_raw[
                        true_id
                    ].sum()
                ),

            "wrong_rate_of_true_class":
                float(
                    top_wrong_rate
                )
        })


class_error_df = pd.DataFrame(
    class_error_rows
)


class_error_df = (
    class_error_df
    .sort_values(
        [
            "wrong_count",
            "wrong_rate_of_true_class"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


print()
print("=" * 80)
print("TOP WRONG DESTINATION FOR EACH TRUE WORD")
print("=" * 80)


display(
    class_error_df.head(
        20
    ).round(
        {
            "wrong_rate_of_true_class":
                4
        }
    )
)


# ============================================================
# EXPORT
# ============================================================

CM_RAW_PATH = (
    MODEL_DIR
    / "01_word_level_baseline_confusion_matrix_raw.csv"
)


CM_NORMALIZED_PATH = (
    MODEL_DIR
    / "01_word_level_baseline_confusion_matrix_normalized.csv"
)


PRED_FREQ_PATH = (
    MODEL_DIR
    / "01_word_level_baseline_prediction_frequency.csv"
)


CLASS_ERROR_PATH = (
    MODEL_DIR
    / "01_word_level_baseline_class_error_destinations.csv"
)


cm_raw_df.to_csv(
    CM_RAW_PATH,
    encoding="utf-8-sig"
)


cm_normalized_df.to_csv(
    CM_NORMALIZED_PATH,
    encoding="utf-8-sig"
)


prediction_frequency_df.to_csv(
    PRED_FREQ_PATH,
    index=False,
    encoding="utf-8-sig"
)


class_error_df.to_csv(
    CLASS_ERROR_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# KEEP VARIABLES FOR CELL 19
# ============================================================

baseline_cm_raw = (
    cm_raw.copy()
)


baseline_cm_normalized = (
    cm_normalized.copy()
)


baseline_prediction_frequency = (
    prediction_frequency_df.copy()
)


baseline_class_error_df = (
    class_error_df.copy()
)


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 80)
print("✅ CELL 18 CONFUSION MATRIX COMPLETE")
print("=" * 80)

print(
    f"Raw CM        : {CM_RAW_PATH}"
)

print(
    f"Normalized CM : {CM_NORMALIZED_PATH}"
)

print(
    f"Prediction freq: {PRED_FREQ_PATH}"
)

print()

print(
    "Next:"
)

print(
    "Cell 19 — Most Confused Word Pairs"
)

print("=" * 80)

In [ ]:
# ============================================================
# CELL 19 : MOST CONFUSED WORD PAIRS
# VocaLip Thai — Word-Level Baseline
#
# Purpose:
# 1. Find strongest directional confusions:
#       TRUE A -> PREDICTED B
#
# 2. Find strongest bidirectional confusion pairs:
#       A <-> B
#
# 3. Separate:
#       - raw confusion count
#       - confusion rate
#       - symmetric pair confusion
#
# Uses Cell 18:
#   baseline_cm_raw
#   baseline_cm_normalized
#   id_to_word
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# SAFETY CHECKS
# ============================================================

assert "baseline_cm_raw" in globals(), (
    "baseline_cm_raw not found. "
    "Run Cell 18 first."
)

assert "baseline_cm_normalized" in globals(), (
    "baseline_cm_normalized not found. "
    "Run Cell 18 first."
)

assert "id_to_word" in globals(), (
    "id_to_word not found. "
    "Run Cell 17 first."
)


cm = np.asarray(
    baseline_cm_raw
)


cm_norm = np.asarray(
    baseline_cm_normalized
)


assert cm.shape == (
    NUM_CLASSES,
    NUM_CLASSES
)


# ============================================================
# CLASS SUPPORT
# ============================================================

class_support = cm.sum(
    axis=1
)


# ============================================================
# PART A
# DIRECTIONAL CONFUSION
#
# TRUE A -> PREDICTED B
# ============================================================

directional_rows = []


for true_id in range(
    NUM_CLASSES
):

    support = int(
        class_support[
            true_id
        ]
    )


    if support == 0:
        continue


    for pred_id in range(
        NUM_CLASSES
    ):

        # Ignore correct predictions
        if true_id == pred_id:
            continue


        count = int(
            cm[
                true_id,
                pred_id
            ]
        )


        if count <= 0:
            continue


        rate = (
            count
            / support
        )


        directional_rows.append({

            "true_class_id":
                int(true_id),

            "true_word":
                id_to_word[
                    true_id
                ],

            "pred_class_id":
                int(pred_id),

            "pred_word":
                id_to_word[
                    pred_id
                ],

            "confusion_count":
                count,

            "true_support":
                support,

            "confusion_rate":
                float(rate)
        })


directional_confusions_df = pd.DataFrame(
    directional_rows
)


directional_confusions_df = (
    directional_confusions_df
    .sort_values(
        [
            "confusion_count",
            "confusion_rate",
            "true_support"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# DISPLAY TOP DIRECTIONAL CONFUSIONS
# ============================================================

TOP_DIRECTIONAL = 25


print("=" * 84)
print("MOST CONFUSED WORD PAIRS")
print("=" * 84)

print()
print("=" * 84)
print("A. TOP DIRECTIONAL CONFUSIONS")
print("TRUE WORD -> PREDICTED WORD")
print("=" * 84)


display(
    directional_confusions_df.head(
        TOP_DIRECTIONAL
    ).round(
        {
            "confusion_rate":
                4
        }
    )
)


# ============================================================
# HIGH-RATE DIRECTIONAL CONFUSIONS
#
# Raw counts alone favor classes with larger support.
# This view emphasizes the proportion of a true class that
# was sent to another class.
#
# Require at least 2 confusion samples to reduce one-off noise.
# ============================================================

MIN_CONFUSION_COUNT_FOR_RATE = 2


high_rate_confusions_df = (
    directional_confusions_df[
        directional_confusions_df[
            "confusion_count"
        ]
        >= MIN_CONFUSION_COUNT_FOR_RATE
    ]
    .sort_values(
        [
            "confusion_rate",
            "confusion_count"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


print()
print("=" * 84)
print("B. HIGHEST DIRECTIONAL CONFUSION RATES")
print(
    f"Minimum confusion count = "
    f"{MIN_CONFUSION_COUNT_FOR_RATE}"
)
print("=" * 84)


display(
    high_rate_confusions_df.head(
        20
    ).round(
        {
            "confusion_rate":
                4
        }
    )
)


# ============================================================
# PART C
# BIDIRECTIONAL / SYMMETRIC CONFUSION
#
# For every unordered pair A,B:
#
#   A -> B
#   B -> A
#
# pair_count = both directions combined
#
# This is especially useful for visual speech because
# two words may have similar mouth-motion patterns.
# ============================================================

pair_rows = []


for class_a in range(
    NUM_CLASSES
):

    for class_b in range(
        class_a + 1,
        NUM_CLASSES
    ):

        a_to_b = int(
            cm[
                class_a,
                class_b
            ]
        )


        b_to_a = int(
            cm[
                class_b,
                class_a
            ]
        )


        pair_count = (
            a_to_b
            + b_to_a
        )


        if pair_count <= 0:
            continue


        support_a = int(
            class_support[
                class_a
            ]
        )


        support_b = int(
            class_support[
                class_b
            ]
        )


        rate_a_to_b = (
            a_to_b / support_a
            if support_a > 0
            else 0.0
        )


        rate_b_to_a = (
            b_to_a / support_b
            if support_b > 0
            else 0.0
        )


        # ----------------------------------------------------
        # Pair confusion rate:
        # errors between these two classes divided by the
        # total number of test samples belonging to A or B.
        # ----------------------------------------------------

        combined_support = (
            support_a
            + support_b
        )


        pair_confusion_rate = (
            pair_count
            / combined_support
            if combined_support > 0
            else 0.0
        )


        # ----------------------------------------------------
        # Bidirectional strength
        #
        # min(A->B, B->A)
        # > 0 means confusion exists in BOTH directions.
        # ----------------------------------------------------

        bidirectional_min = min(
            a_to_b,
            b_to_a
        )


        pair_rows.append({

            "class_a":
                int(class_a),

            "word_a":
                id_to_word[
                    class_a
                ],

            "class_b":
                int(class_b),

            "word_b":
                id_to_word[
                    class_b
                ],

            "A_to_B":
                a_to_b,

            "B_to_A":
                b_to_a,

            "pair_confusion_count":
                pair_count,

            "support_A":
                support_a,

            "support_B":
                support_b,

            "A_to_B_rate":
                float(
                    rate_a_to_b
                ),

            "B_to_A_rate":
                float(
                    rate_b_to_a
                ),

            "pair_confusion_rate":
                float(
                    pair_confusion_rate
                ),

            "bidirectional_min":
                int(
                    bidirectional_min
                )
        })


pair_confusions_df = pd.DataFrame(
    pair_rows
)


# ============================================================
# TOP PAIRS BY TOTAL ERROR COUNT
# ============================================================

pair_by_count_df = (
    pair_confusions_df
    .sort_values(
        [
            "pair_confusion_count",
            "bidirectional_min",
            "pair_confusion_rate"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


print()
print("=" * 84)
print("C. TOP WORD PAIRS BY TOTAL CONFUSION")
print("A -> B plus B -> A")
print("=" * 84)


display(
    pair_by_count_df.head(
        20
    ).round(
        {
            "A_to_B_rate":
                4,

            "B_to_A_rate":
                4,

            "pair_confusion_rate":
                4
        }
    )
)


# ============================================================
# PART D
# TRUE BIDIRECTIONAL CONFUSION ONLY
#
# Keep pairs where:
# A was predicted as B at least once
# AND
# B was predicted as A at least once
# ============================================================

bidirectional_df = (
    pair_confusions_df[
        (
            pair_confusions_df[
                "A_to_B"
            ] > 0
        )
        &
        (
            pair_confusions_df[
                "B_to_A"
            ] > 0
        )
    ]
    .copy()
)


bidirectional_df = (
    bidirectional_df
    .sort_values(
        [
            "pair_confusion_count",
            "bidirectional_min",
            "pair_confusion_rate"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


print()
print("=" * 84)
print("D. TRUE BIDIRECTIONAL CONFUSION PAIRS")
print("Both A -> B and B -> A occurred")
print("=" * 84)

print(
    f"Number of bidirectional pairs : "
    f"{len(bidirectional_df)}"
)


if len(
    bidirectional_df
) > 0:

    display(
        bidirectional_df.head(
            20
        ).round(
            {
                "A_to_B_rate":
                    4,

                "B_to_A_rate":
                    4,

                "pair_confusion_rate":
                    4
            }
        )
    )

else:

    print(
        "No bidirectional confusion pairs found."
    )


# ============================================================
# PART E
# CONFUSION TARGET / ATTRACTOR ANALYSIS
#
# How many WRONG samples from other classes were predicted
# as each word?
#
# This helps detect prediction attractors such as "นั่ง".
# ============================================================

wrong_prediction_inflow = []


for pred_id in range(
    NUM_CLASSES
):

    total_predicted = int(
        cm[
            :,
            pred_id
        ].sum()
    )


    correct_predicted = int(
        cm[
            pred_id,
            pred_id
        ]
    )


    wrong_inflow = (
        total_predicted
        - correct_predicted
    )


    true_support = int(
        class_support[
            pred_id
        ]
    )


    wrong_prediction_inflow.append({

        "class_id":
            pred_id,

        "word":
            id_to_word[
                pred_id
            ],

        "true_support":
            true_support,

        "total_predicted":
            total_predicted,

        "correct_predictions":
            correct_predicted,

        "wrong_inflow":
            wrong_inflow,

        "wrong_fraction_of_predictions":
            (
                wrong_inflow
                / total_predicted
                if total_predicted > 0
                else 0.0
            )
    })


prediction_attractor_df = pd.DataFrame(
    wrong_prediction_inflow
)


prediction_attractor_df = (
    prediction_attractor_df
    .sort_values(
        [
            "wrong_inflow",
            "wrong_fraction_of_predictions"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


print()
print("=" * 84)
print("E. STRONGEST WRONG-PREDICTION ATTRACTORS")
print("=" * 84)


display(
    prediction_attractor_df.head(
        15
    ).round(
        {
            "wrong_fraction_of_predictions":
                4
        }
    )
)


# ============================================================
# PART F
# COMPACT RESEARCH SUMMARY
# ============================================================

print()
print("=" * 84)
print("COMPACT CONFUSION SUMMARY")
print("=" * 84)


if len(
    directional_confusions_df
) > 0:

    top_direction = (
        directional_confusions_df.iloc[
            0
        ]
    )


    print(
        "Strongest directional confusion:"
    )

    print(
        f"  {top_direction['true_word']} "
        f"-> {top_direction['pred_word']} | "
        f"{int(top_direction['confusion_count'])} samples | "
        f"{top_direction['confusion_rate']:.2%} "
        f"of true '{top_direction['true_word']}'"
    )


if len(
    pair_by_count_df
) > 0:

    top_pair = (
        pair_by_count_df.iloc[
            0
        ]
    )


    print()

    print(
        "Strongest pair confusion:"
    )

    print(
        f"  {top_pair['word_a']} "
        f"<-> {top_pair['word_b']} | "
        f"{int(top_pair['pair_confusion_count'])} "
        f"combined errors"
    )


if len(
    prediction_attractor_df
) > 0:

    top_attractor = (
        prediction_attractor_df.iloc[
            0
        ]
    )


    print()

    print(
        "Strongest prediction attractor:"
    )

    print(
        f"  {top_attractor['word']} | "
        f"{int(top_attractor['wrong_inflow'])} "
        f"incorrect samples predicted as this word"
    )


# ============================================================
# EXPORT CSV
# ============================================================

DIRECTIONAL_CONFUSION_PATH = (
    MODEL_DIR
    / "01_word_level_baseline_directional_confusions.csv"
)


PAIR_CONFUSION_PATH = (
    MODEL_DIR
    / "01_word_level_baseline_pair_confusions.csv"
)


BIDIRECTIONAL_CONFUSION_PATH = (
    MODEL_DIR
    / "01_word_level_baseline_bidirectional_confusions.csv"
)


PREDICTION_ATTRACTOR_PATH = (
    MODEL_DIR
    / "01_word_level_baseline_prediction_attractors.csv"
)


directional_confusions_df.to_csv(
    DIRECTIONAL_CONFUSION_PATH,
    index=False,
    encoding="utf-8-sig"
)


pair_by_count_df.to_csv(
    PAIR_CONFUSION_PATH,
    index=False,
    encoding="utf-8-sig"
)


bidirectional_df.to_csv(
    BIDIRECTIONAL_CONFUSION_PATH,
    index=False,
    encoding="utf-8-sig"
)


prediction_attractor_df.to_csv(
    PREDICTION_ATTRACTOR_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# KEEP FOR CELL 20
# ============================================================

baseline_directional_confusions = (
    directional_confusions_df.copy()
)


baseline_pair_confusions = (
    pair_by_count_df.copy()
)


baseline_bidirectional_confusions = (
    bidirectional_df.copy()
)


baseline_prediction_attractors = (
    prediction_attractor_df.copy()
)


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 84)
print("✅ CELL 19 CONFUSION-PAIR ANALYSIS COMPLETE")
print("=" * 84)

print(
    f"Directional : "
    f"{DIRECTIONAL_CONFUSION_PATH}"
)

print(
    f"Pairs       : "
    f"{PAIR_CONFUSION_PATH}"
)

print(
    f"Bidirectional: "
    f"{BIDIRECTIONAL_CONFUSION_PATH}"
)

print(
    f"Attractors  : "
    f"{PREDICTION_ATTRACTOR_PATH}"
)

print()

print(
    "Next:"
)

print(
    "Cell 20 — Prediction Inspection"
)

print("=" * 84)

In [ ]:
# ============================================================
# CELL 20 : PREDICTION INSPECTION
# VocaLip Thai — Word-Level Baseline
#
# Purpose:
# - Inspect REAL mouth ROI predictions
# - Compare correct vs incorrect samples
# - Show Top-3 predictions + confidence
# - Inspect important confusion pairs
#
# Groups:
#   A. Correct — High Confidence
#   B. Wrong   — High Confidence
#   C. Wrong   — Low Confidence
#   D. Important Confusion Cases
#
# IMPORTANT:
# This cell is diagnostic only.
# It does NOT modify the dataset or model.
# ============================================================

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch


# ============================================================
# SAFETY CHECKS
# ============================================================

assert "test_targets" in globals()
assert "test_predictions" in globals()
assert "test_probabilities" in globals()
assert "test_confidences" in globals()
assert "test_dataset" in globals()
assert "test_df" in globals()
assert "id_to_word" in globals()

assert len(test_targets) == len(test_dataset)
assert len(test_predictions) == len(test_dataset)
assert len(test_confidences) == len(test_dataset)

assert test_probabilities.shape == (
    len(test_dataset),
    NUM_CLASSES
)


# ============================================================
# HELPER — GET TOP-K
# ============================================================

def get_topk_predictions(
    probability_vector,
    k=3
):

    top_ids = np.argsort(
        probability_vector
    )[-k:][::-1]

    results = []

    for class_id in top_ids:

        results.append(
            {
                "class_id": int(class_id),
                "word": id_to_word[int(class_id)],
                "probability": float(
                    probability_vector[class_id]
                )
            }
        )

    return results


# ============================================================
# BUILD SAMPLE-LEVEL PREDICTION TABLE
# ============================================================

inspection_rows = []


for idx in range(
    len(test_dataset)
):

    true_id = int(
        test_targets[idx]
    )

    pred_id = int(
        test_predictions[idx]
    )

    confidence = float(
        test_confidences[idx]
    )

    top3 = get_topk_predictions(
        test_probabilities[idx],
        k=3
    )

    true_rank = int(
        np.where(
            np.argsort(
                test_probabilities[idx]
            )[::-1] == true_id
        )[0][0]
        + 1
    )

    row = {

        "dataset_index":
            idx,

        "true_class_id":
            true_id,

        "true_word":
            id_to_word[
                true_id
            ],

        "pred_class_id":
            pred_id,

        "pred_word":
            id_to_word[
                pred_id
            ],

        "correct":
            bool(
                true_id == pred_id
            ),

        "confidence":
            confidence,

        "true_rank":
            true_rank,

        "true_in_top3":
            bool(
                true_rank <= 3
            ),

        "top1_word":
            top3[0]["word"],

        "top1_prob":
            top3[0]["probability"],

        "top2_word":
            top3[1]["word"],

        "top2_prob":
            top3[1]["probability"],

        "top3_word":
            top3[2]["word"],

        "top3_prob":
            top3[2]["probability"]
    }


    # --------------------------------------------------------
    # Add useful manifest metadata when available
    # --------------------------------------------------------

    if idx < len(test_df):

        manifest_row = test_df.iloc[idx]

        for col in [
            "video_path",
            "mouth_cache_path",
            "speaker_id",
            "segment_id",
            "word_start",
            "word_end",
            "clip_start",
            "clip_end"
        ]:

            if col in test_df.columns:

                row[col] = (
                    manifest_row[col]
                )


    inspection_rows.append(
        row
    )


prediction_inspection_df = pd.DataFrame(
    inspection_rows
)


# ============================================================
# VERIFY ORDER ALIGNMENT
#
# test_loader must be deterministic / shuffle=False.
# ============================================================

assert (
    prediction_inspection_df[
        "true_class_id"
    ].to_numpy()
    ==
    np.asarray(
        test_targets
    )
).all()


# ============================================================
# SUMMARY
# ============================================================

print("=" * 88)
print("PREDICTION INSPECTION SUMMARY")
print("=" * 88)

print(
    f"Test samples          : "
    f"{len(prediction_inspection_df)}"
)

print(
    f"Correct               : "
    f"{prediction_inspection_df['correct'].sum()}"
)

print(
    f"Wrong                 : "
    f"{(~prediction_inspection_df['correct']).sum()}"
)

print(
    f"True word in Top-3    : "
    f"{prediction_inspection_df['true_in_top3'].sum()}"
)

print(
    f"Top-3 rate            : "
    f"{prediction_inspection_df['true_in_top3'].mean():.4f}"
)


# ============================================================
# WRONG BUT TRUE WORD IS STILL IN TOP-3
#
# Important for later constrained decoding / rescoring.
# ============================================================

wrong_but_top3 = prediction_inspection_df[
    (~prediction_inspection_df["correct"])
    &
    (prediction_inspection_df["true_in_top3"])
].copy()


print(
    f"Wrong Top-1 but true in Top-3 : "
    f"{len(wrong_but_top3)}"
)

print(
    f"Fraction of all wrong samples : "
    f"{len(wrong_but_top3) / max(1, (~prediction_inspection_df['correct']).sum()):.4f}"
)


# ============================================================
# HELPER — LOAD ORIGINAL CACHED ROI
#
# Dataset returns normalized tensors.
# For visualization, read the raw uint8 cache directly.
# ============================================================

def load_raw_cached_frames(
    dataset_index
):

    row = test_df.iloc[
        dataset_index
    ]

    cache_path = row[
        "mouth_cache_path"
    ]

    with np.load(
        cache_path
    ) as data:

        frames = data[
            "frames"
        ].copy()

    assert frames.ndim == 4

    return frames


# ============================================================
# HELPER — VISUALIZE ONE SAMPLE
# ============================================================

def visualize_prediction(
    dataset_index,
    group_name=""
):

    info = prediction_inspection_df.iloc[
        dataset_index
    ]

    frames = load_raw_cached_frames(
        dataset_index
    )

    T = len(frames)

    frame_indices = [
        0,
        T // 2,
        T - 1
    ]


    fig, axes = plt.subplots(
        1,
        3,
        figsize=(9, 3.6)
    )


    for ax, frame_idx in zip(
        axes,
        frame_indices
    ):

        ax.imshow(
            frames[
                frame_idx
            ]
        )

        ax.set_title(
            f"Frame {frame_idx}",
            fontsize=9
        )

        ax.axis(
            "off"
        )


    status = (
        "CORRECT"
        if info["correct"]
        else "WRONG"
    )


    title = (
        f"{group_name}\n"
        f"{status} | "
        f"TRUE: {info['true_word']} | "
        f"PRED: {info['pred_word']} "
        f"({info['confidence']:.3f})\n"
        f"Top-3: "
        f"{info['top1_word']} "
        f"{info['top1_prob']:.3f} | "
        f"{info['top2_word']} "
        f"{info['top2_prob']:.3f} | "
        f"{info['top3_word']} "
        f"{info['top3_prob']:.3f} | "
        f"True rank={int(info['true_rank'])}"
    )


    fig.suptitle(
        title,
        fontsize=10
    )


    plt.tight_layout(
        rect=[
            0,
            0,
            1,
            0.82
        ]
    )

    plt.show()


# ============================================================
# A. CORRECT — HIGH CONFIDENCE
# ============================================================

N_SHOW = 6


correct_high = (
    prediction_inspection_df[
        prediction_inspection_df[
            "correct"
        ]
    ]
    .sort_values(
        "confidence",
        ascending=False
    )
    .head(
        N_SHOW
    )
)


print()
print("=" * 88)
print("A. CORRECT — HIGH CONFIDENCE")
print("=" * 88)


display(
    correct_high[
        [
            "dataset_index",
            "true_word",
            "pred_word",
            "confidence",
            "top2_word",
            "top2_prob",
            "top3_word",
            "top3_prob"
        ]
    ].round(
        4
    )
)


for idx in correct_high[
    "dataset_index"
]:

    visualize_prediction(
        int(idx),
        "A. Correct / High Confidence"
    )


# ============================================================
# B. WRONG — HIGH CONFIDENCE
#
# Most important errors:
# model is confidently wrong.
# ============================================================

wrong_high = (
    prediction_inspection_df[
        ~prediction_inspection_df[
            "correct"
        ]
    ]
    .sort_values(
        "confidence",
        ascending=False
    )
    .head(
        N_SHOW
    )
)


print()
print("=" * 88)
print("B. WRONG — HIGH CONFIDENCE")
print("=" * 88)


display(
    wrong_high[
        [
            "dataset_index",
            "true_word",
            "pred_word",
            "confidence",
            "true_rank",
            "true_in_top3",
            "top2_word",
            "top2_prob",
            "top3_word",
            "top3_prob"
        ]
    ].round(
        4
    )
)


for idx in wrong_high[
    "dataset_index"
]:

    visualize_prediction(
        int(idx),
        "B. Wrong / High Confidence"
    )


# ============================================================
# C. WRONG — LOW CONFIDENCE
#
# These are more likely genuine uncertainty rather than
# strong classifier bias.
# ============================================================

wrong_low = (
    prediction_inspection_df[
        ~prediction_inspection_df[
            "correct"
        ]
    ]
    .sort_values(
        "confidence",
        ascending=True
    )
    .head(
        N_SHOW
    )
)


print()
print("=" * 88)
print("C. WRONG — LOW CONFIDENCE")
print("=" * 88)


display(
    wrong_low[
        [
            "dataset_index",
            "true_word",
            "pred_word",
            "confidence",
            "true_rank",
            "true_in_top3",
            "top2_word",
            "top2_prob",
            "top3_word",
            "top3_prob"
        ]
    ].round(
        4
    )
)


for idx in wrong_low[
    "dataset_index"
]:

    visualize_prediction(
        int(idx),
        "C. Wrong / Low Confidence"
    )


# ============================================================
# D. KEY CONFUSION CASES
#
# Based on Cell 19:
#
#   ได้ -> นั่ง
#   ที่   -> นี้
#   ไป   -> มา / มัน
#   ไป <-> มา
#
# These cases are useful for visual inspection.
# ============================================================

key_pairs = [

    ("ได้", "นั่ง"),

    ("ที่", "นี้"),

    ("ไป", "มา"),

    ("มา", "ไป"),

    ("ไป", "มัน")
]


key_confusion_parts = []


for true_word, pred_word in key_pairs:

    part = prediction_inspection_df[
        (
            prediction_inspection_df[
                "true_word"
            ] == true_word
        )
        &
        (
            prediction_inspection_df[
                "pred_word"
            ] == pred_word
        )
    ].copy()


    if len(part) > 0:

        part[
            "pair"
        ] = (
            true_word
            + " -> "
            + pred_word
        )

        key_confusion_parts.append(
            part
        )


if len(
    key_confusion_parts
) > 0:

    key_confusion_df = pd.concat(
        key_confusion_parts,
        ignore_index=True
    )

else:

    key_confusion_df = pd.DataFrame()


print()
print("=" * 88)
print("D. KEY CONFUSION CASES")
print("=" * 88)


if len(
    key_confusion_df
) > 0:

    display(
        key_confusion_df[
            [
                "dataset_index",
                "pair",
                "true_word",
                "pred_word",
                "confidence",
                "true_rank",
                "true_in_top3",
                "top1_prob",
                "top2_word",
                "top2_prob",
                "top3_word",
                "top3_prob"
            ]
        ].round(
            4
        )
    )


    # --------------------------------------------------------
    # Show maximum 12 examples to avoid huge output
    # --------------------------------------------------------

    for _, sample in (
        key_confusion_df
        .head(12)
        .iterrows()
    ):

        visualize_prediction(

            int(
                sample[
                    "dataset_index"
                ]
            ),

            f"D. {sample['pair']}"
        )


else:

    print(
        "No selected key confusion samples found."
    )


# ============================================================
# E. WRONG TOP-1 BUT TRUE WORD IN TOP-3
#
# This subset is especially relevant to future:
# - candidate rescoring
# - temporal refinement
# - constrained decoding
# ============================================================

wrong_top3_examples = (
    wrong_but_top3
    .sort_values(
        "confidence",
        ascending=False
    )
    .head(
        15
    )
)


print()
print("=" * 88)
print("E. WRONG TOP-1 BUT TRUE WORD IS IN TOP-3")
print("=" * 88)

print(
    f"Total such samples : "
    f"{len(wrong_but_top3)}"
)


display(
    wrong_top3_examples[
        [
            "dataset_index",
            "true_word",
            "pred_word",
            "confidence",
            "true_rank",
            "top1_word",
            "top1_prob",
            "top2_word",
            "top2_prob",
            "top3_word",
            "top3_prob"
        ]
    ].round(
        4
    )
)


# ============================================================
# F. TRUE-RANK DISTRIBUTION
# ============================================================

rank_counts = (
    prediction_inspection_df[
        "true_rank"
    ]
    .value_counts()
    .sort_index()
)


print()
print("=" * 88)
print("F. TRUE WORD RANK DISTRIBUTION")
print("=" * 88)


for rank in range(
    1,
    min(
        10,
        NUM_CLASSES
    ) + 1
):

    count = int(
        rank_counts.get(
            rank,
            0
        )
    )

    print(
        f"Rank {rank:2d} : "
        f"{count:3d} "
        f"({count / len(prediction_inspection_df):.2%})"
    )


# ============================================================
# EXPORT FULL PREDICTION TABLE
# ============================================================

PREDICTION_INSPECTION_PATH = (
    MODEL_DIR
    / "01_word_level_baseline_test_predictions.csv"
)


prediction_inspection_df.to_csv(
    PREDICTION_INSPECTION_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# COMPACT RESEARCH SUMMARY
# ============================================================

num_correct = int(
    prediction_inspection_df[
        "correct"
    ].sum()
)


num_wrong = (
    len(
        prediction_inspection_df
    )
    - num_correct
)


num_wrong_true_top3 = int(
    len(
        wrong_but_top3
    )
)


wrong_recoverable_top3_rate = (
    num_wrong_true_top3
    / num_wrong
    if num_wrong > 0
    else 0.0
)


print()
print("=" * 88)
print("COMPACT PREDICTION SUMMARY")
print("=" * 88)

print(
    f"Top-1 correct                  : "
    f"{num_correct}/{len(prediction_inspection_df)} "
    f"({num_correct / len(prediction_inspection_df):.2%})"
)

print(
    f"Top-1 wrong                    : "
    f"{num_wrong}/{len(prediction_inspection_df)} "
    f"({num_wrong / len(prediction_inspection_df):.2%})"
)

print(
    f"Wrong Top-1 but TRUE in Top-3 : "
    f"{num_wrong_true_top3}/{num_wrong} "
    f"({wrong_recoverable_top3_rate:.2%} of errors)"
)

print(
    f"Overall Top-3                  : "
    f"{prediction_inspection_df['true_in_top3'].mean():.2%}"
)


print()
print(
    "Interpretation:"
)

print(
    "If many wrong Top-1 samples still contain the true "
    "word in Top-3, the visual model is often producing "
    "a useful candidate set even when rank-1 is incorrect."
)

print(
    "This motivates later temporal refinement and "
    "constrained candidate rescoring experiments."
)


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 88)
print("✅ CELL 20 PREDICTION INSPECTION COMPLETE")
print("=" * 88)

print(
    f"Saved predictions : "
    f"{PREDICTION_INSPECTION_PATH}"
)

print()

print(
    "Experiment 0 diagnostic sequence complete:"
)

print(
    "Cell 16 — Final test evaluation"
)

print(
    "Cell 17 — Per-class report"
)

print(
    "Cell 18 — Confusion matrix"
)

print(
    "Cell 19 — Confusion-pair analysis"
)

print(
    "Cell 20 — Prediction inspection"
)

print("=" * 88)

In [ ]:
# CELL 21 : EXPORT FINAL V3 CONFIG SUMMARY

summary = {
    "project": "Thai visual word recognition",
    "version": "01 Supervised Word-Level Baseline",
    "task": "44-class word classification",
    "input": {
        "stream": "mouth-only",
        "frames": TARGET_FRAMES,
        "roi_size": ROI_SIZE,
        "channels": 3,
    },
    "model": "CNN frame encoder + BiGRU + temporal attention + classifier",
    "loss": "weighted CrossEntropyLoss" if USE_CLASS_WEIGHTED_LOSS else "CrossEntropyLoss",
    "ctc": False,
    "character_tokenizer": False,
    "best_model_path": str(BEST_MODEL_PATH),
}

summary_path = MODEL_DIR / "01_config_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nSaved:", summary_path)

## Next notebooks — after baseline is measured

Do **not** mix these changes into Notebook 01.

The planned sequence is:

1. **02 — Self-Supervised Visual Pretraining**
2. **03 — SSL Fine-Tuning on the 44-word dataset**
3. **04 — Temporal Modeling / Alignment Ablation**
4. **05 — Face + Mouth Dual-Stream Ablation**
5. **06 — LLM-Assisted Decoding / Top-k Rescoring**
6. **07 — Real-Time Inference / Prototype Evaluation**

Primary baseline question:

> Can mouth motion alone reliably classify the current 44-word Thai vocabulary?
